# NHANES Depression Risk Prediction

## Model Development, Fine-Tuning, and Performance Improvement

### Project Objective

This notebook focuses on implementing, fine-tuning, comparing, and improving multiple machine-learning models to identify participants in the National Health and Nutrition Examination Survey (NHANES) 2017–2018 dataset who may have elevated depressive-symptom risk.

Approximately 9% of participants belong to the elevated-risk class, making this an imbalanced classification problem. Therefore, model development will focus on identifying the minority class effectively rather than relying on ordinary accuracy.

### Models

The following classification models will be implemented and fine-tuned:

- Logistic Regression
- Support Vector Machine
- Decision Tree
- Random Forest
- AdaBoost
- Gradient Boosting
- Histogram Gradient Boosting
- XGBoost
- LightGBM
- CatBoost

### Modeling and Performance-Improvement Workflow

The workflow will:

1. Load and validate the preprocessed modeling dataset.
2. Create a stratified training-and-test split.
3. Keep the test set untouched until final evaluation.
4. Build pipelines for numerical and categorical predictors.
5. Establish an initial cross-validation result for each model.
6. Apply an appropriate class-imbalance strategy.
7. Fine-tune each model using model-specific hyperparameters and stratified cross-validation.
8. Generate out-of-fold probabilities using training data only.
9. Select a separate classification threshold for each fine-tuned model using the F2 score.
10. Compare the initial and fine-tuned performance of every model.
11. Select the strongest model based on overall screening performance.
12. Evaluate the selected model once on the untouched test set.
13. Save the complete model pipeline, selected threshold, predictors, and metadata for deployment.

### Class-Imbalance Strategy

Class weighting will be the primary imbalance-handling strategy for models that support it. Selected models will also be tested with the Synthetic Minority Over-sampling Technique (SMOTE) to determine whether synthetic minority-class sampling improves performance.

SMOTE will be applied only to the training portion of each cross-validation fold using an `imblearn` pipeline. It will not be applied before the train-test split or to validation and test observations.

Model-specific imbalance parameters will also be considered:

- XGBoost: training-derived `scale_pos_weight`
- LightGBM: class weighting or `scale_pos_weight`
- CatBoost: automatic class weighting

SMOTE will be retained only when cross-validation results show that it improves minority-class performance without producing an unacceptable number of false-positive predictions.

### Performance-Improvement Methods

Model performance will be improved through:

- Model-specific hyperparameter fine-tuning
- Class and sample weighting
- Controlled SMOTE experiments
- Threshold optimization
- Feature selection when supported by validation results
- Probability calibration when necessary
- Overfitting analysis
- Investigation of unusually low recall or substantial performance differences

### Evaluation Metrics

Models will be evaluated using:

- Balanced accuracy
- Precision
- Recall (sensitivity)
- Specificity
- F1 score
- F2 score
- ROC-AUC
- PR-AUC

PR-AUC will be the primary fine-tuning and ranking metric because it focuses on performance for the uncommon elevated-risk class. The approximate no-skill PR-AUC baseline is the positive-class prevalence of 0.0906.

The F2 score will guide threshold selection because recall is especially important when screening participants for possible elevated depressive-symptom risk.

### Fair-Comparison Approach

Every model will use:

- The same training and test observations
- The same stratified cross-validation folds
- Consistent preprocessing
- Appropriate imbalance handling
- Model-specific hyperparameter fine-tuning
- The same threshold-selection method
- The same evaluation metrics

Model performance will not be forced to be similar. Any substantial performance differences—particularly unusually low recall—will be investigated before selecting the final model.

All model-development and performance-improvement decisions will be based exclusively on training and cross-validation results. The test set will remain untouched until the final evaluation.

## 1. Modeling Environment

This section verifies the Python environment and confirms that the libraries required for preprocessing, imbalance handling, model development, and fine-tuning are available.

In [1]:
import sys
import importlib.util
import pandas as pd
import numpy as np
import sklearn

required_packages = {
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
    "CatBoost": "catboost",
    "Imbalanced-learn": "imblearn"
}

environment_results = []

for package_name, import_name in required_packages.items():
    installed = importlib.util.find_spec(import_name) is not None

    environment_results.append({
        "package": package_name,
        "installed": installed
    })

environment_check = pd.DataFrame(environment_results)

print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

display(environment_check)

Python version: 3.11.4
Pandas version: 3.0.6
NumPy version: 2.4.6
Scikit-learn version: 1.9.1


,package,installed
0,XGBoost,True
1,LightGBM,True
2,CatBoost,True
3,Imbalanced-learn,True


In [2]:
%pip install xgboost lightgbm catboost imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Library Imports and Reproducibility

This section imports the tools required for data preparation, class-imbalance handling, model development, fine-tuning, threshold optimization, and evaluation. A fixed random seed is used to make the results reproducible.

In [67]:
import warnings
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Data splitting and cross-validation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_val_predict
)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Machine-learning models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbalancedPipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC

# Evaluation
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    make_scorer
)

# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

warnings.filterwarnings("ignore")

print("Modeling libraries imported successfully.")
print(f"Random state: {RANDOM_STATE}")

Modeling libraries imported successfully.
Random state: 42


## 3. Load and Inspect the Modeling Dataset

The preprocessed NHANES modeling dataset is loaded and checked before creating the training and test sets. This step confirms the dataset dimensions, column names, missing values, and possible target and identifier columns.

In [4]:
from pathlib import Path

# Define the processed modeling dataset location
data_path = Path("../data/processed/nhanes_2017_2018_preprocessed.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {data_path.resolve()}"
    )

# Load the modeling dataset
model_df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print(f"Dataset location: {data_path.resolve()}")
print(f"Dataset shape: {model_df.shape}")
print(f"Number of rows: {model_df.shape[0]}")
print(f"Number of columns: {model_df.shape[1]}")
print(f"Duplicate rows: {model_df.duplicated().sum()}")
print(f"Total missing values: {model_df.isna().sum().sum()}")

print("\nDataset columns:")
display(pd.DataFrame({"column_name": model_df.columns}))

print("\nFirst five observations:")
display(model_df.head())

Dataset loaded successfully.
Dataset location: /Users/sibelsezen/Desktop/Magnimind/nhanes-depression-risk-prediction/data/processed/nhanes_2017_2018_preprocessed.csv
Dataset shape: (5068, 31)
Number of rows: 5068
Number of columns: 31
Duplicate rows: 0
Total missing values: 1982

Dataset columns:


,column_name
0,SEQN
1,RIAGENDR
2,RIDAGEYR
3,RIDRETH3
4,DMDEDUC2
5,DMDMARTL
6,INDFMPIR
7,SLQ030
8,SLQ040
9,SLQ050



First five observations:


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,DMDMARTL,INDFMPIR,SLQ030,SLQ040,SLQ050,SLQ120,BMXBMI,BMXWAIST,sedentary_minutes_per_day,cigarettes_per_day,alcohol_frequency,average_drinks_per_day,binge_drinking_frequency,binge_drinking_episodes_30d,elevated_depression_risk,smoking_status,alcohol_use_status,ever_heavy_daily_drinking,average_sleep_hours,weekend_sleep_difference,vigorous_work_activity,moderate_work_activity,walking_or_bicycling,vigorous_recreation,moderate_recreation,active_domain_count
0,93705.0000,2.0000,66.0000,4.0000,2.0000,3.0000,0.8200,2.0000,0.0000,2.0000,0.0000,31.7000,101.8000,300.0000,0.0000,7.0000,1.0000,0.0000,5.0000,0,1.0000,2.0000,0.0000,8.0000,0.0000,0.0000,0.0000,0,0,1,1.0000
1,93706.0000,1.0000,18.0000,6.0000,0.0000,0.0000,NaN,1.0000,0.0000,2.0000,1.0000,21.5000,79.3000,240.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,10.7857,1.0000,0.0000,0.0000,1,0,1,2.0000
2,93708.0000,2.0000,66.0000,6.0000,1.0000,1.0000,1.6300,NaN,0.0000,2.0000,2.0000,23.7000,88.2000,120.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,8.0000,0.0000,0.0000,0.0000,0,0,1,1.0000
3,93711.0000,1.0000,56.0000,6.0000,5.0000,1.0000,5.0000,2.0000,1.0000,1.0000,3.0000,21.3000,86.6000,420.0000,0.0000,5.0000,1.0000,0.0000,0.0000,0,0.0000,2.0000,0.0000,7.5714,2.0000,0.0000,0.0000,1,1,1,3.0000
4,93712.0000,1.0000,18.0000,1.0000,0.0000,0.0000,0.7600,1.0000,1.0000,2.0000,2.0000,19.7000,72.0000,120.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0,2.0000,1.0000,0.0000,7.9286,1.5000,1.0000,1.0000,0,1,1,4.0000


## 4. Define the Target and Predictor Variables

The participant identifier is stored separately and excluded from modeling. The binary elevated-depression-risk variable is used as the target, while the remaining 29 variables are used as predictors. The target distribution is validated before splitting the dataset.

In [5]:
# Define the identifier and target columns
identifier_column = "SEQN"
target_column = "elevated_depression_risk"

# Confirm that the required columns exist
required_columns = {identifier_column, target_column}
missing_required_columns = required_columns.difference(model_df.columns)

if missing_required_columns:
    raise KeyError(
        f"Required columns are missing: {sorted(missing_required_columns)}"
    )

# Validate participant identifiers
if model_df[identifier_column].isna().any():
    raise ValueError("Missing participant identifiers were found.")

if model_df[identifier_column].duplicated().any():
    raise ValueError("Duplicate participant identifiers were found.")

# Validate the target
if model_df[target_column].isna().any():
    raise ValueError("Missing target values were found.")

target_values = sorted(model_df[target_column].unique().tolist())

if target_values != [0, 1]:
    raise ValueError(
        f"Target must contain only 0 and 1. Found: {target_values}"
    )

# Separate identifiers, predictors, and target
participant_ids = model_df[identifier_column].copy()

X = model_df.drop(
    columns=[identifier_column, target_column]
).copy()

y = model_df[target_column].astype(int).copy()

# Summarize the target distribution
target_summary = (
    y.value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(y) * 100
).round(2)

target_summary["class_label"] = target_summary["target"].map({
    0: "Below threshold",
    1: "Elevated risk"
})

display(target_summary)

print("Target and predictors validated successfully.")
print(f"Number of participants: {len(model_df)}")
print(f"Number of predictors: {X.shape[1]}")
print(f"Missing target values: {y.isna().sum()}")
print(f"Duplicate participant IDs: {participant_ids.duplicated().sum()}")
print(f"Predictor matrix shape: {X.shape}")

,target,count,percentage,class_label
0,0,4609,90.9400,Below threshold
1,1,459,9.0600,Elevated risk


Target and predictors validated successfully.
Number of participants: 5068
Number of predictors: 29
Missing target values: 0
Duplicate participant IDs: 0
Predictor matrix shape: (5068, 29)


## 5. Stratified Training and Test Split

The dataset is divided into an 80% training set and a 20% test set using stratification. Stratification preserves the elevated-risk rate in both subsets.

The test set will remain untouched during preprocessing decisions, cross-validation, imbalance handling, hyperparameter fine-tuning, threshold selection, and model selection. It will be used only once for the final evaluation.

In [6]:
# Create one stratified 80/20 split
(
    X_train,
    X_test,
    y_train,
    y_test,
    participant_ids_train,
    participant_ids_test
) = train_test_split(
    X,
    y,
    participant_ids,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Verify index alignment
train_alignment = (
    X_train.index.equals(y_train.index)
    and X_train.index.equals(participant_ids_train.index)
)

test_alignment = (
    X_test.index.equals(y_test.index)
    and X_test.index.equals(participant_ids_test.index)
)

# Summarize each dataset split
def summarize_split(split_name, target):
    negative_cases = int((target == 0).sum())
    positive_cases = int((target == 1).sum())

    return {
        "dataset_split": split_name,
        "total_records": len(target),
        "negative_cases": negative_cases,
        "positive_cases": positive_cases,
        "positive_percentage": round(
            positive_cases / len(target) * 100,
            2
        )
    }

split_summary = pd.DataFrame([
    summarize_split("Complete dataset", y),
    summarize_split("Training set", y_train),
    summarize_split("Test set", y_test)
]).set_index("dataset_split")

display(split_summary)

print("Stratified training-test split completed.")
print(f"Training predictor shape: {X_train.shape}")
print(f"Test predictor shape: {X_test.shape}")
print(f"Training indices aligned: {train_alignment}")
print(f"Test indices aligned: {test_alignment}")
print("Test set reserved for final evaluation: True")

,total_records,negative_cases,positive_cases,positive_percentage
dataset_split,,,,
Complete dataset,5068,4609,459,9.0600
Training set,4054,3687,367,9.0500
Test set,1014,922,92,9.0700


Stratified training-test split completed.
Training predictor shape: (4054, 29)
Test predictor shape: (1014, 29)
Training indices aligned: True
Test indices aligned: True
Test set reserved for final evaluation: True


## 6. Numerical and Categorical Predictor Assignment

The 29 predictors are assigned to numerical and categorical groups based on their meaning rather than their stored data type. This is necessary because several NHANES categorical variables are represented using numerical codes.

Numerical variables will be imputed and standardized. Categorical variables will be imputed and one-hot encoded within each model pipeline.

In [7]:
# Continuous and count-based predictors
numeric_columns = [
    "RIDAGEYR",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST",
    "sedentary_minutes_per_day",
    "cigarettes_per_day",
    "alcohol_frequency",
    "average_drinks_per_day",
    "binge_drinking_frequency",
    "binge_drinking_episodes_30d",
    "average_sleep_hours",
    "weekend_sleep_difference",
    "active_domain_count"
]

# Categorical and coded-category predictors
categorical_columns = [
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "SLQ030",
    "SLQ040",
    "SLQ050",
    "SLQ120",
    "smoking_status",
    "alcohol_use_status",
    "ever_heavy_daily_drinking",
    "vigorous_work_activity",
    "moderate_work_activity",
    "walking_or_bicycling",
    "vigorous_recreation",
    "moderate_recreation"
]

# Validate the feature assignments
assigned_columns = numeric_columns + categorical_columns

unassigned_columns = sorted(
    set(X_train.columns) - set(assigned_columns)
)

unknown_columns = sorted(
    set(assigned_columns) - set(X_train.columns)
)

overlapping_columns = sorted(
    set(numeric_columns).intersection(categorical_columns)
)

duplicate_assignments = sorted({
    column
    for column in assigned_columns
    if assigned_columns.count(column) > 1
})

if unassigned_columns:
    raise ValueError(
        f"Unassigned predictors found: {unassigned_columns}"
    )

if unknown_columns:
    raise ValueError(
        f"Assigned columns not found in the dataset: {unknown_columns}"
    )

if overlapping_columns:
    raise ValueError(
        f"Predictors assigned to both groups: {overlapping_columns}"
    )

if duplicate_assignments:
    raise ValueError(
        f"Duplicate feature assignments found: {duplicate_assignments}"
    )

feature_type_summary = pd.DataFrame({
    "feature_type": [
        "Numerical",
        "Categorical",
        "Total assigned",
        "Unassigned",
        "Overlapping"
    ],
    "number_of_features": [
        len(numeric_columns),
        len(categorical_columns),
        len(assigned_columns),
        len(unassigned_columns),
        len(overlapping_columns)
    ]
})

display(feature_type_summary)

print("Feature assignment validated successfully.")
print(f"Numerical predictors: {len(numeric_columns)}")
print(f"Categorical predictors: {len(categorical_columns)}")
print(f"Total predictors assigned: {len(assigned_columns)}")
print(f"Unassigned predictors: {unassigned_columns}")
print(f"Overlapping predictors: {overlapping_columns}")

,feature_type,number_of_features
0,Numerical,13
1,Categorical,16
2,Total assigned,29
3,Unassigned,0
4,Overlapping,0


Feature assignment validated successfully.
Numerical predictors: 13
Categorical predictors: 16
Total predictors assigned: 29
Unassigned predictors: []
Overlapping predictors: []


## 7. Preprocessing Pipeline

Numerical predictors are imputed using their training-fold median and standardized. Categorical predictors are imputed using their most frequent training-fold category and converted into one-hot encoded variables.

Preprocessing will remain inside each model pipeline so that its parameters are learned separately within every cross-validation training fold. This prevents information leakage.

In [8]:
from sklearn.base import clone

# Numerical preprocessing
numeric_pipeline = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical preprocessing
categorical_pipeline = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Validate the preprocessing design using training data only
preprocessing_check = clone(preprocessor).fit_transform(X_train)

print("Preprocessing pipeline created successfully.")
print(f"Original training shape: {X_train.shape}")
print(f"Transformed training shape: {preprocessing_check.shape}")
print(
    "Missing values after transformation:",
    int(np.isnan(preprocessing_check).sum())
)
print(f"Test data used to fit preprocessing: False")

Preprocessing pipeline created successfully.
Original training shape: (4054, 29)
Transformed training shape: (4054, 68)
Missing values after transformation: 0
Test data used to fit preprocessing: False


In [146]:
# Verify preprocessing used by the final Logistic Regression pipeline

# Locate all Logistic Regression pipeline variables
# and verify whether they contain StandardScaler

logistic_pipeline_names = [
    variable_name
    for variable_name, variable_value in globals().items()
    if "logistic" in variable_name.lower()
    and "pipeline" in variable_name.lower()
    and hasattr(variable_value, "named_steps")
]

if not logistic_pipeline_names:
    print("No Logistic Regression pipeline variable was found.")
else:
    print("Logistic Regression pipelines found:\n")

    for variable_name in logistic_pipeline_names:
        pipeline_object = globals()[variable_name]
        pipeline_text = repr(pipeline_object)

        print("Variable:", variable_name)
        print(
            "StandardScaler included:",
            "StandardScaler" in pipeline_text,
        )
        print(pipeline_object)
        print("-" * 80)

Logistic Regression pipelines found:

Variable: initial_logistic_pipeline
StandardScaler included: True
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['RIDAGEYR', 'INDFMPIR',
                                                   'BMXBMI', 'BMXWAIST',
                                                   'sedentary_minutes_per_day',
                                                   'cigarettes_per_day',
                                                   'alcohol_frequency',
                                                   'average_drinks_per_day',
       

## Cross-Validation and Evaluation Framework

Five-fold stratified cross-validation will be used to preserve the elevated-risk proportion within every fold. The same folds and evaluation metrics will be used for all models.

PR-AUC will be the primary fine-tuning metric. Ordinary accuracy is excluded because it can be misleading for this imbalanced dataset. F2 will later be used to select a model-specific classification threshold using out-of-fold training probabilities.

In [9]:
# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Specificity: proportion of below-threshold cases correctly identified
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


# Define scorers without ordinary accuracy
scoring_metrics = {
    "balanced_accuracy": make_scorer(
        balanced_accuracy_score
    ),
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "specificity": make_scorer(
        specificity_score
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "f2": make_scorer(
        fbeta_score,
        beta=2,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

# Training-set imbalance information
training_negative_cases = int((y_train == 0).sum())
training_positive_cases = int((y_train == 1).sum())

scale_pos_weight = (
    training_negative_cases / training_positive_cases
)

training_pr_auc_baseline = y_train.mean()

print("Cross-validation and scoring framework created.")
print(f"Number of folds: {cv.get_n_splits()}")
print(f"Training negative cases: {training_negative_cases}")
print(f"Training positive cases: {training_positive_cases}")
print(f"Training imbalance ratio: {scale_pos_weight:.4f}")
print(f"No-skill PR-AUC baseline: {training_pr_auc_baseline:.4f}")
print("Primary fine-tuning metric: PR-AUC")
print("Threshold-selection metric: F2")
print("Ordinary accuracy included: False")
print("Test set used: False")

Cross-validation and scoring framework created.
Number of folds: 5
Training negative cases: 3687
Training positive cases: 367
Training imbalance ratio: 10.0463
No-skill PR-AUC baseline: 0.0905
Primary fine-tuning metric: PR-AUC
Threshold-selection metric: F2
Ordinary accuracy included: False
Test set used: False


## Threshold Selection and Model Evaluation Functions

The following functions ensure that every model is evaluated using the same process.

A model-specific threshold is selected by maximizing the F2 score using out-of-fold training probabilities. The selected threshold is then used to calculate balanced accuracy, precision, recall, specificity, F1, and F2. ROC-AUC and PR-AUC are calculated directly from predicted probabilities.

In [10]:
def select_f2_threshold(y_true, probabilities):
    """
    Select the probability threshold that maximizes the F2 score.
    """
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probabilities
    )

    # The last precision and recall values do not have a corresponding threshold
    precision = precision[:-1]
    recall = recall[:-1]

    beta_squared = 2 ** 2

    denominator = (
        beta_squared * precision + recall
    )

    f2_scores = np.divide(
        (1 + beta_squared) * precision * recall,
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator != 0
    )

    best_index = int(np.argmax(f2_scores))
    best_threshold = float(thresholds[best_index])

    return best_threshold, float(f2_scores[best_index])


def evaluate_probabilities(
    y_true,
    probabilities,
    threshold
):
    """
    Evaluate predicted probabilities at a selected threshold.
    Ordinary accuracy is intentionally excluded.
    """
    predictions = (
        np.asarray(probabilities) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_true,
            probabilities
        ),
        "predicted_positive_cases": int(
            predictions.sum()
        )
    }


def create_oof_evaluation(
    model_name,
    pipeline,
    X_training,
    y_training
):
    """
    Generate out-of-fold probabilities and evaluate a model using
    an F2-optimized threshold.
    """
    oof_probabilities = cross_val_predict(
        pipeline,
        X_training,
        y_training,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    selected_threshold, maximum_f2 = select_f2_threshold(
        y_training,
        oof_probabilities
    )

    results = evaluate_probabilities(
        y_training,
        oof_probabilities,
        selected_threshold
    )

    results["model"] = model_name
    results["maximum_threshold_f2"] = maximum_f2

    return results, oof_probabilities


print("Evaluation functions created successfully.")
print("Threshold-selection rule: Maximum F2")
print("Evaluation source: Out-of-fold training probabilities")
print("Ordinary accuracy calculated: False")
print("Test set used: False")

Evaluation functions created successfully.
Threshold-selection rule: Maximum F2
Evaluation source: Out-of-fold training probabilities
Ordinary accuracy calculated: False
Test set used: False


##  Logistic Regression

###  Initial Class-Weighted Model

Logistic Regression is used as the first interpretable model. Because the elevated-risk class is uncommon, balanced class weights are applied so that errors involving either class receive appropriate importance.

The initial model uses standard parameters. Its out-of-fold results will provide a reference point for measuring the improvement achieved through fine-tuning.

In [33]:
# Create the initial class-weighted Logistic Regression pipeline
initial_logistic_pipeline = SklearnPipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                penalty="l2",
                solver="liblinear",
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

print("Evaluating initial Logistic Regression...")

# Generate out-of-fold probabilities and optimize the threshold
(
    initial_logistic_results,
    initial_logistic_oof_probabilities
) = create_oof_evaluation(
    model_name="Logistic Regression",
    pipeline=initial_logistic_pipeline,
    X_training=X_train,
    y_training=y_train
)

initial_logistic_results["model_stage"] = "Initial"

# Display results in a clear order
initial_logistic_table = pd.DataFrame(
    [initial_logistic_results]
)[[
    "model",
    "model_stage",
    "threshold",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
    "predicted_positive_cases"
]]

display(
    initial_logistic_table.set_index(
        ["model", "model_stage"]
    )
)

print("Initial Logistic Regression evaluation completed.")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Test set used: False")

Evaluating initial Logistic Regression...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Logistic Regression,Initial,0.4125,0.7480,0.2063,0.8038,0.6922,0.3283,0.5090,0.8077,0.2955,1430


Initial Logistic Regression evaluation completed.
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Test set used: False


In [36]:
# Step: Evaluate the tuned Logistic Regression
# This evaluation uses only the training data.
# The untouched test set is reserved for the final model evaluation.

# Generate out-of-fold predicted probabilities using stratified cross-validation.
# Out-of-fold predictions prevent the model from evaluating rows it trained on.
logistic_tuned_results, logistic_tuned_oof_probabilities = create_oof_evaluation(
    model_name="Logistic Regression",
    pipeline=tuned_logistic_pipeline,
    X_training=X_train,
    y_training=y_train
)

# Label the model and its development stage for the comparison table.
logistic_tuned_results["model"] = "Logistic Regression"
logistic_tuned_results["model_stage"] = "Tuned"

# Convert the evaluation results into a readable table.
logistic_tuned_results_table = pd.DataFrame(
    [logistic_tuned_results]
).set_index(["model", "model_stage"])

# Display all evaluation metrics rounded to four decimal places.
display(logistic_tuned_results_table.round(4))

# Document the evaluation decisions.
print("Tuned Logistic Regression evaluation completed.")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Test set used: False")

,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases,maximum_threshold_f2
model,model_stage,,,,,,,,,,,
Logistic Regression,Tuned,0.4369,0.7475,0.2124,0.7847,0.7103,0.3343,0.5099,0.8082,0.2961,1356,0.5099


Tuned Logistic Regression evaluation completed.
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Test set used: False


In [37]:
# Step: Compare the initial and tuned Logistic Regression models
# Both models were evaluated with the same stratified cross-validation process.
# The test set was not used in either evaluation.

# Create independent copies so the original result dictionaries remain unchanged.
initial_logistic_comparison = initial_logistic_results.copy()
tuned_logistic_comparison = logistic_tuned_results.copy()

# Add clear development-stage labels.
initial_logistic_comparison["model_stage"] = "Initial"
tuned_logistic_comparison["model_stage"] = "Tuned"

# Combine both results into one comparison table.
logistic_comparison_table = pd.DataFrame([
    initial_logistic_comparison,
    tuned_logistic_comparison
])

# Select only the meaningful metrics for this imbalanced dataset.
# Accuracy is intentionally excluded.
logistic_comparison_table = logistic_comparison_table[
    [
        "model_stage",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("model_stage")

# Display the comparison.
display(logistic_comparison_table.round(4))

# Calculate the change from the initial model to the tuned model.
recall_change = (
    logistic_comparison_table.loc["Tuned", "recall"]
    - logistic_comparison_table.loc["Initial", "recall"]
)

f2_change = (
    logistic_comparison_table.loc["Tuned", "f2"]
    - logistic_comparison_table.loc["Initial", "f2"]
)

pr_auc_change = (
    logistic_comparison_table.loc["Tuned", "pr_auc"]
    - logistic_comparison_table.loc["Initial", "pr_auc"]
)

# Summarize the effect of fine-tuning.
print("Initial and tuned Logistic Regression comparison completed.")
print(f"Recall change: {recall_change:+.4f}")
print(f"F2 change: {f2_change:+.4f}")
print(f"PR-AUC change: {pr_auc_change:+.4f}")
print("Test set used: False")

,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
model_stage,,,,,,,,,
Initial,0.4125,0.7480,0.2063,0.8038,0.6922,0.3283,0.5090,0.8077,0.2955
Tuned,0.4369,0.7475,0.2124,0.7847,0.7103,0.3343,0.5099,0.8082,0.2961


Initial and tuned Logistic Regression comparison completed.
Recall change: -0.0191
F2 change: +0.0009
PR-AUC change: +0.0005
Test set used: False


In [38]:
# Step: Evaluate Logistic Regression with SMOTE
# SMOTE creates synthetic minority-class examples inside each training fold.
# Keeping SMOTE inside the pipeline prevents validation-data leakage.

from sklearn.base import clone
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Copy the tuned Logistic Regression classifier.
smote_logistic_classifier = clone(
    tuned_logistic_pipeline.named_steps["classifier"]
)

# Remove balanced class weights because SMOTE already addresses imbalance.
# Using both methods together could overcorrect the minority class.
smote_logistic_classifier.set_params(
    class_weight=None,
    max_iter=5000,
    random_state=42
)

# Construct a leakage-safe pipeline:
# 1. Fit preprocessing on each training fold.
# 2. Apply SMOTE only to that processed training fold.
# 3. Fit Logistic Regression on the resampled data.
smote_logistic_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            clone(tuned_logistic_pipeline.named_steps["preprocessor"])
        ),
        (
            "smote",
            SMOTE(
                sampling_strategy="auto",
                k_neighbors=5,
                random_state=42
            )
        ),
        (
            "classifier",
            smote_logistic_classifier
        )
    ]
)

# Generate out-of-fold predictions and select the maximum-F2 threshold.
logistic_smote_results, logistic_smote_oof_probabilities = (
    create_oof_evaluation(
        model_name="Logistic Regression",
        pipeline=smote_logistic_pipeline,
        X_training=X_train,
        y_training=y_train
    )
)

# Add labels for the final comparison table.
logistic_smote_results["model"] = "Logistic Regression"
logistic_smote_results["model_stage"] = "Tuned + SMOTE"

# Display the SMOTE evaluation results.
logistic_smote_results_table = pd.DataFrame(
    [logistic_smote_results]
).set_index(["model", "model_stage"])

display(logistic_smote_results_table.round(4))

print("SMOTE Logistic Regression evaluation completed.")
print("Imbalance handling: SMOTE inside each cross-validation training fold")
print("Class weights used: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Test set used: False")

,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases,maximum_threshold_f2
model,model_stage,,,,,,,,,,,
Logistic Regression,Tuned + SMOTE,0.4523,0.7407,0.2200,0.7439,0.7375,0.3396,0.5039,0.8042,0.2869,1241,0.5039


SMOTE Logistic Regression evaluation completed.
Imbalance handling: SMOTE inside each cross-validation training fold
Class weights used: False
Threshold selected using: Maximum out-of-fold F2
Test set used: False


In [39]:
# Step: Compare all Logistic Regression configurations
# Accuracy is excluded because the target is highly imbalanced.
# Recall is the primary selection metric for this screening project.

# Create separate copies to avoid modifying stored results.
initial_result = initial_logistic_results.copy()
tuned_result = logistic_tuned_results.copy()
smote_result = logistic_smote_results.copy()

# Assign clear configuration labels.
initial_result["configuration"] = "Initial + Class Weight"
tuned_result["configuration"] = "Tuned + Class Weight"
smote_result["configuration"] = "Tuned + SMOTE"

# Combine all Logistic Regression results.
logistic_final_comparison = pd.DataFrame([
    initial_result,
    tuned_result,
    smote_result
])

# Keep the metrics relevant to the imbalanced classification problem.
logistic_final_comparison = logistic_final_comparison[
    [
        "configuration",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("configuration")

# Sort by recall because sensitivity is the primary model-selection metric.
logistic_final_comparison = logistic_final_comparison.sort_values(
    by="recall",
    ascending=False
)

display(logistic_final_comparison.round(4))

# Identify the strongest configuration using recall.
best_logistic_configuration = (
    logistic_final_comparison["recall"].idxmax()
)

best_logistic_recall = (
    logistic_final_comparison.loc[
        best_logistic_configuration,
        "recall"
    ]
)

# Identify the strongest configuration using F2 as supporting evidence.
best_logistic_f2_configuration = (
    logistic_final_comparison["f2"].idxmax()
)

print("Logistic Regression configuration comparison completed.")
print(f"Best configuration by recall: {best_logistic_configuration}")
print(f"Best recall: {best_logistic_recall:.4f}")
print(f"Best configuration by F2: {best_logistic_f2_configuration}")
print("Primary selection metric: Recall")
print("Test set used: False")

,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
configuration,,,,,,,,,
Initial + Class Weight,0.4125,0.7480,0.2063,0.8038,0.6922,0.3283,0.5090,0.8077,0.2955
Tuned + Class Weight,0.4369,0.7475,0.2124,0.7847,0.7103,0.3343,0.5099,0.8082,0.2961
Tuned + SMOTE,0.4523,0.7407,0.2200,0.7439,0.7375,0.3396,0.5039,0.8042,0.2869


Logistic Regression configuration comparison completed.
Best configuration by recall: Initial + Class Weight
Best recall: 0.8038
Best configuration by F2: Tuned + Class Weight
Primary selection metric: Recall
Test set used: False


In [147]:
# Finalize the selected Logistic Regression configuration

# The initial class-weighted model is retained because it has
# the highest recall, while its ROC-AUC is nearly identical
# to the tuned model.
logistic_final_pipeline = clone(
    selected_logistic_pipeline
)

logistic_final_results = (
    logistic_initial_results.copy()
)

# Use consistent labels for the final model comparison.
logistic_final_results["model"] = "Logistic Regression"
logistic_final_results["model_stage"] = "Final selected"

logistic_final_threshold = float(
    logistic_final_results["threshold"]
)

# Verify that numerical scaling is present.
logistic_scaling_included = (
    "StandardScaler"
    in repr(logistic_final_pipeline)
)

# Check the model-selection guardrails.
logistic_final_passes_guardrails = (
    logistic_final_results["balanced_accuracy"] >= 0.70
    and logistic_final_results["specificity"] >= 0.65
)

print("Final Logistic Regression assigned successfully.")
print("Selected configuration: Initial + Class Weight")
print(
    "Numerical scaling included:",
    logistic_scaling_included,
)
print("Scaler: StandardScaler")
print("Class weighting retained: True")
print("SMOTE retained: False")
print(
    "Selected threshold:",
    round(logistic_final_threshold, 4),
)
print(
    "Recall:",
    round(logistic_final_results["recall"], 4),
)
print(
    "ROC-AUC:",
    round(logistic_final_results["roc_auc"], 4),
)
print(
    "Specificity:",
    round(logistic_final_results["specificity"], 4),
)
print(
    "Balanced accuracy:",
    round(
        logistic_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "Passes performance guardrails:",
    logistic_final_passes_guardrails,
)
print("Primary presentation metric: Recall")
print("Secondary presentation metric: ROC-AUC")
print("Test set evaluated: False")

Final Logistic Regression assigned successfully.
Selected configuration: Initial + Class Weight
Numerical scaling included: True
Scaler: StandardScaler
Class weighting retained: True
SMOTE retained: False
Selected threshold: 0.4125
Recall: 0.8038
ROC-AUC: 0.8077
Specificity: 0.6922
Balanced accuracy: 0.748
Passes performance guardrails: True
Primary presentation metric: Recall
Secondary presentation metric: ROC-AUC
Test set evaluated: False


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

print("Pipeline and clone imported successfully.")

Pipeline and clone imported successfully.


### Correlated-Feature Reduction

BMI and waist circumference were strongly correlated during exploratory analysis, with a Spearman correlation of approximately 0.91. These predictors may provide redundant information and may reduce the stability of a linear model.

The selected class-balanced Logistic Regression is therefore evaluated after removing BMI and waist circumference. The reduced 27-predictor configuration is compared with the original 29-predictor configuration using the same stratified cross-validation and out-of-fold F2 threshold-selection framework.

Only training data are used in this experiment. The reduced configuration will be retained only if it maintains or improves recall without causing a meaningful decline in the supporting metrics.

In [139]:
# Evaluate Logistic Regression without BMI and waist circumference

removed_correlated_features = ["BMXBMI", "BMXWAIST"]

reduced_numeric_columns = [
    column
    for column in numeric_columns
    if column not in removed_correlated_features
]

reduced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            reduced_numeric_columns,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

logistic_reduced_pipeline = Pipeline(
    steps=[
        ("preprocessor", reduced_preprocessor),
        (
            "classifier",
            LogisticRegression(
                C=logistic_search.best_params_["classifier__C"],
                penalty=logistic_search.best_params_["classifier__penalty"],
                solver=logistic_search.best_params_["classifier__solver"],
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("Evaluating Logistic Regression without BMI and waist circumference...")

(
    logistic_reduced_results,
    logistic_reduced_oof_probabilities,
) = create_oof_evaluation(
    model_name="Logistic Regression",
    pipeline=logistic_reduced_pipeline,
    X_training=X_train,
    y_training=y_train,
)

logistic_reduced_results["model_stage"] = "Tuned - BMI and waist"

feature_comparison = pd.DataFrame(
    [
        logistic_tuned_results,
        logistic_reduced_results,
    ]
)

display(
    feature_comparison[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Correlated-feature experiment completed.")
print(f"Removed predictors: {removed_correlated_features}")
print("Primary comparison metric: PR-AUC")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating Logistic Regression without BMI and waist circumference...


threshold  balanced_accuracy  \
model               model_stage                                           
Logistic Regression Tuned                     0.4369             0.7475   
                    Tuned - BMI and waist     0.4362             0.7462   

                                           precision  recall  specificity  \
model               model_stage                                             
Logistic Regression Tuned                     0.2124  0.7847       0.7103   
                    Tuned - BMI and waist     0.2099  0.7875       0.7049   

                                              f1     f2  roc_auc  pr_auc  \
model               model_stage                                            
Logistic Regression Tuned                 0.3343 0.5099   0.8082  0.2961   
                    Tuned - BMI and waist 0.3314 0.5079   0.8079  0.2988   

                                           predicted_positive_cases  
model               model_stage                                      
Logistic Regression Tuned                                      1356  
                    Tuned - BMI and waist                      1377

Correlated-feature experiment completed.
Removed predictors: ['BMXBMI', 'BMXWAIST']
Primary comparison metric: PR-AUC
Ordinary accuracy included: False
Test set evaluated: False


In [140]:
# Compare different BMI and waist feature configurations

def build_logistic_feature_pipeline(features_to_remove):
    selected_numeric_columns = [
        column
        for column in numeric_columns
        if column not in features_to_remove
    ]

    feature_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                selected_numeric_columns,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return Pipeline(
        steps=[
            ("preprocessor", feature_preprocessor),
            (
                "classifier",
                LogisticRegression(
                    C=logistic_search.best_params_["classifier__C"],
                    penalty=logistic_search.best_params_[
                        "classifier__penalty"
                    ],
                    solver=logistic_search.best_params_[
                        "classifier__solver"
                    ],
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


feature_experiments = [
    ("Tuned - waist only", ["BMXWAIST"]),
    ("Tuned - BMI only", ["BMXBMI"]),
]

additional_feature_results = []

for stage_name, features_to_remove in feature_experiments:
    print(f"Evaluating: {stage_name}...")

    experiment_pipeline = build_logistic_feature_pipeline(
        features_to_remove
    )

    experiment_results, _ = create_oof_evaluation(
        model_name="Logistic Regression",
        pipeline=experiment_pipeline,
        X_training=X_train,
        y_training=y_train,
    )

    experiment_results["model_stage"] = stage_name
    additional_feature_results.append(experiment_results)


complete_feature_comparison = pd.DataFrame(
    [
        logistic_tuned_results,
        *additional_feature_results,
        logistic_reduced_results,
    ]
)

display(
    complete_feature_comparison[comparison_columns]
    .sort_values("pr_auc", ascending=False)
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Feature-configuration comparison completed.")
print("Configurations ranked by PR-AUC.")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating: Tuned - waist only...
Evaluating: Tuned - BMI only...


threshold  balanced_accuracy  \
model               model_stage                                           
Logistic Regression Tuned - waist only        0.4444             0.7459   
                    Tuned - BMI and waist     0.4362             0.7462   
                    Tuned - BMI only          0.5074             0.7421   
                    Tuned                     0.4369             0.7475   

                                           precision  recall  specificity  \
model               model_stage                                             
Logistic Regression Tuned - waist only        0.2125  0.7793       0.7125   
                    Tuned - BMI and waist     0.2099  0.7875       0.7049   
                    Tuned - BMI only          0.2320  0.7221       0.7621   
                    Tuned                     0.2124  0.7847       0.7103   

                                              f1     f2  roc_auc  pr_auc  \
model               model_stage                                            
Logistic Regression Tuned - waist only    0.3339 0.5082   0.8076  0.2995   
                    Tuned - BMI and waist 0.3314 0.5079   0.8079  0.2988   
                    Tuned - BMI only      0.3512 0.5077   0.8076  0.2972   
                    Tuned                 0.3343 0.5099   0.8082  0.2961   

                                           predicted_positive_cases  
model               model_stage                                      
Logistic Regression Tuned - waist only                         1346  
                    Tuned - BMI and waist                      1377  
                    Tuned - BMI only                           1142  
                    Tuned                                      1356

Feature-configuration comparison completed.
Configurations ranked by PR-AUC.
Ordinary accuracy included: False
Test set evaluated: False


### Model-Based Feature Selection

L1-regularized Logistic Regression is used to identify informative transformed predictors. L1 regularization can reduce unhelpful feature coefficients to zero.

Feature selection is performed inside each cross-validation training fold to prevent information leakage. The selected features are then passed to the tuned, class-balanced Logistic Regression model.

In [141]:
# Step: Evaluate correlated-feature configurations
# Four configurations are compared:
# 1. All 29 predictors
# 2. Without BMI
# 3. Without waist circumference
# 4. Without both BMI and waist circumference
#
# The same Logistic Regression classifier and preprocessing logic are used
# for every configuration. The test set remains untouched.

from sklearn.base import clone
from sklearn.pipeline import Pipeline as SklearnPipeline


# Confirm that the correlated predictors exist in the training data.
correlated_features = ["BMXBMI", "BMXWAIST"]

missing_correlated_features = [
    feature
    for feature in correlated_features
    if feature not in X_train.columns
]

if missing_correlated_features:
    raise ValueError(
        f"Missing correlated predictors: {missing_correlated_features}"
    )


# Create a pipeline that preserves the original preprocessing steps
# while retaining only the columns available in each configuration.
def build_logistic_pipeline_for_columns(selected_columns):

    selected_columns = set(selected_columns)

    # Copy the original Logistic Regression preprocessor.
    reduced_preprocessor = clone(
        initial_logistic_pipeline.named_steps["preprocessor"]
    )

    # Update the column assignments without changing the transformations.
    reduced_transformers = []

    for transformer_name, transformer, transformer_columns in (
        initial_logistic_pipeline
        .named_steps["preprocessor"]
        .transformers
    ):
        retained_columns = [
            column
            for column in transformer_columns
            if column in selected_columns
        ]

        if retained_columns:
            reduced_transformers.append(
                (
                    transformer_name,
                    clone(transformer),
                    retained_columns
                )
            )

    reduced_preprocessor.transformers = reduced_transformers

    # Copy the original class-balanced Logistic Regression classifier.
    reduced_classifier = clone(
        initial_logistic_pipeline.named_steps["classifier"]
    )

    # Return the complete leakage-safe pipeline.
    return SklearnPipeline(
        steps=[
            ("preprocessor", reduced_preprocessor),
            ("classifier", reduced_classifier)
        ]
    )


# Define the feature-removal experiments.
correlated_feature_configurations = {
    "All 29 predictors": [],
    "Without BMI": ["BMXBMI"],
    "Without waist circumference": ["BMXWAIST"],
    "Without BMI and waist": ["BMXBMI", "BMXWAIST"]
}


# Store every pipeline, result, and set of out-of-fold probabilities.
correlated_feature_pipelines = {
    "All 29 predictors": initial_logistic_pipeline
}

correlated_feature_results = {}
correlated_feature_probabilities = {}


# Reuse the existing result for the complete 29-predictor model.
all_predictor_result = initial_logistic_results.copy()
all_predictor_result["configuration"] = "All 29 predictors"
all_predictor_result["predictor_count"] = X_train.shape[1]

correlated_feature_results["All 29 predictors"] = (
    all_predictor_result
)

correlated_feature_probabilities["All 29 predictors"] = (
    initial_logistic_oof_probabilities
)


# Evaluate each reduced feature configuration.
for configuration_name, removed_features in (
    correlated_feature_configurations.items()
):

    # The complete model was already evaluated above.
    if configuration_name == "All 29 predictors":
        continue

    # Remove only the predictors assigned to this experiment.
    X_train_configuration = X_train.drop(
        columns=removed_features
    )

    # Build a pipeline containing only the retained predictors.
    configuration_pipeline = (
        build_logistic_pipeline_for_columns(
            X_train_configuration.columns
        )
    )

    # Generate out-of-fold probabilities and select the F2 threshold.
    configuration_result, configuration_probabilities = (
        create_oof_evaluation(
            model_name="Logistic Regression",
            pipeline=configuration_pipeline,
            X_training=X_train_configuration,
            y_training=y_train
        )
    )

    # Add labels for the comparison table.
    configuration_result["configuration"] = configuration_name
    configuration_result["predictor_count"] = (
        X_train_configuration.shape[1]
    )

    # Save the pipeline and evaluation outputs.
    correlated_feature_pipelines[configuration_name] = (
        configuration_pipeline
    )

    correlated_feature_results[configuration_name] = (
        configuration_result
    )

    correlated_feature_probabilities[configuration_name] = (
        configuration_probabilities
    )


# Combine all four configurations into one comparison table.
correlated_feature_comparison = pd.DataFrame(
    correlated_feature_results.values()
)

# Accuracy is intentionally excluded.
correlated_feature_comparison = correlated_feature_comparison[
    [
        "configuration",
        "predictor_count",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("configuration")

# Sort by recall, the primary screening metric.
correlated_feature_comparison = (
    correlated_feature_comparison
    .sort_values(by="recall", ascending=False)
)

display(correlated_feature_comparison.round(4))


# Identify the strongest configurations.
best_feature_configuration_by_recall = (
    correlated_feature_comparison["recall"].idxmax()
)

best_feature_configuration_by_f2 = (
    correlated_feature_comparison["f2"].idxmax()
)

best_feature_configuration_by_pr_auc = (
    correlated_feature_comparison["pr_auc"].idxmax()
)


print("Correlated-feature analysis completed.")
print(
    "Best configuration by recall:",
    best_feature_configuration_by_recall
)
print(
    "Best configuration by F2:",
    best_feature_configuration_by_f2
)
print(
    "Best configuration by PR-AUC:",
    best_feature_configuration_by_pr_auc
)
print("Primary selection metric: Recall")
print("Threshold method: Maximum out-of-fold F2")
print("Test set used: False")

,predictor_count,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
configuration,,,,,,,,,,
All 29 predictors,29,0.4125,0.7480,0.2063,0.8038,0.6922,0.3283,0.5090,0.8077,0.2955
Without waist circumference,28,0.4384,0.7465,0.2111,0.7847,0.7082,0.3328,0.5085,0.8071,0.2986
Without BMI,28,0.5107,0.7432,0.2337,0.7221,0.7643,0.3531,0.5092,0.8071,0.2967
Without BMI and waist,27,0.5140,0.7418,0.2330,0.7193,0.7643,0.3520,0.5075,0.8075,0.2985


Correlated-feature analysis completed.
Best configuration by recall: All 29 predictors
Best configuration by F2: Without BMI
Best configuration by PR-AUC: Without waist circumference
Primary selection metric: Recall
Threshold method: Maximum out-of-fold F2
Test set used: False


In [142]:
# Apply model-based feature selection to the best feature configuration

from sklearn.feature_selection import SelectFromModel

best_numeric_columns = [
    column
    for column in numeric_columns
    if column != "BMXWAIST"
]

best_feature_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            best_numeric_columns,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

feature_selector = SelectFromModel(
    estimator=LogisticRegression(
        C=0.10,
        penalty="l1",
        solver="liblinear",
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE,
    ),
    threshold="median",
)

logistic_feature_selection_pipeline = Pipeline(
    steps=[
        ("preprocessor", best_feature_preprocessor),
        ("feature_selector", feature_selector),
        (
            "classifier",
            LogisticRegression(
                C=logistic_search.best_params_["classifier__C"],
                penalty=logistic_search.best_params_[
                    "classifier__penalty"
                ],
                solver=logistic_search.best_params_[
                    "classifier__solver"
                ],
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("Evaluating Logistic Regression with feature selection...")

(
    logistic_selection_results,
    logistic_selection_oof_probabilities,
) = create_oof_evaluation(
    model_name="Logistic Regression",
    pipeline=logistic_feature_selection_pipeline,
    X_training=X_train,
    y_training=y_train,
)

logistic_selection_results["model_stage"] = (
    "Tuned - waist + L1 selection"
)

# The waist-only result was the first feature experiment
logistic_waist_only_results = additional_feature_results[0]

selection_comparison = pd.DataFrame(
    [
        logistic_tuned_results,
        logistic_waist_only_results,
        logistic_selection_results,
    ]
)

display(
    selection_comparison[comparison_columns]
    .sort_values("pr_auc", ascending=False)
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Feature-selection experiment completed.")
print("Waist circumference removed: True")
print("Feature selection performed inside CV folds: True")
print("Primary comparison metric: PR-AUC")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating Logistic Regression with feature selection...


threshold  \
model               model_stage                               
Logistic Regression Tuned - waist only               0.4444   
                    Tuned                            0.4369   
                    Tuned - waist + L1 selection     0.4342   

                                                  balanced_accuracy  \
model               model_stage                                       
Logistic Regression Tuned - waist only                       0.7459   
                    Tuned                                    0.7475   
                    Tuned - waist + L1 selection             0.7458   

                                                  precision  recall  \
model               model_stage                                       
Logistic Regression Tuned - waist only               0.2125  0.7793   
                    Tuned                            0.2124  0.7847   
                    Tuned - waist + L1 selection     0.2085  0.7902   

                                                  specificity     f1     f2  \
model               model_stage                                               
Logistic Regression Tuned - waist only                 0.7125 0.3339 0.5082   
                    Tuned                              0.7103 0.3343 0.5099   
                    Tuned - waist + L1 selection       0.7014 0.3299 0.5072   

                                                  roc_auc  pr_auc  \
model               model_stage                                     
Logistic Regression Tuned - waist only             0.8076  0.2995   
                    Tuned                          0.8082  0.2961   
                    Tuned - waist + L1 selection   0.8068  0.2910   

                                                  predicted_positive_cases  
model               model_stage                                             
Logistic Regression Tuned - waist only                                1346  
                    Tuned                                             1356  
                    Tuned - waist + L1 selection                      1391

Feature-selection experiment completed.
Waist circumference removed: True
Feature selection performed inside CV folds: True
Primary comparison metric: PR-AUC
Ordinary accuracy included: False
Test set evaluated: False


## Correlated-Feature Reduction Conclusion

BMI and waist circumference were evaluated because exploratory analysis showed that they were strongly correlated. The alternative predictor configurations were tested using class-balanced Logistic Regression, stratified cross-validation, and out-of-fold F2 threshold optimization.

Removing BMI, waist circumference, or both did not produce a meaningful overall improvement. The complete 29-predictor configuration achieved the highest recall of **0.8038** and the highest balanced accuracy of **0.7480**. It also maintained an ROC-AUC of **0.8077** and passed the specificity guardrail.

Although some reduced configurations produced slightly higher F2 or PR-AUC values, those differences were very small and were accompanied by lower recall. Because recall is the primary model-selection metric for this screening application, all 29 predictors were retained.

The L1 feature-selection experiment also did not provide a meaningful improvement, so automated feature reduction was not retained.

## Final Logistic Regression Overfitting Check

The retained Logistic Regression uses all 29 predictors, balanced class weights, and StandardScaler for the numerical predictors. Its selected probability threshold is **0.4125**, determined from out-of-fold training predictions by maximizing F2.

Potential overfitting is evaluated by comparing training performance with out-of-fold cross-validation performance. Recall is the primary evaluation metric, while ROC-AUC is the secondary metric. Specificity and balanced accuracy are used as performance guardrails.

A large training-to-validation performance gap would suggest that the model does not generalize consistently. The untouched test set is not used during this analysis.

In [143]:
# Step: Check the final Logistic Regression for overfitting
# Selected configuration:
# - All 29 predictors
# - Balanced class weights
# - Out-of-fold F2 threshold optimization

# Retain the selected pipeline, results, probabilities, and threshold.
selected_logistic_pipeline = clone(initial_logistic_pipeline)
selected_logistic_results = initial_logistic_results.copy()

selected_logistic_oof_probabilities = (
    initial_logistic_oof_probabilities.copy()
)

selected_logistic_threshold = (
    selected_logistic_results["threshold"]
)

# Fit a fresh copy using the complete training data.
# The test set remains untouched.
selected_logistic_pipeline.fit(X_train, y_train)

# Generate in-sample probabilities for the training observations.
logistic_training_probabilities = (
    selected_logistic_pipeline.predict_proba(X_train)[:, 1]
)

# Evaluate training performance using the same threshold selected
# from the out-of-fold training predictions.
logistic_training_results = evaluate_probabilities(
    y_true=y_train,
    probabilities=logistic_training_probabilities,
    threshold=selected_logistic_threshold
)

# Create copies for the comparison table.
training_results = logistic_training_results.copy()
validation_results = selected_logistic_results.copy()

training_results["evaluation_set"] = "Training"
validation_results["evaluation_set"] = "Out-of-Fold Validation"

# Combine training and validation performance.
logistic_overfitting_table = pd.DataFrame([
    training_results,
    validation_results
])

# Accuracy is intentionally excluded.
overfitting_metrics = [
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc"
]

logistic_overfitting_table = logistic_overfitting_table[
    ["evaluation_set"] + overfitting_metrics
].set_index("evaluation_set")

display(logistic_overfitting_table.round(4))

# Calculate training minus validation for every metric.
logistic_overfitting_gaps = (
    logistic_overfitting_table.loc["Training"]
    - logistic_overfitting_table.loc["Out-of-Fold Validation"]
)

logistic_overfitting_gap_table = pd.DataFrame({
    "training_minus_validation": logistic_overfitting_gaps,
    "absolute_gap": logistic_overfitting_gaps.abs()
})

display(logistic_overfitting_gap_table.round(4))

# Summarize the largest observed difference.
largest_gap_metric = (
    logistic_overfitting_gap_table["absolute_gap"].idxmax()
)

largest_gap_value = (
    logistic_overfitting_gap_table.loc[
        largest_gap_metric,
        "absolute_gap"
    ]
)

print("Final Logistic Regression overfitting check completed.")
print("Selected configuration: All 29 predictors + class weights")
print(f"Selected threshold: {selected_logistic_threshold:.4f}")
print(f"Largest metric gap: {largest_gap_metric}")
print(f"Largest absolute gap: {largest_gap_value:.4f}")
print("Test set used: False")

,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
evaluation_set,,,,,,,,
Training,0.7652,0.2119,0.8420,0.6884,0.3386,0.5280,0.8402,0.3254
Out-of-Fold Validation,0.7480,0.2063,0.8038,0.6922,0.3283,0.5090,0.8077,0.2955


,training_minus_validation,absolute_gap
balanced_accuracy,0.0172,0.0172
precision,0.0056,0.0056
recall,0.0381,0.0381
specificity,-0.0038,0.0038
f1,0.0103,0.0103
f2,0.0191,0.0191
roc_auc,0.0325,0.0325
pr_auc,0.0298,0.0298


Final Logistic Regression overfitting check completed.
Selected configuration: All 29 predictors + class weights
Selected threshold: 0.4125
Largest metric gap: recall
Largest absolute gap: 0.0381
Test set used: False


### Logistic Regression Overfitting Conclusion

The final class-balanced Logistic Regression with all 29 predictors showed similar training and out-of-fold validation performance.

The largest difference was observed in recall, with a gap of 0.0381. The ROC-AUC, PR-AUC, balanced-accuracy, and F2 gaps also remained below 0.05. These relatively small differences indicate no strong evidence of overfitting.

The final Logistic Regression configuration therefore retains all 29 predictors, balanced class weights, and the out-of-fold F2-optimized threshold of 0.4125.

In [44]:
# Build and evaluate the initial Support Vector Machine
# The SVM uses:
# - RBF kernel to capture nonlinear relationships
# - Feature scaling through the existing preprocessor
# - Balanced class weights to address the imbalanced target
# - Out-of-fold predictions for leakage-safe threshold selection

print("Evaluating initial Support Vector Machine...")


# Construct the initial SVM pipeline.
# All preprocessing is fitted separately inside each cross-validation fold.
svm_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            SVC(
                C=1.0,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            )
        )
    ]
)


# Generate out-of-fold probabilities using the training data.
# The probability threshold is selected by maximizing the F2 score.
(
    svm_initial_results,
    svm_initial_oof_probabilities
) = create_oof_evaluation(
    model_name="Support Vector Machine",
    pipeline=svm_initial_pipeline,
    X_training=X_train,
    y_training=y_train
)


# Label this model as the initial SVM configuration.
svm_initial_results["model_stage"] = "Initial"


# Display the evaluation metrics.
# The comparison columns exclude ordinary accuracy.
display(
    pd.DataFrame([svm_initial_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)


# Document the model-development decisions.
print("Initial Support Vector Machine evaluation completed.")
print("Kernel: Radial Basis Function")
print("Feature scaling: Applied through the preprocessing pipeline")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary selection metric: Recall")
print("Supporting metrics: F2, balanced accuracy, ROC-AUC, and PR-AUC")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating initial Support Vector Machine...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Support Vector Machine,Initial,0.0767,0.7214,0.1873,0.7793,0.6634,0.3020,0.4775,0.7763,0.2396,1527


Initial Support Vector Machine evaluation completed.
Kernel: Radial Basis Function
Feature scaling: Applied through the preprocessing pipeline
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Primary selection metric: Recall
Supporting metrics: F2, balanced accuracy, ROC-AUC, and PR-AUC
Ordinary accuracy included: False
Test set used: False


### Support Vector Machine Fine-Tuning

The initial Support Vector Machine uses an RBF kernel, balanced class weights, and scaled predictors. Its hyperparameters are fine-tuned using stratified cross-validation.

Average precision, equivalent to PR-AUC, is used during hyperparameter tuning because it evaluates probability ranking without depending on a fixed classification threshold. After tuning, out-of-fold probabilities are used to select the maximum-F2 threshold, and recall remains the primary metric for the final model comparison.

The test set is not used during tuning or threshold selection.

In [46]:
# Step: Fine-tune the Support Vector Machine
# C controls regularization.
# Gamma controls the influence of individual observations.
# Class weights test different levels of minority-class emphasis.

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold


# Create a separate stratified cross-validation strategy for tuning.
svm_tuning_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Use probability=False during the search to reduce processing time.
# Average precision can use the SVM decision function directly.
svm_tuning_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            SVC(
                kernel="rbf",
                probability=False,
                random_state=RANDOM_STATE
            )
        )
    ]
)


# Define the SVM hyperparameter search space.
svm_parameter_distributions = {
    "classifier__C": [
        0.05,
        0.1,
        0.25,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
        25.0
    ],
    "classifier__gamma": [
        "scale",
        "auto",
        0.001,
        0.005,
        0.01,
        0.05,
        0.1,
        0.5
    ],
    "classifier__class_weight": [
        "balanced",
        {0: 1, 1: 2},
        {0: 1, 1: 3},
        {0: 1, 1: 4},
        {0: 1, 1: 5}
    ]
}


# Search a reproducible sample of parameter combinations.
# PR-AUC is used here as a threshold-independent tuning metric.
svm_search = RandomizedSearchCV(
    estimator=svm_tuning_pipeline,
    param_distributions=svm_parameter_distributions,
    n_iter=40,
    scoring="average_precision",
    cv=svm_tuning_cv,
    refit=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    return_train_score=True,
    verbose=1
)


print("Fine-tuning Support Vector Machine...")

# Fit the search using training data only.
svm_search.fit(X_train, y_train)


# Create the final tuned pipeline from the best search result.
svm_tuned_pipeline = clone(svm_search.best_estimator_)

# Enable probability estimates for out-of-fold threshold optimization.
svm_tuned_pipeline.set_params(
    classifier__probability=True
)


# Evaluate the tuned SVM using out-of-fold probabilities.
(
    svm_tuned_results,
    svm_tuned_oof_probabilities
) = create_oof_evaluation(
    model_name="Support Vector Machine",
    pipeline=svm_tuned_pipeline,
    X_training=X_train,
    y_training=y_train
)


# Label the tuned model for comparison.
svm_tuned_results["model_stage"] = "Tuned"


# Display the tuned model results.
display(
    pd.DataFrame([svm_tuned_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)


print("Support Vector Machine fine-tuning completed.")
print("Best parameters:", svm_search.best_params_)
print(f"Best tuning PR-AUC: {svm_search.best_score_:.4f}")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Test set used: False")

Fine-tuning Support Vector Machine...
Fitting 5 folds for each of 40 candidates, totalling 200 fits


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Support Vector Machine,Tuned,0.1030,0.7430,0.2363,0.7166,0.7695,0.3554,0.5095,0.8070,0.2916,1113


Support Vector Machine fine-tuning completed.
Best parameters: {'classifier__gamma': 0.001, 'classifier__class_weight': {0: 1, 1: 5}, 'classifier__C': 25.0}
Best tuning PR-AUC: 0.3022
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Test set used: False


In [47]:
# Step: Compare the initial and tuned Support Vector Machine
# Recall is the primary screening metric.
# The other metrics show whether tuning improved the model's
# overall discrimination and balance.

# Create independent copies of both result dictionaries.
svm_initial_comparison = svm_initial_results.copy()
svm_tuned_comparison = svm_tuned_results.copy()

# Assign clear configuration labels.
svm_initial_comparison["configuration"] = "Initial + Class Weight"
svm_tuned_comparison["configuration"] = "Tuned + Class Weight"

# Combine the results into one comparison table.
svm_comparison_table = pd.DataFrame([
    svm_initial_comparison,
    svm_tuned_comparison
])

# Accuracy is intentionally excluded.
svm_comparison_table = svm_comparison_table[
    [
        "configuration",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("configuration")

display(svm_comparison_table.round(4))


# Calculate tuned-minus-initial changes.
svm_metric_changes = (
    svm_comparison_table.loc["Tuned + Class Weight"]
    - svm_comparison_table.loc["Initial + Class Weight"]
)

svm_change_table = pd.DataFrame({
    "tuned_minus_initial": svm_metric_changes
})

display(svm_change_table.round(4))


# Identify the leading configuration for important metrics.
best_svm_by_recall = svm_comparison_table["recall"].idxmax()
best_svm_by_f2 = svm_comparison_table["f2"].idxmax()
best_svm_by_pr_auc = svm_comparison_table["pr_auc"].idxmax()


print("Initial and tuned SVM comparison completed.")
print(f"Best configuration by recall: {best_svm_by_recall}")
print(f"Best configuration by F2: {best_svm_by_f2}")
print(f"Best configuration by PR-AUC: {best_svm_by_pr_auc}")
print("Final SVM configuration not selected yet.")
print("Next experiment: Tuned SVM with SMOTE")
print("Test set used: False")

,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
configuration,,,,,,,,,
Initial + Class Weight,0.0767,0.7214,0.1873,0.7793,0.6634,0.3020,0.4775,0.7763,0.2396
Tuned + Class Weight,0.1030,0.7430,0.2363,0.7166,0.7695,0.3554,0.5095,0.8070,0.2916


,tuned_minus_initial
threshold,0.0262
balanced_accuracy,0.0217
precision,0.0490
recall,-0.0627
specificity,0.1060
f1,0.0534
f2,0.0320
roc_auc,0.0307
pr_auc,0.0520


Initial and tuned SVM comparison completed.
Best configuration by recall: Initial + Class Weight
Best configuration by F2: Tuned + Class Weight
Best configuration by PR-AUC: Tuned + Class Weight
Final SVM configuration not selected yet.
Next experiment: Tuned SVM with SMOTE
Test set used: False


### Tuned Support Vector Machine with SMOTENC

The tuned Support Vector Machine improved balanced accuracy, precision, specificity, F2, ROC-AUC, and PR-AUC, but recall decreased.

A categorical-aware SMOTE experiment is therefore conducted to determine whether synthetic minority-class sampling can recover recall while retaining the tuned SVM's stronger overall discrimination.

SMOTENC is applied only inside each cross-validation training fold. Class weights are disabled in this configuration to avoid correcting the class imbalance twice. The test set remains untouched.

In [49]:
# Step: Evaluate the tuned SVM with SMOTENC
# Missing values are imputed inside each cross-validation training fold
# before synthetic observations are generated.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline


# Create a preprocessing step used only before SMOTENC.
# It imputes missing values but does not scale or one-hot encode yet.
#
# Keeping this step inside the pipeline prevents data leakage.
pre_smote_imputer = ColumnTransformer(
    transformers=[
        (
            "numeric_imputation",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical_imputation",
            SimpleImputer(strategy="most_frequent"),
            categorical_columns
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Return a pandas DataFrame so SMOTENC can identify categorical
# predictors by their column names.
pre_smote_imputer.set_output(transform="pandas")


# Copy the tuned SVM classifier.
svm_smote_classifier = clone(
    svm_tuned_pipeline.named_steps["classifier"]
)

# Disable class weights because SMOTENC will handle imbalance.
# Using both could overcorrect the minority class.
svm_smote_classifier.set_params(
    class_weight=None,
    probability=True,
    random_state=RANDOM_STATE
)


# Construct the leakage-safe pipeline:
# 1. Impute missing values inside each training fold.
# 2. Generate synthetic minority observations with SMOTENC.
# 3. Apply the regular scaling and one-hot encoding.
# 4. Fit the tuned SVM.
svm_smote_pipeline = ImbPipeline(
    steps=[
        (
            "pre_smote_imputer",
            pre_smote_imputer
        ),
        (
            "smote",
            SMOTENC(
                categorical_features=categorical_columns,
                sampling_strategy="auto",
                k_neighbors=5,
                random_state=RANDOM_STATE
            )
        ),
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            svm_smote_classifier
        )
    ]
)


print("Evaluating tuned Support Vector Machine with SMOTENC...")


# Generate leakage-safe out-of-fold probabilities.
(
    svm_smote_results,
    svm_smote_oof_probabilities
) = create_oof_evaluation(
    model_name="Support Vector Machine",
    pipeline=svm_smote_pipeline,
    X_training=X_train,
    y_training=y_train
)


# Label this configuration for comparison.
svm_smote_results["model_stage"] = "Tuned + SMOTENC"


# Display the results.
display(
    pd.DataFrame([svm_smote_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)


print("SMOTENC SVM evaluation completed.")
print("Missing-value handling: Inside each cross-validation fold")
print("Imbalance handling: SMOTENC inside each training fold")
print("Class weights used: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating tuned Support Vector Machine with SMOTENC...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Support Vector Machine,Tuned + SMOTENC,0.1690,0.6977,0.1723,0.7575,0.6379,0.2808,0.4512,0.7511,0.2270,1613


SMOTENC SVM evaluation completed.
Missing-value handling: Inside each cross-validation fold
Imbalance handling: SMOTENC inside each training fold
Class weights used: False
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Ordinary accuracy included: False
Test set used: False


In [50]:
# Step: Compare all Support Vector Machine configurations
# The comparison includes:
# 1. Initial SVM with balanced class weights
# 2. Tuned SVM with class weights
# 3. Tuned SVM with SMOTENC

# Create independent copies of the stored results.
svm_initial_final = svm_initial_results.copy()
svm_tuned_final = svm_tuned_results.copy()
svm_smote_final = svm_smote_results.copy()

# Assign clear configuration labels.
svm_initial_final["configuration"] = "Initial + Class Weight"
svm_tuned_final["configuration"] = "Tuned + Class Weight"
svm_smote_final["configuration"] = "Tuned + SMOTENC"

# Combine the three configurations.
svm_final_comparison = pd.DataFrame([
    svm_initial_final,
    svm_tuned_final,
    svm_smote_final
])

# Accuracy is intentionally excluded.
svm_final_comparison = svm_final_comparison[
    [
        "configuration",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("configuration")

# Sort by recall, the primary screening metric.
svm_final_comparison = svm_final_comparison.sort_values(
    by="recall",
    ascending=False
)

display(svm_final_comparison.round(4))


# Identify the leading configuration for each important metric.
best_svm_recall_configuration = (
    svm_final_comparison["recall"].idxmax()
)

best_svm_f2_configuration = (
    svm_final_comparison["f2"].idxmax()
)

best_svm_roc_auc_configuration = (
    svm_final_comparison["roc_auc"].idxmax()
)

best_svm_pr_auc_configuration = (
    svm_final_comparison["pr_auc"].idxmax()
)


print("Final SVM configuration comparison completed.")
print(
    "Best configuration by recall:",
    best_svm_recall_configuration
)
print(
    "Best configuration by F2:",
    best_svm_f2_configuration
)
print(
    "Best configuration by ROC-AUC:",
    best_svm_roc_auc_configuration
)
print(
    "Best configuration by PR-AUC:",
    best_svm_pr_auc_configuration
)
print("SMOTENC retained: False")
print("Test set used: False")

,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
configuration,,,,,,,,,
Initial + Class Weight,0.0767,0.7214,0.1873,0.7793,0.6634,0.3020,0.4775,0.7763,0.2396
Tuned + SMOTENC,0.1690,0.6977,0.1723,0.7575,0.6379,0.2808,0.4512,0.7511,0.2270
Tuned + Class Weight,0.1030,0.7430,0.2363,0.7166,0.7695,0.3554,0.5095,0.8070,0.2916


Final SVM configuration comparison completed.
Best configuration by recall: Initial + Class Weight
Best configuration by F2: Tuned + Class Weight
Best configuration by ROC-AUC: Tuned + Class Weight
Best configuration by PR-AUC: Tuned + Class Weight
SMOTENC retained: False
Test set used: False


### Tuned SVM Recall-Constrained Threshold

The tuned class-weighted SVM achieved the strongest F2, ROC-AUC, PR-AUC, precision, specificity, and balanced accuracy, but its maximum-F2 threshold produced lower recall than the initial SVM.

A recall-constrained threshold is therefore selected from the tuned model's out-of-fold probabilities. The threshold must achieve at least the initial SVM recall, and the configuration with the highest precision among the eligible thresholds is selected.

This changes only the classification threshold; the model is not retrained. The test set remains untouched.

In [54]:
# Step: Select a recall-constrained threshold for the tuned SVM
# Target: achieve at least the initial SVM recall.
# Among eligible thresholds, select the one with the highest precision.

from sklearn.metrics import precision_recall_curve


# Use the initial SVM recall as the minimum acceptable sensitivity.
svm_target_recall = svm_initial_results["recall"]


# Calculate precision and recall across all possible thresholds
# using only tuned out-of-fold training probabilities.
(
    svm_precision_values,
    svm_recall_values,
    svm_threshold_values
) = precision_recall_curve(
    y_train,
    svm_tuned_oof_probabilities
)


# The final precision and recall entries do not have a corresponding
# threshold, so exclude them when identifying eligible thresholds.
eligible_svm_indices = np.where(
    svm_recall_values[:-1] >= svm_target_recall
)[0]


if len(eligible_svm_indices) == 0:
    raise ValueError(
        "The tuned SVM could not reach the target recall."
    )


# Among thresholds that meet the recall requirement,
# select the threshold with the highest precision.
best_eligible_position = np.argmax(
    svm_precision_values[:-1][eligible_svm_indices]
)

selected_svm_index = eligible_svm_indices[
    best_eligible_position
]

svm_recall_threshold = svm_threshold_values[
    selected_svm_index
]


# Evaluate the tuned SVM at the recall-constrained threshold.
svm_recall_threshold_results = evaluate_probabilities(
    y_true=y_train,
    probabilities=svm_tuned_oof_probabilities,
    threshold=svm_recall_threshold
)

svm_recall_threshold_results["model"] = (
    "Support Vector Machine"
)

svm_recall_threshold_results["model_stage"] = (
    "Tuned + Recall-Constrained Threshold"
)


# Compare the initial, tuned maximum-F2, and tuned recall-constrained results.
svm_threshold_comparison = pd.DataFrame([
    {
        **svm_initial_results,
        "configuration": "Initial + Maximum-F2 Threshold"
    },
    {
        **svm_tuned_results,
        "configuration": "Tuned + Maximum-F2 Threshold"
    },
    {
        **svm_recall_threshold_results,
        "configuration": "Tuned + Recall-Constrained Threshold"
    }
])


# Accuracy is intentionally excluded.
svm_threshold_comparison = svm_threshold_comparison[
    [
        "configuration",
        "threshold",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc"
    ]
].set_index("configuration")

display(svm_threshold_comparison.round(4))


print("Tuned SVM threshold analysis completed.")
print(f"Minimum required recall: {svm_target_recall:.4f}")
print(f"Selected threshold: {svm_recall_threshold:.4f}")
print(
    "Achieved recall:",
    f"{svm_recall_threshold_results['recall']:.4f}"
)
print(
    "Achieved precision:",
    f"{svm_recall_threshold_results['precision']:.4f}"
)
print("Threshold selected from out-of-fold training predictions.")
print("Test set used: False")

,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
configuration,,,,,,,,,
Initial + Maximum-F2 Threshold,0.0767,0.7214,0.1873,0.7793,0.6634,0.3020,0.4775,0.7763,0.2396
Tuned + Maximum-F2 Threshold,0.1030,0.7430,0.2363,0.7166,0.7695,0.3554,0.5095,0.8070,0.2916
Tuned + Recall-Constrained Threshold,0.0832,0.7441,0.2104,0.7793,0.7090,0.3314,0.5058,0.8070,0.2916


Tuned SVM threshold analysis completed.
Minimum required recall: 0.7793
Selected threshold: 0.0832
Achieved recall: 0.7793
Achieved precision: 0.2104
Threshold selected from out-of-fold training predictions.
Test set used: False


### Final SVM Configuration and Overfitting Check

The tuned class-weighted SVM with a recall-constrained threshold was selected as the final SVM configuration.

At the same recall as the initial SVM, the selected configuration achieved better balanced accuracy, precision, specificity, F1, F2, ROC-AUC, and PR-AUC. SMOTENC was not retained because it reduced overall performance.

The selected SVM is now evaluated for potential overfitting by comparing its training and out-of-fold validation performance. The test set remains untouched.

In [149]:
# Finalize the tuned class-weighted SVM and check for overfitting

print("Finalizing Support Vector Machine...")

# The Initial SVM produced the highest recall, but its large
# training-validation gaps indicated meaningful overfitting.
#
# Therefore, retain the tuned class-weighted SVM because it
# generalizes more consistently and passes the performance guardrails.
svm_final_pipeline = clone(svm_tuned_pipeline)
svm_final_results = svm_tuned_results.copy()
svm_final_oof_probabilities = (
    svm_tuned_oof_probabilities.copy()
)

# Use the threshold selected by maximizing F2 from the
# out-of-fold predictions. Do not use the recall-constrained threshold.
svm_final_threshold = float(
    svm_final_results["threshold"]
)

# Fit a fresh copy of the retained model on the complete training set.
# The test set remains untouched.
svm_overfitting_pipeline = clone(svm_final_pipeline)
svm_overfitting_pipeline.fit(X_train, y_train)

# Generate in-sample training probabilities.
svm_training_probabilities = (
    svm_overfitting_pipeline.predict_proba(X_train)[:, 1]
)

# Convert probabilities into class predictions using the
# selected maximum-F2 threshold.
svm_training_predictions = (
    svm_training_probabilities >= svm_final_threshold
).astype(int)

# Calculate the training confusion matrix.
tn, fp, fn, tp = confusion_matrix(
    y_train,
    svm_training_predictions,
).ravel()

# Calculate training specificity.
svm_training_specificity = tn / (tn + fp)

# Collect the in-sample training metrics.
svm_training_metrics = {
    "balanced_accuracy": balanced_accuracy_score(
        y_train,
        svm_training_predictions,
    ),
    "precision": precision_score(
        y_train,
        svm_training_predictions,
        zero_division=0,
    ),
    "recall": recall_score(
        y_train,
        svm_training_predictions,
    ),
    "specificity": svm_training_specificity,
    "f1": f1_score(
        y_train,
        svm_training_predictions,
    ),
    "f2": fbeta_score(
        y_train,
        svm_training_predictions,
        beta=2,
    ),
    "roc_auc": roc_auc_score(
        y_train,
        svm_training_probabilities,
    ),
    "pr_auc": average_precision_score(
        y_train,
        svm_training_probabilities,
    ),
}

# Retrieve the out-of-fold validation results already produced
# for the tuned class-weighted SVM.
svm_validation_metrics = {
    metric: float(svm_final_results[metric])
    for metric in [
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "f2",
        "roc_auc",
        "pr_auc",
    ]
}

# Compare training and out-of-fold validation performance.
svm_overfitting_comparison = pd.DataFrame(
    [
        svm_training_metrics,
        svm_validation_metrics,
    ],
    index=[
        "Training",
        "Out-of-Fold Validation",
    ],
)

display(
    svm_overfitting_comparison.round(4)
)

# Calculate the training-validation gap for every metric.
svm_overfitting_gaps = pd.DataFrame(
    {
        "training_minus_validation": {
            metric: (
                svm_training_metrics[metric]
                - svm_validation_metrics[metric]
            )
            for metric in svm_training_metrics
        }
    }
)

svm_overfitting_gaps["absolute_gap"] = (
    svm_overfitting_gaps[
        "training_minus_validation"
    ].abs()
)

display(
    svm_overfitting_gaps.round(4)
)

# Identify the largest absolute performance gap.
svm_largest_gap_metric = (
    svm_overfitting_gaps["absolute_gap"].idxmax()
)

svm_largest_gap = float(
    svm_overfitting_gaps.loc[
        svm_largest_gap_metric,
        "absolute_gap",
    ]
)

# A gap greater than 0.10 indicates possible meaningful overfitting.
svm_meaningful_overfitting = (
    svm_largest_gap > 0.10
)

# Confirm that the validation results pass the project guardrails.
svm_passes_guardrails = (
    svm_final_results["balanced_accuracy"] >= 0.70
    and svm_final_results["specificity"] >= 0.65
)

print("\nFinal Support Vector Machine assigned successfully.")
print("Selected configuration: Tuned + Class Weight")
print("Numerical scaling included: True")
print("Class weighting retained: True")
print("SMOTE retained: False")
print("Threshold method: Maximum out-of-fold F2")
print(
    "Selected threshold:",
    round(svm_final_threshold, 4),
)
print(
    "Recall:",
    round(svm_final_results["recall"], 4),
)
print(
    "ROC-AUC:",
    round(svm_final_results["roc_auc"], 4),
)
print(
    "Specificity:",
    round(svm_final_results["specificity"], 4),
)
print(
    "Balanced accuracy:",
    round(
        svm_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "Passes performance guardrails:",
    svm_passes_guardrails,
)
print(
    "Largest metric gap:",
    svm_largest_gap_metric,
)
print(
    "Largest absolute gap:",
    round(svm_largest_gap, 4),
)
print(
    "Meaningful overfitting:",
    svm_meaningful_overfitting,
)
print("Primary presentation metric: Recall")
print("Secondary presentation metric: ROC-AUC")
print("Test set evaluated: False")

Finalizing Support Vector Machine...


,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc
Training,0.7818,0.2628,0.7820,0.7817,0.3934,0.5605,0.8531,0.3637
Out-of-Fold Validation,0.7430,0.2363,0.7166,0.7695,0.3554,0.5095,0.8070,0.2916


,training_minus_validation,absolute_gap
balanced_accuracy,0.0388,0.0388
precision,0.0265,0.0265
recall,0.0654,0.0654
specificity,0.0122,0.0122
f1,0.0380,0.0380
f2,0.0511,0.0511
roc_auc,0.0461,0.0461
pr_auc,0.0721,0.0721



Final Support Vector Machine assigned successfully.
Selected configuration: Tuned + Class Weight
Numerical scaling included: True
Class weighting retained: True
SMOTE retained: False
Threshold method: Maximum out-of-fold F2
Selected threshold: 0.103
Recall: 0.7166
ROC-AUC: 0.807
Specificity: 0.7695
Balanced accuracy: 0.743
Passes performance guardrails: True
Largest metric gap: pr_auc
Largest absolute gap: 0.0721
Meaningful overfitting: False
Primary presentation metric: Recall
Secondary presentation metric: ROC-AUC
Test set evaluated: False


## Support Vector Machine Conclusion

Three SVM configurations were evaluated: Initial with balanced class weights, Tuned with balanced class weights, and Tuned with SMOTE. Numerical predictors were standardized because SVM is sensitive to differences in feature scale.

The Initial class-weighted SVM achieved the highest recall of **0.7793**, but the overfitting analysis revealed large training-to-validation gaps, including a PR-AUC gap of **0.4450** and a ROC-AUC gap of **0.1997**. Therefore, the Initial SVM was not retained.

The SMOTE configuration achieved a recall of **0.7575**, but its specificity of **0.6379** and balanced accuracy of **0.6977** failed the project guardrails. Therefore, SMOTE was also not retained.

The Tuned class-weighted SVM provided the most reliable performance. Using the maximum out-of-fold F2 threshold of **0.1030**, it achieved:

- Recall: **0.7166**
- ROC-AUC: **0.8070**
- Specificity: **0.7695**
- Balanced accuracy: **0.7430**
- PR-AUC: **0.2916**

Its largest training-to-validation gap was **0.0721**, which was below the **0.10** overfitting threshold. Therefore, no meaningful overfitting was detected.

The **Tuned SVM with balanced class weights** was retained for the final model comparison. The test set remained untouched.

 ## Decision Tree

A Decision Tree is evaluated as an interpretable nonlinear model. Balanced class weights are used to address the class imbalance. Its initial performance is measured with out-of-fold predictions before tuning tree depth, split size, leaf size, and pruning strength.

In [61]:
# Build and evaluate the initial Decision Tree
# This is an unrestricted baseline tree.
# It will later be tuned and regularized to reduce overfitting.

print("Evaluating initial Decision Tree...")


# Construct the initial Decision Tree pipeline.
# Preprocessing remains inside the pipeline to prevent leakage
# and keep model evaluation consistent.
decision_tree_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ]
)


# Generate out-of-fold probabilities using stratified cross-validation.
# The untouched test set is not used.
(
    decision_tree_initial_results,
    decision_tree_initial_oof_probabilities
) = create_oof_evaluation(
    model_name="Decision Tree",
    pipeline=decision_tree_initial_pipeline,
    X_training=X_train,
    y_training=y_train
)


# Label this model as the initial Decision Tree.
decision_tree_initial_results["model_stage"] = "Initial"


# Display the evaluation metrics.
# Ordinary accuracy is excluded from the comparison.
display(
    pd.DataFrame([decision_tree_initial_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)


# Document the evaluation decisions.
print("Initial Decision Tree evaluation completed.")
print("Tree restrictions: None — initial baseline")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary selection metric: Recall")
print("Supporting metrics: F2, balanced accuracy, ROC-AUC, and PR-AUC")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating initial Decision Tree...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Decision Tree,Initial,0.0000,0.5000,0.0905,1.0000,0.0000,0.1660,0.3323,0.5578,0.1104,4054


Initial Decision Tree evaluation completed.
Tree restrictions: None — initial baseline
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Primary selection metric: Recall
Supporting metrics: F2, balanced accuracy, ROC-AUC, and PR-AUC
Ordinary accuracy included: False
Test set used: False


### Decision Tree Fine-Tuning and Regularization

The unrestricted Decision Tree produced a degenerate maximum-F2 threshold of 0.0000. At this threshold, it classified essentially every participant as elevated risk, resulting in recall of 1.0000 but specificity of 0.0000 and balanced accuracy of 0.5000.

The Decision Tree is therefore fine-tuned and regularized using constraints on tree depth, split size, leaf size, feature availability, and cost-complexity pruning.

PR-AUC is used during hyperparameter tuning because it measures minority-class ranking without relying on a fixed threshold. Recall remains the primary final-selection metric but is interpreted together with balanced accuracy and specificity to prevent an all-positive classifier from being selected.

In [62]:
# Step: Fine-tune and regularize the Decision Tree
# The search controls tree complexity to reduce overfitting
# and improve minority-class probability ranking.

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold


# Use stratified folds to preserve the class distribution.
decision_tree_tuning_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Construct the tuning pipeline.
decision_tree_tuning_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)


# Define regularized Decision Tree configurations.
decision_tree_parameter_distributions = {
    "classifier__criterion": [
        "gini",
        "entropy",
        "log_loss"
    ],
    "classifier__max_depth": [
        3,
        4,
        5,
        6,
        8,
        10,
        12,
        None
    ],
    "classifier__min_samples_split": [
        10,
        20,
        30,
        50,
        75,
        100
    ],
    "classifier__min_samples_leaf": [
        5,
        10,
        20,
        30,
        50,
        75,
        100
    ],
    "classifier__max_features": [
        None,
        "sqrt",
        "log2",
        0.50,
        0.75
    ],
    "classifier__ccp_alpha": [
        0.0,
        0.0001,
        0.0005,
        0.001,
        0.002,
        0.005,
        0.01
    ],
    "classifier__class_weight": [
        "balanced",
        {0: 1, 1: 2},
        {0: 1, 1: 3},
        {0: 1, 1: 4},
        {0: 1, 1: 5}
    ]
}


# Search a reproducible sample of the parameter combinations.
decision_tree_search = RandomizedSearchCV(
    estimator=decision_tree_tuning_pipeline,
    param_distributions=decision_tree_parameter_distributions,
    n_iter=60,
    scoring="average_precision",
    cv=decision_tree_tuning_cv,
    refit=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    return_train_score=True,
    verbose=1
)


print("Fine-tuning and regularizing Decision Tree...")


# Use only the training data during tuning.
decision_tree_search.fit(X_train, y_train)


# Retain the best cross-validated Decision Tree pipeline.
decision_tree_tuned_pipeline = clone(
    decision_tree_search.best_estimator_
)


# Evaluate it using out-of-fold probabilities and F2 optimization.
(
    decision_tree_tuned_results,
    decision_tree_tuned_oof_probabilities
) = create_oof_evaluation(
    model_name="Decision Tree",
    pipeline=decision_tree_tuned_pipeline,
    X_training=X_train,
    y_training=y_train
)


# Label the model for comparison.
decision_tree_tuned_results["model_stage"] = "Tuned"


# Display the tuned results.
display(
    pd.DataFrame([decision_tree_tuned_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)


print("Decision Tree fine-tuning completed.")
print("Best parameters:", decision_tree_search.best_params_)
print(
    "Best tuning PR-AUC:",
    f"{decision_tree_search.best_score_:.4f}"
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrail applied: True")
print("Test set used: False")

Fine-tuning and regularizing Decision Tree...
Fitting 5 folds for each of 60 candidates, totalling 300 fits


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Decision Tree,Tuned,0.3378,0.7056,0.1863,0.7275,0.6838,0.2967,0.4602,0.7678,0.2456,1433


Decision Tree fine-tuning completed.
Best parameters: {'classifier__min_samples_split': 50, 'classifier__min_samples_leaf': 100, 'classifier__max_features': 0.5, 'classifier__max_depth': 10, 'classifier__criterion': 'entropy', 'classifier__class_weight': {0: 1, 1: 5}, 'classifier__ccp_alpha': 0.002}
Best tuning PR-AUC: 0.2410
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrail applied: True
Test set used: False


In [63]:
# Compare the initial and tuned Decision Tree models

decision_tree_comparison = pd.DataFrame(
    [
        decision_tree_initial_results,
        decision_tree_tuned_results,
    ]
)[comparison_columns]

display(
    decision_tree_comparison
    .set_index(["model", "model_stage"])
    .round(4)
)

# Check whether each model predicts both outcome classes
decision_tree_validity_check = pd.DataFrame(
    {
        "model_stage": ["Initial", "Tuned"],
        "predicts_both_classes": [
            (
                decision_tree_initial_results["recall"] < 1.0
                and decision_tree_initial_results["specificity"] > 0.0
            ),
            (
                decision_tree_tuned_results["recall"] < 1.0
                and decision_tree_tuned_results["specificity"] > 0.0
            ),
        ],
    }
).set_index("model_stage")

display(decision_tree_validity_check)

print("Decision Tree comparison completed.")
print("Initial model rejected: It predicted every participant as elevated risk.")
print("Tuned model retained: It provides a meaningful recall-specificity balance.")
print("Selected Decision Tree stage: Tuned")
print("Test set used: False")

threshold  balanced_accuracy  precision  recall  \
model         model_stage                                                    
Decision Tree Initial         0.0000             0.5000     0.0905  1.0000   
              Tuned           0.3378             0.7056     0.1863  0.7275   

                           specificity     f1     f2  roc_auc  pr_auc  \
model         model_stage                                               
Decision Tree Initial           0.0000 0.1660 0.3323   0.5578  0.1104   
              Tuned             0.6838 0.2967 0.4602   0.7678  0.2456   

                           predicted_positive_cases  
model         model_stage                            
Decision Tree Initial                          4054  
              Tuned                            1433

,predicts_both_classes
model_stage,
Initial,False
Tuned,True


Decision Tree comparison completed.
Initial model rejected: It predicted every participant as elevated risk.
Tuned model retained: It provides a meaningful recall-specificity balance.
Selected Decision Tree stage: Tuned
Test set used: False


### Decision Tree with SMOTENC

Because the dataset is imbalanced and contains both numerical and categorical
predictors, SMOTENC is evaluated as an alternative imbalance-handling method.

SMOTENC is applied only within each cross-validation training fold to prevent
data leakage. Class weights are disabled in this experiment because combining
balanced synthetic sampling with class weighting could overcorrect the class
imbalance.

The result will be compared with the tuned class-weighted Decision Tree. The
test set remains untouched.

In [70]:
# Evaluate the tuned Decision Tree with SMOTENC

print("Evaluating tuned Decision Tree with SMOTENC...")

# Prepare the numerical predictors
smotenc_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Prepare the categorical predictors
smotenc_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Place numerical predictors first and categorical predictors second
smotenc_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", smotenc_numeric_pipeline, numeric_columns),
        ("categorical", smotenc_categorical_pipeline, categorical_columns),
    ]
)

# Identify the categorical positions in the transformed dataset
smotenc_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns) + len(categorical_columns),
    )
)

# Copy the classifier from the completed tuned Decision Tree pipeline
decision_tree_smote_classifier = clone(
    decision_tree_tuned_pipeline.named_steps["classifier"]
)

# SMOTENC handles the imbalance, so class weighting is disabled
decision_tree_smote_classifier.set_params(class_weight=None)

# Keep SMOTENC inside the pipeline to prevent data leakage
decision_tree_smote_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", smotenc_preprocessor),
        (
            "smotenc",
            SMOTENC(
                categorical_features=smotenc_categorical_indices,
                sampling_strategy="auto",
                random_state=RANDOM_STATE,
                k_neighbors=5,
            ),
        ),
        ("classifier", decision_tree_smote_classifier),
    ]
)

# Generate out-of-fold probabilities and optimize the threshold
(
    decision_tree_smote_results,
    decision_tree_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="Decision Tree",
    pipeline=decision_tree_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

decision_tree_smote_results["model_stage"] = "Tuned + SMOTENC"

# Display the SMOTENC results
display(
    pd.DataFrame([decision_tree_smote_results])[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Decision Tree SMOTENC evaluation completed.")
print("Imbalance handling: SMOTENC inside cross-validation")
print("Class weighting used: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrails applied: True")
print("Test set used: False")

Evaluating tuned Decision Tree with SMOTENC...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Decision Tree,Tuned + SMOTENC,0.4444,0.6847,0.2178,0.5749,0.7944,0.3159,0.4329,0.7366,0.2202,969


Decision Tree SMOTENC evaluation completed.
Imbalance handling: SMOTENC inside cross-validation
Class weighting used: False
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Test set used: False


In [71]:
# Compare all Decision Tree configurations and select the final candidate

decision_tree_all_results = pd.DataFrame(
    [
        decision_tree_initial_results,
        decision_tree_tuned_results,
        decision_tree_smote_results,
    ]
)[comparison_columns]

display(
    decision_tree_all_results
    .set_index(["model", "model_stage"])
    .round(4)
)

# Select the tuned class-weighted Decision Tree
decision_tree_final_pipeline = clone(decision_tree_tuned_pipeline)
decision_tree_final_results = decision_tree_tuned_results.copy()
decision_tree_final_oof_probabilities = (
    decision_tree_tuned_oof_probabilities.copy()
)

decision_tree_final_results["model_stage"] = "Final selected"

print("Decision Tree configuration comparison completed.")
print("Initial model rejected: It predicted every participant as elevated risk.")
print("SMOTENC model rejected: Recall and overall discrimination decreased.")
print("Final Decision Tree selected: Tuned class-weighted Decision Tree")
print(
    "Selected threshold:",
    round(decision_tree_final_results["threshold"], 4),
)
print(
    "Selected recall:",
    round(decision_tree_final_results["recall"], 4),
)
print(
    "Selected specificity:",
    round(decision_tree_final_results["specificity"], 4),
)
print(
    "Selected balanced accuracy:",
    round(decision_tree_final_results["balanced_accuracy"], 4),
)
print("Test set used: False")

threshold  balanced_accuracy  precision  \
model         model_stage                                                
Decision Tree Initial             0.0000             0.5000     0.0905   
              Tuned               0.3378             0.7056     0.1863   
              Tuned + SMOTENC     0.4444             0.6847     0.2178   

                               recall  specificity     f1     f2  roc_auc  \
model         model_stage                                                   
Decision Tree Initial          1.0000       0.0000 0.1660 0.3323   0.5578   
              Tuned            0.7275       0.6838 0.2967 0.4602   0.7678   
              Tuned + SMOTENC  0.5749       0.7944 0.3159 0.4329   0.7366   

                               pr_auc  predicted_positive_cases  
model         model_stage                                        
Decision Tree Initial          0.1104                      4054  
              Tuned            0.2456                      1433  
              Tuned + SMOTENC  0.2202                       969

Decision Tree configuration comparison completed.
Initial model rejected: It predicted every participant as elevated risk.
SMOTENC model rejected: Recall and overall discrimination decreased.
Final Decision Tree selected: Tuned class-weighted Decision Tree
Selected threshold: 0.3378
Selected recall: 0.7275
Selected specificity: 0.6838
Selected balanced accuracy: 0.7056
Test set used: False


### Decision Tree Configuration Conclusion

Three Decision Tree configurations were evaluated: the initial model, the tuned
class-weighted model, and the tuned model with SMOTENC.

The initial model was rejected because it classified every participant as having
elevated risk. SMOTENC improved specificity but substantially reduced recall,
balanced accuracy, F2, ROC-AUC, and PR-AUC.

The tuned class-weighted Decision Tree provided the strongest overall screening
performance and was retained as the final Decision Tree candidate. The next step
is to check this selected configuration for overfitting.

In [72]:
# Check the selected Decision Tree for overfitting

print("Checking the selected Decision Tree for overfitting...")

# Use the threshold selected from out-of-fold training predictions
decision_tree_selected_threshold = decision_tree_final_results["threshold"]

# Fit a fresh copy of the selected pipeline on the complete training set
decision_tree_overfit_pipeline = clone(decision_tree_final_pipeline)

decision_tree_overfit_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample training probabilities
decision_tree_training_probabilities = (
    decision_tree_overfit_pipeline.predict_proba(X_train)[:, 1]
)

# Evaluate training probabilities using the selected OOF threshold
decision_tree_training_metrics = evaluate_probabilities(
    y_true=y_train,
    probabilities=decision_tree_training_probabilities,
    threshold=decision_tree_selected_threshold,
)

# Recalculate OOF metrics using exactly the same threshold
decision_tree_oof_metrics = evaluate_probabilities(
    y_true=y_train,
    probabilities=decision_tree_final_oof_probabilities,
    threshold=decision_tree_selected_threshold,
)

# Metrics used to inspect generalization
decision_tree_overfit_metrics = [
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
]

# Build the training-versus-OOF comparison
decision_tree_overfit_comparison = pd.DataFrame(
    {
        "Training": {
            metric: decision_tree_training_metrics[metric]
            for metric in decision_tree_overfit_metrics
        },
        "Out-of-fold": {
            metric: decision_tree_oof_metrics[metric]
            for metric in decision_tree_overfit_metrics
        },
    }
)

# Calculate the generalization gap
decision_tree_overfit_comparison["Absolute gap"] = (
    decision_tree_overfit_comparison["Training"]
    - decision_tree_overfit_comparison["Out-of-fold"]
).abs()

display(decision_tree_overfit_comparison.round(4))

# Summarize the largest observed gap
decision_tree_largest_gap_metric = (
    decision_tree_overfit_comparison["Absolute gap"].idxmax()
)

decision_tree_largest_gap = decision_tree_overfit_comparison.loc[
    decision_tree_largest_gap_metric,
    "Absolute gap",
]

print("Decision Tree overfitting check completed.")
print(
    "Largest performance gap:",
    decision_tree_largest_gap_metric,
    round(decision_tree_largest_gap, 4),
)
print(
    "Selected threshold:",
    round(decision_tree_selected_threshold, 4),
)
print("Training and OOF evaluated at the same threshold: True")
print("Test set used: False")

Checking the selected Decision Tree for overfitting...


,Training,Out-of-fold,Absolute gap
balanced_accuracy,0.7440,0.7056,0.0384
precision,0.2066,0.1863,0.0202
recall,0.7902,0.7275,0.0627
specificity,0.6979,0.6838,0.0141
f1,0.3275,0.2967,0.0308
f2,0.5049,0.4602,0.0447
roc_auc,0.8134,0.7678,0.0456
pr_auc,0.2547,0.2456,0.0091


Decision Tree overfitting check completed.
Largest performance gap: recall 0.0627
Selected threshold: 0.3378
Training and OOF evaluated at the same threshold: True
Test set used: False


## Decision Tree Conclusion

Three Decision Tree configurations were evaluated: Initial, Tuned, and Tuned with SMOTENC. Decision Trees do not require numerical feature scaling because their predictions are based on decision rules rather than distances between values.

The Initial Decision Tree selected a threshold of **0.0000**, producing a recall of **1.0000** but a specificity of **0.0000**. This means it classified essentially every participant as elevated risk, so it was not a useful screening model.

The SMOTENC configuration achieved a recall of **0.5749** and balanced accuracy of **0.6847**. Because it reduced recall and failed the balanced-accuracy guardrail, SMOTENC was not retained.

The Tuned Decision Tree provided the most reliable overall performance. Using the maximum out-of-fold F2 threshold of **0.3378**, it achieved:

- Recall: **0.7275**
- ROC-AUC: **0.7678**
- Specificity: **0.6838**
- Balanced accuracy: **0.7056**
- PR-AUC: **0.2456**

Its largest training-to-validation gap was **0.0627**, which was below the **0.10** overfitting threshold. Therefore, no meaningful overfitting was detected.

The **Tuned Decision Tree with balanced class weights** was retained for the final model comparison. The test set remained untouched.

In [73]:
# Build and evaluate the initial Random Forest

print("Evaluating initial Random Forest...")

random_forest_initial_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

(
    random_forest_initial_results,
    random_forest_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="Random Forest",
    pipeline=random_forest_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

random_forest_initial_results["model_stage"] = "Initial"

display(
    pd.DataFrame([random_forest_initial_results])[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Initial Random Forest evaluation completed.")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrails applied: True")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating initial Random Forest...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Random Forest,Initial,0.2567,0.7401,0.2578,0.6730,0.8072,0.3728,0.5091,0.8013,0.2672,958


Initial Random Forest evaluation completed.
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Ordinary accuracy included: False
Test set used: False


In [74]:
# Fine-tune and evaluate the Random Forest

print("Fine-tuning Random Forest...")

# Build the Random Forest tuning pipeline
random_forest_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Search tree complexity, feature sampling, and imbalance handling
random_forest_parameter_distributions = {
    "classifier__n_estimators": [200, 300, 400, 500, 600],
    "classifier__max_depth": [4, 6, 8, 10, 12, 16, None],
    "classifier__min_samples_split": [2, 5, 10, 20, 40, 60],
    "classifier__min_samples_leaf": [1, 2, 5, 10, 20, 30],
    "classifier__max_features": ["sqrt", "log2", 0.5, 0.75],
    "classifier__bootstrap": [True],
    "classifier__class_weight": [
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 2},
        {0: 1, 1: 3},
        {0: 1, 1: 4},
        {0: 1, 1: 5},
    ],
}

# Use a reproducible stratified cross-validation strategy
random_forest_tuning_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Tune probability-ranking performance under class imbalance
random_forest_search = RandomizedSearchCV(
    estimator=random_forest_tuning_pipeline,
    param_distributions=random_forest_parameter_distributions,
    n_iter=50,
    scoring="average_precision",
    cv=random_forest_tuning_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

random_forest_search.fit(
    X_train,
    y_train,
)

# Retain the best complete pipeline
random_forest_tuned_pipeline = clone(
    random_forest_search.best_estimator_
)

# Generate fresh out-of-fold probabilities and optimize the threshold
(
    random_forest_tuned_results,
    random_forest_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="Random Forest",
    pipeline=random_forest_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

random_forest_tuned_results["model_stage"] = "Tuned"

# Display the tuned model results
display(
    pd.DataFrame([random_forest_tuned_results])[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Random Forest fine-tuning completed.")
print("Best parameters:", random_forest_search.best_params_)
print(
    "Best tuning PR-AUC:",
    round(random_forest_search.best_score_, 4),
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrails applied: True")
print("Test set used: False")

Fine-tuning Random Forest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Random Forest,Tuned,0.4642,0.7397,0.2155,0.7520,0.7274,0.3350,0.5020,0.8096,0.2956,1281


Random Forest fine-tuning completed.
Best parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 40, 'classifier__min_samples_leaf': 20, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 6, 'classifier__class_weight': 'balanced', 'classifier__bootstrap': True}
Best tuning PR-AUC: 0.3068
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Test set used: False


In [75]:
# Compare the initial and tuned Random Forest models

random_forest_comparison = pd.DataFrame(
    [
        random_forest_initial_results,
        random_forest_tuned_results,
    ]
)[comparison_columns]

display(
    random_forest_comparison
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes produced by fine-tuning
random_forest_tuning_changes = pd.DataFrame(
    {
        "Initial": {
            metric: random_forest_initial_results[metric]
            for metric in [
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "f2",
                "roc_auc",
                "pr_auc",
            ]
        },
        "Tuned": {
            metric: random_forest_tuned_results[metric]
            for metric in [
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "f2",
                "roc_auc",
                "pr_auc",
            ]
        },
    }
)

random_forest_tuning_changes["Change"] = (
    random_forest_tuning_changes["Tuned"]
    - random_forest_tuning_changes["Initial"]
)

display(random_forest_tuning_changes.round(4))

# Determine whether the tuned model satisfies the screening guardrails
random_forest_tuned_passes_guardrails = (
    random_forest_tuned_results["balanced_accuracy"] >= 0.70
    and random_forest_tuned_results["specificity"] >= 0.65
)

print("Random Forest tuning comparison completed.")
print(
    "Recall improvement:",
    round(
        random_forest_tuned_results["recall"]
        - random_forest_initial_results["recall"],
        4,
    ),
)
print(
    "PR-AUC improvement:",
    round(
        random_forest_tuned_results["pr_auc"]
        - random_forest_initial_results["pr_auc"],
        4,
    ),
)
print(
    "Tuned model passes balanced-accuracy and specificity guardrails:",
    random_forest_tuned_passes_guardrails,
)
print("Provisional Random Forest selection: Tuned")
print("Test set used: False")

threshold  balanced_accuracy  precision  recall  \
model         model_stage                                                    
Random Forest Initial         0.2567             0.7401     0.2578  0.6730   
              Tuned           0.4642             0.7397     0.2155  0.7520   

                           specificity     f1     f2  roc_auc  pr_auc  \
model         model_stage                                               
Random Forest Initial           0.8072 0.3728 0.5091   0.8013  0.2672   
              Tuned             0.7274 0.3350 0.5020   0.8096  0.2956   

                           predicted_positive_cases  
model         model_stage                            
Random Forest Initial                           958  
              Tuned                            1281

,Initial,Tuned,Change
balanced_accuracy,0.7401,0.7397,-0.0004
precision,0.2578,0.2155,-0.0424
recall,0.6730,0.7520,0.0790
specificity,0.8072,0.7274,-0.0797
f1,0.3728,0.3350,-0.0379
f2,0.5091,0.5020,-0.0071
roc_auc,0.8013,0.8096,0.0083
pr_auc,0.2672,0.2956,0.0283


Random Forest tuning comparison completed.
Recall improvement: 0.079
PR-AUC improvement: 0.0283
Tuned model passes balanced-accuracy and specificity guardrails: True
Provisional Random Forest selection: Tuned
Test set used: False


In [76]:
# Evaluate the tuned Random Forest with SMOTENC

print("Evaluating tuned Random Forest with SMOTENC...")

# Prepare numerical predictors before SMOTENC
random_forest_smote_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Prepare categorical predictors before SMOTENC
random_forest_smote_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Place numerical predictors first and categorical predictors second
random_forest_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            random_forest_smote_numeric_pipeline,
            numeric_columns,
        ),
        (
            "categorical",
            random_forest_smote_categorical_pipeline,
            categorical_columns,
        ),
    ]
)

# Identify categorical positions after preprocessing
random_forest_smote_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns) + len(categorical_columns),
    )
)

# Copy the classifier from the tuned Random Forest
random_forest_smote_classifier = clone(
    random_forest_tuned_pipeline.named_steps["classifier"]
)

# Disable class weighting because SMOTENC handles the imbalance
random_forest_smote_classifier.set_params(class_weight=None)

# Keep SMOTENC inside the pipeline to prevent data leakage
random_forest_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            random_forest_smote_preprocessor,
        ),
        (
            "smotenc",
            SMOTENC(
                categorical_features=(
                    random_forest_smote_categorical_indices
                ),
                sampling_strategy="auto",
                random_state=RANDOM_STATE,
                k_neighbors=5,
            ),
        ),
        (
            "classifier",
            random_forest_smote_classifier,
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the threshold
(
    random_forest_smote_results,
    random_forest_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="Random Forest",
    pipeline=random_forest_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

random_forest_smote_results["model_stage"] = "Tuned + SMOTENC"

# Display the SMOTENC results
display(
    pd.DataFrame([random_forest_smote_results])[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Random Forest SMOTENC evaluation completed.")
print("Imbalance handling: SMOTENC inside cross-validation")
print("Class weighting used: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrails applied: True")
print("Test set used: False")

Evaluating tuned Random Forest with SMOTENC...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Random Forest,Tuned + SMOTENC,0.3289,0.7140,0.1835,0.7684,0.6596,0.2962,0.4692,0.7793,0.2524,1537


Random Forest SMOTENC evaluation completed.
Imbalance handling: SMOTENC inside cross-validation
Class weighting used: False
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Test set used: False


In [77]:
# Compare all Random Forest configurations and select the final candidate

random_forest_all_results = pd.DataFrame(
    [
        random_forest_initial_results,
        random_forest_tuned_results,
        random_forest_smote_results,
    ]
)[comparison_columns]

display(
    random_forest_all_results
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the recall gain and guardrail losses from SMOTENC
random_forest_smote_recall_gain = (
    random_forest_smote_results["recall"]
    - random_forest_tuned_results["recall"]
)

random_forest_smote_balanced_accuracy_change = (
    random_forest_smote_results["balanced_accuracy"]
    - random_forest_tuned_results["balanced_accuracy"]
)

random_forest_smote_specificity_change = (
    random_forest_smote_results["specificity"]
    - random_forest_tuned_results["specificity"]
)

random_forest_smote_pr_auc_change = (
    random_forest_smote_results["pr_auc"]
    - random_forest_tuned_results["pr_auc"]
)

# Select the tuned class-weighted Random Forest
random_forest_final_pipeline = clone(
    random_forest_tuned_pipeline
)

random_forest_final_results = (
    random_forest_tuned_results.copy()
)

random_forest_final_oof_probabilities = (
    random_forest_tuned_oof_probabilities.copy()
)

random_forest_final_results["model_stage"] = "Final selected"

print("Random Forest configuration comparison completed.")
print(
    "SMOTENC recall change:",
    round(random_forest_smote_recall_gain, 4),
)
print(
    "SMOTENC balanced-accuracy change:",
    round(random_forest_smote_balanced_accuracy_change, 4),
)
print(
    "SMOTENC specificity change:",
    round(random_forest_smote_specificity_change, 4),
)
print(
    "SMOTENC PR-AUC change:",
    round(random_forest_smote_pr_auc_change, 4),
)
print(
    "Final Random Forest selected: "
    "Tuned class-weighted Random Forest"
)
print(
    "Selected threshold:",
    round(random_forest_final_results["threshold"], 4),
)
print(
    "Selected recall:",
    round(random_forest_final_results["recall"], 4),
)
print(
    "Selected specificity:",
    round(random_forest_final_results["specificity"], 4),
)
print(
    "Selected balanced accuracy:",
    round(
        random_forest_final_results["balanced_accuracy"],
        4,
    ),
)
print("Test set used: False")

threshold  balanced_accuracy  precision  \
model         model_stage                                                
Random Forest Initial             0.2567             0.7401     0.2578   
              Tuned               0.4642             0.7397     0.2155   
              Tuned + SMOTENC     0.3289             0.7140     0.1835   

                               recall  specificity     f1     f2  roc_auc  \
model         model_stage                                                   
Random Forest Initial          0.6730       0.8072 0.3728 0.5091   0.8013   
              Tuned            0.7520       0.7274 0.3350 0.5020   0.8096   
              Tuned + SMOTENC  0.7684       0.6596 0.2962 0.4692   0.7793   

                               pr_auc  predicted_positive_cases  
model         model_stage                                        
Random Forest Initial          0.2672                       958  
              Tuned            0.2956                      1281  
              Tuned + SMOTENC  0.2524                      1537

Random Forest configuration comparison completed.
SMOTENC recall change: 0.0163
SMOTENC balanced-accuracy change: -0.0257
SMOTENC specificity change: -0.0678
SMOTENC PR-AUC change: -0.0432
Final Random Forest selected: Tuned class-weighted Random Forest
Selected threshold: 0.4642
Selected recall: 0.752
Selected specificity: 0.7274
Selected balanced accuracy: 0.7397
Test set used: False


In [78]:
# Check the selected Random Forest for overfitting

print("Checking the selected Random Forest for overfitting...")

# Use the threshold selected from out-of-fold predictions
random_forest_selected_threshold = (
    random_forest_final_results["threshold"]
)

# Fit a fresh copy of the selected pipeline on all training observations
random_forest_overfit_pipeline = clone(
    random_forest_final_pipeline
)

random_forest_overfit_pipeline.fit(
    X_train,
    y_train,
)

# Generate probabilities for the complete training data
random_forest_training_probabilities = (
    random_forest_overfit_pipeline.predict_proba(X_train)[:, 1]
)

# Evaluate in-sample training performance
random_forest_training_metrics = evaluate_probabilities(
    y_true=y_train,
    probabilities=random_forest_training_probabilities,
    threshold=random_forest_selected_threshold,
)

# Evaluate out-of-fold performance using the same threshold
random_forest_oof_metrics = evaluate_probabilities(
    y_true=y_train,
    probabilities=random_forest_final_oof_probabilities,
    threshold=random_forest_selected_threshold,
)

# Metrics used to evaluate generalization
random_forest_overfit_metrics = [
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
]

# Build the training-versus-out-of-fold comparison
random_forest_overfit_comparison = pd.DataFrame(
    {
        "Training": {
            metric: random_forest_training_metrics[metric]
            for metric in random_forest_overfit_metrics
        },
        "Out-of-fold": {
            metric: random_forest_oof_metrics[metric]
            for metric in random_forest_overfit_metrics
        },
    }
)

# Calculate the absolute generalization gap
random_forest_overfit_comparison["Absolute gap"] = (
    random_forest_overfit_comparison["Training"]
    - random_forest_overfit_comparison["Out-of-fold"]
).abs()

display(random_forest_overfit_comparison.round(4))

# Identify the largest performance gap
random_forest_largest_gap_metric = (
    random_forest_overfit_comparison["Absolute gap"].idxmax()
)

random_forest_largest_gap = (
    random_forest_overfit_comparison.loc[
        random_forest_largest_gap_metric,
        "Absolute gap",
    ]
)

print("Random Forest overfitting check completed.")
print(
    "Largest performance gap:",
    random_forest_largest_gap_metric,
    round(random_forest_largest_gap, 4),
)
print(
    "Selected threshold:",
    round(random_forest_selected_threshold, 4),
)
print("Training and OOF evaluated at the same threshold: True")
print("Test set used: False")

Checking the selected Random Forest for overfitting...


,Training,Out-of-fold,Absolute gap
balanced_accuracy,0.7922,0.7397,0.0524
precision,0.2402,0.2155,0.0248
recall,0.8529,0.7520,0.1008
specificity,0.7315,0.7274,0.0041
f1,0.3749,0.3350,0.0399
f2,0.5648,0.5020,0.0628
roc_auc,0.8781,0.8096,0.0685
pr_auc,0.4487,0.2956,0.1531


Random Forest overfitting check completed.
Largest performance gap: pr_auc 0.1531
Selected threshold: 0.4642
Training and OOF evaluated at the same threshold: True
Test set used: False


In [79]:
# Apply stronger regularization to reduce Random Forest overfitting

print("Testing a more strongly regularized Random Forest...")

# Build a new pipeline for the targeted regularization search
random_forest_regularized_search_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            RandomForestClassifier(
                bootstrap=True,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Focus the search on simpler, more strongly regularized forests
random_forest_regularized_parameters = {
    "classifier__n_estimators": [300, 500, 700],
    "classifier__max_depth": [3, 4, 5, 6],
    "classifier__min_samples_split": [20, 40, 60, 80, 100],
    "classifier__min_samples_leaf": [20, 30, 40, 60, 80],
    "classifier__max_features": ["sqrt", "log2", 0.3, 0.5],
    "classifier__max_samples": [0.60, 0.75, 0.90, None],
    "classifier__class_weight": [
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 3},
        {0: 1, 1: 4},
        {0: 1, 1: 5},
    ],
}

# Use reproducible stratified cross-validation
random_forest_regularized_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Search for a simpler model with strong validation PR-AUC
random_forest_regularized_search = RandomizedSearchCV(
    estimator=random_forest_regularized_search_pipeline,
    param_distributions=random_forest_regularized_parameters,
    n_iter=40,
    scoring="average_precision",
    cv=random_forest_regularized_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

random_forest_regularized_search.fit(
    X_train,
    y_train,
)

# Retain the best strongly regularized pipeline
random_forest_regularized_pipeline = clone(
    random_forest_regularized_search.best_estimator_
)

# Evaluate it with fresh out-of-fold predictions
(
    random_forest_regularized_results,
    random_forest_regularized_oof_probabilities,
) = create_oof_evaluation(
    model_name="Random Forest",
    pipeline=random_forest_regularized_pipeline,
    X_training=X_train,
    y_training=y_train,
)

random_forest_regularized_results[
    "model_stage"
] = "Stronger regularization"

# Display the out-of-fold results
display(
    pd.DataFrame(
        [random_forest_regularized_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Regularized Random Forest experiment completed.")
print(
    "Best parameters:",
    random_forest_regularized_search.best_params_,
)
print(
    "Best tuning PR-AUC:",
    round(
        random_forest_regularized_search.best_score_,
        4,
    ),
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Test set used: False")

Testing a more strongly regularized Random Forest...
Fitting 5 folds for each of 40 candidates, totalling 200 fits


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Random Forest,Stronger regularization,0.4010,0.7380,0.2701,0.6512,0.8248,0.3818,0.5079,0.8105,0.2956,885


Regularized Random Forest experiment completed.
Best parameters: {'classifier__n_estimators': 700, 'classifier__min_samples_split': 60, 'classifier__min_samples_leaf': 20, 'classifier__max_samples': None, 'classifier__max_features': 'log2', 'classifier__max_depth': 6, 'classifier__class_weight': {0: 1, 1: 5}}
Best tuning PR-AUC: 0.304
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Test set used: False


In [80]:
# Compare the regularized Random Forest and check its overfitting

print("Comparing and checking the regularized Random Forest...")

# Compare the two class-weighted Random Forest configurations
random_forest_regularization_comparison = pd.DataFrame(
    [
        random_forest_tuned_results,
        random_forest_regularized_results,
    ]
)[comparison_columns]

display(
    random_forest_regularization_comparison
    .set_index(["model", "model_stage"])
    .round(4)
)

# Use the threshold selected from the regularized OOF predictions
random_forest_regularized_threshold = (
    random_forest_regularized_results["threshold"]
)

# Fit the regularized pipeline on all training observations
random_forest_regularized_fitted_pipeline = clone(
    random_forest_regularized_pipeline
)

random_forest_regularized_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample probabilities
random_forest_regularized_training_probabilities = (
    random_forest_regularized_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Evaluate training performance at the selected threshold
random_forest_regularized_training_metrics = (
    evaluate_probabilities(
        y_true=y_train,
        probabilities=(
            random_forest_regularized_training_probabilities
        ),
        threshold=random_forest_regularized_threshold,
    )
)

# Evaluate OOF performance at exactly the same threshold
random_forest_regularized_oof_metrics = (
    evaluate_probabilities(
        y_true=y_train,
        probabilities=(
            random_forest_regularized_oof_probabilities
        ),
        threshold=random_forest_regularized_threshold,
    )
)

# Metrics used to inspect generalization
random_forest_regularized_overfit_metrics = [
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
]

# Build the training-versus-OOF comparison
random_forest_regularized_overfit_comparison = pd.DataFrame(
    {
        "Training": {
            metric: (
                random_forest_regularized_training_metrics[
                    metric
                ]
            )
            for metric in (
                random_forest_regularized_overfit_metrics
            )
        },
        "Out-of-fold": {
            metric: (
                random_forest_regularized_oof_metrics[metric]
            )
            for metric in (
                random_forest_regularized_overfit_metrics
            )
        },
    }
)

# Calculate absolute generalization gaps
random_forest_regularized_overfit_comparison[
    "Absolute gap"
] = (
    random_forest_regularized_overfit_comparison[
        "Training"
    ]
    - random_forest_regularized_overfit_comparison[
        "Out-of-fold"
    ]
).abs()

display(
    random_forest_regularized_overfit_comparison.round(4)
)

# Identify the largest gap
random_forest_regularized_largest_gap_metric = (
    random_forest_regularized_overfit_comparison[
        "Absolute gap"
    ].idxmax()
)

random_forest_regularized_largest_gap = (
    random_forest_regularized_overfit_comparison.loc[
        random_forest_regularized_largest_gap_metric,
        "Absolute gap",
    ]
)

print("Regularized Random Forest overfitting check completed.")
print(
    "Largest performance gap:",
    random_forest_regularized_largest_gap_metric,
    round(random_forest_regularized_largest_gap, 4),
)
print(
    "Selected threshold:",
    round(random_forest_regularized_threshold, 4),
)
print("Training and OOF evaluated at the same threshold: True")
print("Test set used: False")

Comparing and checking the regularized Random Forest...


threshold  balanced_accuracy  \
model         model_stage                                             
Random Forest Tuned                       0.4642             0.7397   
              Stronger regularization     0.4010             0.7380   

                                       precision  recall  specificity     f1  \
model         model_stage                                                      
Random Forest Tuned                       0.2155  0.7520       0.7274 0.3350   
              Stronger regularization     0.2701  0.6512       0.8248 0.3818   

                                          f2  roc_auc  pr_auc  \
model         model_stage                                       
Random Forest Tuned                   0.5020   0.8096  0.2956   
              Stronger regularization 0.5079   0.8105  0.2956   

                                       predicted_positive_cases  
model         model_stage                                        
Random Forest Tuned                                        1281  
              Stronger regularization                       885

,Training,Out-of-fold,Absolute gap
balanced_accuracy,0.7718,0.7380,0.0338
precision,0.2919,0.2701,0.0218
recall,0.7166,0.6512,0.0654
specificity,0.8270,0.8248,0.0022
f1,0.4148,0.3818,0.0330
f2,0.5551,0.5079,0.0472
roc_auc,0.8711,0.8105,0.0607
pr_auc,0.4368,0.2956,0.1413


Regularized Random Forest overfitting check completed.
Largest performance gap: pr_auc 0.1413
Selected threshold: 0.401
Training and OOF evaluated at the same threshold: True
Test set used: False


In [81]:
# Finalize the Random Forest configuration

print("Finalizing the Random Forest configuration...")

# Compare generalization gaps
random_forest_overfitting_summary = pd.DataFrame(
    {
        "Tuned": (
            random_forest_overfit_comparison["Absolute gap"]
        ),
        "Stronger regularization": (
            random_forest_regularized_overfit_comparison[
                "Absolute gap"
            ]
        ),
    }
)

display(random_forest_overfitting_summary.round(4))

# Calculate the OOF recall reduction caused by stronger regularization
random_forest_regularized_recall_change = (
    random_forest_regularized_results["recall"]
    - random_forest_tuned_results["recall"]
)

# Calculate the improvement in recall generalization gap
random_forest_recall_gap_improvement = (
    random_forest_overfit_comparison.loc[
        "recall",
        "Absolute gap",
    ]
    - random_forest_regularized_overfit_comparison.loc[
        "recall",
        "Absolute gap",
    ]
)

# Retain the tuned class-weighted Random Forest
random_forest_final_pipeline = clone(
    random_forest_tuned_pipeline
)

random_forest_final_results = (
    random_forest_tuned_results.copy()
)

random_forest_final_oof_probabilities = (
    random_forest_tuned_oof_probabilities.copy()
)

random_forest_final_results["model_stage"] = "Final selected"

# Record the model-development conclusion
random_forest_overfitting_status = (
    "Meaningful overfitting risk; retain with caution"
)

print("Random Forest final selection completed.")
print(
    "Selected configuration: "
    "Tuned class-weighted Random Forest"
)
print(
    "Selected OOF recall:",
    round(random_forest_final_results["recall"], 4),
)
print(
    "Selected OOF specificity:",
    round(random_forest_final_results["specificity"], 4),
)
print(
    "Selected OOF balanced accuracy:",
    round(
        random_forest_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "Selected OOF PR-AUC:",
    round(random_forest_final_results["pr_auc"], 4),
)
print(
    "Recall change from stronger regularization:",
    round(random_forest_regularized_recall_change, 4),
)
print(
    "Recall-gap improvement from stronger regularization:",
    round(random_forest_recall_gap_improvement, 4),
)
print(
    "Overfitting status:",
    random_forest_overfitting_status,
)
print("Test set used: False")

Finalizing the Random Forest configuration...


,Tuned,Stronger regularization
balanced_accuracy,0.0524,0.0338
precision,0.0248,0.0218
recall,0.1008,0.0654
specificity,0.0041,0.0022
f1,0.0399,0.0330
f2,0.0628,0.0472
roc_auc,0.0685,0.0607
pr_auc,0.1531,0.1413


Random Forest final selection completed.
Selected configuration: Tuned class-weighted Random Forest
Selected OOF recall: 0.752
Selected OOF specificity: 0.7274
Selected OOF balanced accuracy: 0.7397
Selected OOF PR-AUC: 0.2956
Recall change from stronger regularization: -0.1008
Recall-gap improvement from stronger regularization: 0.0354
Overfitting status: Meaningful overfitting risk; retain with caution
Test set used: False


## Random Forest Conclusion

The Initial, Tuned, and Tuned with SMOTENC Random Forest configurations were evaluated using stratified cross-validation and maximum out-of-fold F2 threshold optimization. Random Forest did not require numerical scaling.

The Initial model achieved a recall of **0.6730**. Hyperparameter tuning improved recall to **0.7520** while maintaining strong overall discrimination.

SMOTENC increased recall slightly to **0.7684**, but the improvement was only **0.0164** and was accompanied by meaningful reductions in specificity, balanced accuracy, ROC-AUC, and PR-AUC. Therefore, SMOTENC was not retained.

A stronger-regularization experiment was also conducted to reduce overfitting. It reduced the recall gap from **0.1008** to **0.0654**, but validation recall decreased from **0.7520** to **0.6512**. The PR-AUC gap also remained above **0.10**, so stronger regularization did not provide a better final model.

The Tuned class-weighted Random Forest was retained with:

- Threshold: **0.4642**
- Recall: **0.7520**
- ROC-AUC: **0.8096**
- Specificity: **0.7274**
- Balanced accuracy: **0.7397**
- PR-AUC: **0.2956**

The model showed possible overfitting because its training-to-validation PR-AUC gap was **0.1531**. It was therefore retained for the final model comparison with an overfitting caution, but it should not automatically be selected for deployment.

The test set remained untouched.

In [85]:
# Step 30: Build and fine-tune AdaBoost using cross-validation PR-AUC

print("Fine-tuning AdaBoost...")

# Create a shallow class-balanced Decision Tree
# to serve as the AdaBoost base estimator
adaboost_base_tree = DecisionTreeClassifier(
    max_depth=1,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

# Build the complete AdaBoost pipeline
adaboost_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            AdaBoostClassifier(
                estimator=adaboost_base_tree,
                n_estimators=100,
                learning_rate=0.10,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# Search AdaBoost parameters and the complexity of its
# underlying class-balanced Decision Tree
adaboost_parameter_distributions = {
    "classifier__n_estimators": [
        50,
        100,
        200,
        300,
        500,
    ],
    "classifier__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.10,
        0.20,
        0.50,
    ],
    "classifier__estimator__max_depth": [
        1,
        2,
        3,
    ],
    "classifier__estimator__min_samples_split": [
        2,
        10,
        20,
        40,
    ],
    "classifier__estimator__min_samples_leaf": [
        5,
        10,
        20,
        30,
        50,
    ],
}

# Create a reproducible stratified cross-validation strategy
adaboost_cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# PR-AUC is used for tuning because the target is imbalanced
# Recall remains the primary final-selection metric
adaboost_search = RandomizedSearchCV(
    estimator=adaboost_tuning_pipeline,
    param_distributions=adaboost_parameter_distributions,
    n_iter=35,
    scoring="average_precision",
    cv=adaboost_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fit only on the training set
adaboost_search.fit(
    X_train,
    y_train,
)

# Locate the selected parameter combination
best_adaboost_index = adaboost_search.best_index_

# Extract mean training and validation PR-AUC
adaboost_training_pr_auc = adaboost_search.cv_results_[
    "mean_train_score"
][best_adaboost_index]

adaboost_validation_pr_auc = adaboost_search.cv_results_[
    "mean_test_score"
][best_adaboost_index]

# Calculate a preliminary overfitting gap
adaboost_overfitting_gap = (
    adaboost_training_pr_auc
    - adaboost_validation_pr_auc
)

# Save the best complete pipeline
adaboost_tuned_pipeline = clone(
    adaboost_search.best_estimator_
)

print("\nAdaBoost fine-tuning completed.")
print("Best parameters:")
display(adaboost_search.best_params_)

print(
    "Mean training PR-AUC:",
    round(adaboost_training_pr_auc, 4),
)
print(
    "Mean validation PR-AUC:",
    round(adaboost_validation_pr_auc, 4),
)
print(
    "Preliminary overfitting gap:",
    round(adaboost_overfitting_gap, 4),
)
print("Hyperparameter optimization metric: PR-AUC")
print("Primary final-selection metric: Recall")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Fine-tuning AdaBoost...
Fitting 5 folds for each of 35 candidates, totalling 175 fits

AdaBoost fine-tuning completed.
Best parameters:


{'classifier__n_estimators': 50,
 'classifier__learning_rate': 0.5,
 'classifier__estimator__min_samples_split': 2,
 'classifier__estimator__min_samples_leaf': 5,
 'classifier__estimator__max_depth': 3}

Mean training PR-AUC: 0.2654
Mean validation PR-AUC: 0.2308
Preliminary overfitting gap: 0.0346
Hyperparameter optimization metric: PR-AUC
Primary final-selection metric: Recall
Ordinary accuracy used: False
Test set evaluated: False


In [86]:
# Step 31: Evaluate initial and tuned AdaBoost with OOF predictions

print("Evaluating initial and tuned AdaBoost models...")

# Build a reproducible initial AdaBoost model for comparison
adaboost_initial_base_tree = DecisionTreeClassifier(
    max_depth=1,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

adaboost_initial_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            AdaBoostClassifier(
                estimator=adaboost_initial_base_tree,
                n_estimators=100,
                learning_rate=0.10,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# Evaluate the initial AdaBoost model
(
    adaboost_initial_results,
    adaboost_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="AdaBoost",
    pipeline=adaboost_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

adaboost_initial_results["model_stage"] = "Initial"

# Evaluate the tuned AdaBoost model
(
    adaboost_tuned_results,
    adaboost_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="AdaBoost",
    pipeline=adaboost_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

adaboost_tuned_results["model_stage"] = "Tuned"

# Compare both models
adaboost_initial_tuned_comparison = pd.DataFrame(
    [
        adaboost_initial_results,
        adaboost_tuned_results,
    ]
)[comparison_columns]

display(
    adaboost_initial_tuned_comparison
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the changes produced by fine-tuning
adaboost_tuning_changes = pd.DataFrame(
    {
        "Initial": {
            metric: adaboost_initial_results[metric]
            for metric in [
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "f2",
                "roc_auc",
                "pr_auc",
            ]
        },
        "Tuned": {
            metric: adaboost_tuned_results[metric]
            for metric in [
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "f2",
                "roc_auc",
                "pr_auc",
            ]
        },
    }
)

adaboost_tuning_changes["Change"] = (
    adaboost_tuning_changes["Tuned"]
    - adaboost_tuning_changes["Initial"]
)

display(adaboost_tuning_changes.round(4))

# Check whether the tuned model passes the guardrails
adaboost_tuned_passes_guardrails = (
    adaboost_tuned_results["balanced_accuracy"] >= 0.70
    and adaboost_tuned_results["specificity"] >= 0.65
)

print("AdaBoost OOF evaluation completed.")
print(
    "Tuned threshold:",
    round(adaboost_tuned_results["threshold"], 4),
)
print(
    "Recall change:",
    round(
        adaboost_tuned_results["recall"]
        - adaboost_initial_results["recall"],
        4,
    ),
)
print(
    "PR-AUC change:",
    round(
        adaboost_tuned_results["pr_auc"]
        - adaboost_initial_results["pr_auc"],
        4,
    ),
)
print(
    "Tuned model passes guardrails:",
    adaboost_tuned_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating initial and tuned AdaBoost models...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                    
AdaBoost Initial         0.2070             0.7186     0.1913  0.7548   
         Tuned           0.3504             0.7348     0.1970  0.7902   

                      specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                               
AdaBoost Initial           0.6824 0.3052 0.4750   0.7472  0.2122   
         Tuned             0.6794 0.3154 0.4932   0.7834  0.2273   

                      predicted_positive_cases  
model    model_stage                            
AdaBoost Initial                          1448  
         Tuned                            1472

,Initial,Tuned,Change
balanced_accuracy,0.7186,0.7348,0.0162
precision,0.1913,0.1970,0.0057
recall,0.7548,0.7902,0.0354
specificity,0.6824,0.6794,-0.0030
f1,0.3052,0.3154,0.0102
f2,0.4750,0.4932,0.0182
roc_auc,0.7472,0.7834,0.0361
pr_auc,0.2122,0.2273,0.0151


AdaBoost OOF evaluation completed.
Tuned threshold: 0.3504
Recall change: 0.0354
PR-AUC change: 0.0151
Tuned model passes guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Ordinary accuracy included: False
Test set used: False


### AdaBoost with SMOTENC

SMOTENC is evaluated as an alternative imbalance-handling strategy for the tuned
AdaBoost model.

Synthetic minority-class observations are generated only within each
cross-validation training fold to prevent data leakage. The base Decision Tree's
class weighting is disabled because SMOTENC already balances the training data.

The SMOTENC result will be compared with the initial and tuned class-weighted
AdaBoost models. The test set remains untouched.

In [87]:
# Step 32: Evaluate tuned AdaBoost with SMOTENC

print("Evaluating tuned AdaBoost with SMOTENC...")

# Prepare numerical predictors
adaboost_smote_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# Prepare categorical predictors
adaboost_smote_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Numerical predictors are placed before categorical predictors
adaboost_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            adaboost_smote_numeric_pipeline,
            numeric_columns,
        ),
        (
            "categorical",
            adaboost_smote_categorical_pipeline,
            categorical_columns,
        ),
    ]
)

# Identify the categorical positions after preprocessing
adaboost_smote_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns) + len(categorical_columns),
    )
)

# Copy the tuned AdaBoost classifier
adaboost_smote_classifier = clone(
    adaboost_tuned_pipeline.named_steps["classifier"]
)

# Disable the base tree's class weighting because SMOTENC
# will handle the class imbalance
adaboost_smote_classifier.set_params(
    estimator__class_weight=None
)

# Keep SMOTENC inside the pipeline to prevent data leakage
adaboost_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            adaboost_smote_preprocessor,
        ),
        (
            "smotenc",
            SMOTENC(
                categorical_features=(
                    adaboost_smote_categorical_indices
                ),
                sampling_strategy="auto",
                random_state=RANDOM_STATE,
                k_neighbors=5,
            ),
        ),
        (
            "classifier",
            adaboost_smote_classifier,
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the threshold
(
    adaboost_smote_results,
    adaboost_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="AdaBoost",
    pipeline=adaboost_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

adaboost_smote_results["model_stage"] = "Tuned + SMOTENC"

# Display the result
display(
    pd.DataFrame(
        [adaboost_smote_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("AdaBoost SMOTENC evaluation completed.")
print("Imbalance handling: SMOTENC inside cross-validation")
print("Base-tree class weighting used: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-selection metric: Recall")
print("Balanced-accuracy and specificity guardrails applied: True")
print("Test set used: False")

Evaluating tuned AdaBoost with SMOTENC...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
AdaBoost,Tuned + SMOTENC,0.3865,0.6973,0.1709,0.7629,0.6317,0.2793,0.4507,0.7632,0.2253,1638


AdaBoost SMOTENC evaluation completed.
Imbalance handling: SMOTENC inside cross-validation
Base-tree class weighting used: False
Threshold selected using: Maximum out-of-fold F2
Primary final-selection metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Test set used: False


In [88]:
# Step 33: Compare all AdaBoost configurations

print("Comparing all AdaBoost configurations...")

# Combine the results from the three AdaBoost experiments.
adaboost_comparison = pd.DataFrame(
    [
        adaboost_initial_results,
        adaboost_tuned_results,
        adaboost_smote_results,
    ]
)

# Display every AdaBoost result in one table.
display(
    adaboost_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Compare the tuned model with the initial model.
tuned_recall_improvement = (
    adaboost_tuned_results["recall"]
    - adaboost_initial_results["recall"]
)

tuned_balanced_accuracy_improvement = (
    adaboost_tuned_results["balanced_accuracy"]
    - adaboost_initial_results["balanced_accuracy"]
)

tuned_pr_auc_improvement = (
    adaboost_tuned_results["pr_auc"]
    - adaboost_initial_results["pr_auc"]
)

# Compare SMOTENC with the tuned class-weighted model.
smote_recall_difference = (
    adaboost_smote_results["recall"]
    - adaboost_tuned_results["recall"]
)

smote_balanced_accuracy_difference = (
    adaboost_smote_results["balanced_accuracy"]
    - adaboost_tuned_results["balanced_accuracy"]
)

smote_specificity_difference = (
    adaboost_smote_results["specificity"]
    - adaboost_tuned_results["specificity"]
)

smote_pr_auc_difference = (
    adaboost_smote_results["pr_auc"]
    - adaboost_tuned_results["pr_auc"]
)

# Apply the project guardrails.
smote_passes_guardrails = (
    adaboost_smote_results["balanced_accuracy"] >= 0.70
    and adaboost_smote_results["specificity"] >= 0.65
)

# Retain the tuned class-weighted model.
# It produced the strongest recall while satisfying both guardrails.
adaboost_final_pipeline = clone(adaboost_search.best_estimator_)

adaboost_final_results = adaboost_tuned_results.copy()
adaboost_final_oof_probabilities = (
    adaboost_tuned_oof_probabilities.copy()
)

print("\nAdaBoost comparison completed.")

print("\nTuned model versus initial model:")
print(
    "Recall improvement:",
    round(tuned_recall_improvement, 4),
)
print(
    "Balanced-accuracy improvement:",
    round(tuned_balanced_accuracy_improvement, 4),
)
print(
    "PR-AUC improvement:",
    round(tuned_pr_auc_improvement, 4),
)

print("\nSMOTENC versus tuned class-weighted model:")
print(
    "Recall difference:",
    round(smote_recall_difference, 4),
)
print(
    "Balanced-accuracy difference:",
    round(smote_balanced_accuracy_difference, 4),
)
print(
    "Specificity difference:",
    round(smote_specificity_difference, 4),
)
print(
    "PR-AUC difference:",
    round(smote_pr_auc_difference, 4),
)
print(
    "SMOTENC passes both guardrails:",
    smote_passes_guardrails,
)

print("\nSelected AdaBoost configuration: Tuned class-weighted AdaBoost")
print(
    "Selected recall:",
    round(adaboost_final_results["recall"], 4),
)
print(
    "Selected balanced accuracy:",
    round(
        adaboost_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "Selected specificity:",
    round(adaboost_final_results["specificity"], 4),
)
print(
    "Selected PR-AUC:",
    round(adaboost_final_results["pr_auc"], 4),
)
print("SMOTENC retained: False")
print("Test set evaluated: False")

Comparing all AdaBoost configurations...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                        
AdaBoost Initial             0.2070             0.7186     0.1913  0.7548   
         Tuned               0.3504             0.7348     0.1970  0.7902   
         Tuned + SMOTENC     0.3865             0.6973     0.1709  0.7629   

                          specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                                   
AdaBoost Initial               0.6824 0.3052 0.4750   0.7472  0.2122   
         Tuned                 0.6794 0.3154 0.4932   0.7834  0.2273   
         Tuned + SMOTENC       0.6317 0.2793 0.4507   0.7632  0.2253   

                          predicted_positive_cases  
model    model_stage                                
AdaBoost Initial                              1448  
         Tuned                                1472  
         Tuned + SMOTENC                      1638


AdaBoost comparison completed.

Tuned model versus initial model:
Recall improvement: 0.0354
Balanced-accuracy improvement: 0.0162
PR-AUC improvement: 0.0151

SMOTENC versus tuned class-weighted model:
Recall difference: -0.0272
Balanced-accuracy difference: -0.0375
Specificity difference: -0.0477
PR-AUC difference: -0.002
SMOTENC passes both guardrails: False

Selected AdaBoost configuration: Tuned class-weighted AdaBoost
Selected recall: 0.7902
Selected balanced accuracy: 0.7348
Selected specificity: 0.6794
Selected PR-AUC: 0.2273
SMOTENC retained: False
Test set evaluated: False


In [89]:
# Step 34: Check the selected AdaBoost model for overfitting

print("Checking the selected AdaBoost model for overfitting...")

# Use the threshold selected from out-of-fold training predictions.
adaboost_selected_threshold = float(
    adaboost_final_results["threshold"]
)

# Fit a separate copy on the complete training set.
# The test set remains untouched.
adaboost_overfitting_pipeline = clone(
    adaboost_final_pipeline
)

adaboost_overfitting_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample training probabilities.
adaboost_training_probabilities = (
    adaboost_overfitting_pipeline.predict_proba(X_train)[:, 1]
)

# Convert probabilities into class predictions using the same
# threshold selected from the out-of-fold predictions.
adaboost_training_predictions = (
    adaboost_training_probabilities
    >= adaboost_selected_threshold
).astype(int)

adaboost_oof_predictions = (
    adaboost_final_oof_probabilities
    >= adaboost_selected_threshold
).astype(int)

# Calculate training performance.
adaboost_training_recall = recall_score(
    y_train,
    adaboost_training_predictions,
)

adaboost_training_pr_auc = average_precision_score(
    y_train,
    adaboost_training_probabilities,
)

# Calculate out-of-fold performance.
adaboost_oof_recall = recall_score(
    y_train,
    adaboost_oof_predictions,
)

adaboost_oof_pr_auc = average_precision_score(
    y_train,
    adaboost_final_oof_probabilities,
)

# Calculate the training-to-validation gaps.
adaboost_recall_gap = (
    adaboost_training_recall
    - adaboost_oof_recall
)

adaboost_pr_auc_gap = (
    adaboost_training_pr_auc
    - adaboost_oof_pr_auc
)

# Summarize the overfitting results.
adaboost_overfitting_results = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "PR-AUC",
        ],
        "training_score": [
            adaboost_training_recall,
            adaboost_training_pr_auc,
        ],
        "out_of_fold_score": [
            adaboost_oof_recall,
            adaboost_oof_pr_auc,
        ],
        "overfitting_gap": [
            adaboost_recall_gap,
            adaboost_pr_auc_gap,
        ],
    }
).set_index("metric")

display(
    adaboost_overfitting_results.round(4)
)

# Interpret the size of the performance gaps.
# A gap above 0.10 is treated as meaningful overfitting.
adaboost_meaningful_overfitting = (
    adaboost_recall_gap > 0.10
    or adaboost_pr_auc_gap > 0.10
)

if adaboost_meaningful_overfitting:
    adaboost_overfitting_status = (
        "Meaningful overfitting detected"
    )
else:
    adaboost_overfitting_status = (
        "No meaningful overfitting detected"
    )

print("\nAdaBoost overfitting check completed.")
print(
    "Selected threshold:",
    round(adaboost_selected_threshold, 4),
)
print(
    "Recall gap:",
    round(adaboost_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(adaboost_pr_auc_gap, 4),
)
print(
    "Overfitting assessment:",
    adaboost_overfitting_status,
)
print(
    "Final AdaBoost configuration:",
    "Tuned class-weighted AdaBoost",
)
print("Test set evaluated: False")

Checking the selected AdaBoost model for overfitting...


,training_score,out_of_fold_score,overfitting_gap
metric,,,
Recall,0.8392,0.7902,0.0490
PR-AUC,0.2809,0.2273,0.0536



AdaBoost overfitting check completed.
Selected threshold: 0.3504
Recall gap: 0.049
PR-AUC gap: 0.0536
Overfitting assessment: No meaningful overfitting detected
Final AdaBoost configuration: Tuned class-weighted AdaBoost
Test set evaluated: False


## AdaBoost Conclusion

Three AdaBoost configurations were evaluated: Initial, Tuned with balanced class weights, and Tuned with SMOTENC. AdaBoost did not require numerical feature scaling.

Hyperparameter tuning improved the model compared with the Initial configuration. Recall increased from **0.7548** to **0.7902**, balanced accuracy increased from **0.7186** to **0.7348**, and PR-AUC increased from **0.2122** to **0.2273**.

The SMOTENC configuration reduced recall to **0.7629**, specificity to **0.6317**, and balanced accuracy to **0.6973**. Because it did not improve recall and failed both performance guardrails, SMOTENC was not retained.

The Tuned class-weighted AdaBoost achieved:

- Threshold: **0.3504**
- Recall: **0.7902**
- ROC-AUC: **0.7834**
- Specificity: **0.6794**
- Balanced accuracy: **0.7348**
- PR-AUC: **0.2273**

The training-to-validation recall gap was **0.0490**, and the PR-AUC gap was **0.0536**. Both were below the **0.10** overfitting threshold, so no meaningful overfitting was detected.

The **Tuned class-weighted AdaBoost** was retained for the final model comparison. The test set remained untouched.

## Gradient Boosting

Gradient Boosting combines a sequence of shallow decision trees, with each tree attempting to correct errors made by the previous trees.

The initial model is evaluated using stratified cross-validation and out-of-fold probabilities. Because `GradientBoostingClassifier` does not provide a native `class_weight` parameter, class imbalance is initially addressed through F2-based threshold optimization. SMOTENC will be tested separately after fine-tuning.

The test set remains untouched during model development.

In [90]:
# Step 35: Build and evaluate the initial Gradient Boosting model

print("Evaluating initial Gradient Boosting...")

# Create the initial Gradient Boosting pipeline.
# GradientBoostingClassifier does not support class_weight.
gradient_boosting_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=3,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# Generate out-of-fold probabilities and select the threshold
# that maximizes F2 while applying the project guardrails.
(
    gradient_boosting_initial_results,
    gradient_boosting_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="Gradient Boosting",
    pipeline=gradient_boosting_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Identify this result as the initial model.
gradient_boosting_initial_results[
    "model_stage"
] = "Initial"

# Display the initial Gradient Boosting performance.
display(
    pd.DataFrame(
        [gradient_boosting_initial_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print(
    "Initial Gradient Boosting evaluation completed."
)
print(
    "Imbalance handling:",
    "F2-based threshold optimization",
)
print(
    "Threshold selected using:",
    "Maximum out-of-fold F2",
)
print(
    "Balanced-accuracy and specificity guardrails applied:",
    True,
)
print("Primary comparison metric: Recall")
print("Fine-tuning completed: False")
print("SMOTENC applied: False")
print("Ordinary accuracy included: False")
print("Test set used: False")

Evaluating initial Gradient Boosting...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Gradient Boosting,Initial,0.0891,0.7440,0.2308,0.7302,0.7578,0.3508,0.5097,0.8001,0.2620,1161


Initial Gradient Boosting evaluation completed.
Imbalance handling: F2-based threshold optimization
Threshold selected using: Maximum out-of-fold F2
Balanced-accuracy and specificity guardrails applied: True
Primary comparison metric: Recall
Fine-tuning completed: False
SMOTENC applied: False
Ordinary accuracy included: False
Test set used: False


In [92]:
# Step 36: Fine-tune Gradient Boosting using cross-validation PR-AUC

print("Fine-tuning Gradient Boosting...")

# Define reproducible stratified cross-validation.
# Stratification preserves the elevated-risk class proportion
# inside every training and validation fold.
gradient_boosting_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Search parameters that control learning speed,
# tree complexity and regularization.
gradient_boosting_parameter_distributions = {
    "classifier__n_estimators": [
        50,
        100,
        150,
        200,
        300,
    ],
    "classifier__learning_rate": [
        0.02,
        0.03,
        0.05,
        0.075,
        0.10,
        0.15,
    ],
    "classifier__max_depth": [
        1,
        2,
        3,
        4,
    ],
    "classifier__min_samples_split": [
        2,
        10,
        20,
        40,
    ],
    "classifier__min_samples_leaf": [
        5,
        10,
        20,
        30,
        50,
    ],
    "classifier__subsample": [
        0.60,
        0.80,
        1.00,
    ],
    "classifier__max_features": [
        None,
        "sqrt",
        "log2",
    ],
}

# PR-AUC is used for fine-tuning because the target
# is highly imbalanced.
#
# Recall will remain the primary metric when selecting
# the final model after threshold optimization.
gradient_boosting_search = RandomizedSearchCV(
    estimator=clone(
        gradient_boosting_initial_pipeline
    ),
    param_distributions=(
        gradient_boosting_parameter_distributions
    ),
    n_iter=40,
    scoring="average_precision",
    cv=gradient_boosting_cv,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fit using only the training data.
gradient_boosting_search.fit(
    X_train,
    y_train,
)

# Locate the best parameter combination.
best_gradient_boosting_index = (
    gradient_boosting_search.best_index_
)

# Extract training and validation PR-AUC scores
# for a preliminary overfitting check.
gradient_boosting_training_pr_auc = (
    gradient_boosting_search.cv_results_[
        "mean_train_score"
    ][best_gradient_boosting_index]
)

gradient_boosting_validation_pr_auc = (
    gradient_boosting_search.cv_results_[
        "mean_test_score"
    ][best_gradient_boosting_index]
)

gradient_boosting_search_gap = (
    gradient_boosting_training_pr_auc
    - gradient_boosting_validation_pr_auc
)

print("\nGradient Boosting fine-tuning completed.")

print("\nBest parameters:")
display(
    gradient_boosting_search.best_params_
)

print(
    "Best cross-validation PR-AUC:",
    round(
        gradient_boosting_search.best_score_,
        4,
    ),
)

print(
    "Training PR-AUC:",
    round(
        gradient_boosting_training_pr_auc,
        4,
    ),
)

print(
    "Validation PR-AUC:",
    round(
        gradient_boosting_validation_pr_auc,
        4,
    ),
)

print(
    "Preliminary overfitting gap:",
    round(
        gradient_boosting_search_gap,
        4,
    ),
)

print("Fine-tuning optimization metric: PR-AUC")
print("Final model-selection metric: Recall")
print("Ordinary accuracy used: False")
print("Threshold optimization completed: False")
print("Test set evaluated: False")

Fine-tuning Gradient Boosting...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Gradient Boosting fine-tuning completed.

Best parameters:


{'classifier__subsample': 1.0,
 'classifier__n_estimators': 100,
 'classifier__min_samples_split': 2,
 'classifier__min_samples_leaf': 30,
 'classifier__max_features': 'log2',
 'classifier__max_depth': 2,
 'classifier__learning_rate': 0.02}

Best cross-validation PR-AUC: 0.3049
Training PR-AUC: 0.3548
Validation PR-AUC: 0.3049
Preliminary overfitting gap: 0.0499
Fine-tuning optimization metric: PR-AUC
Final model-selection metric: Recall
Ordinary accuracy used: False
Threshold optimization completed: False
Test set evaluated: False


In [93]:
# Step 37: Evaluate and compare the tuned Gradient Boosting model

print(
    "Evaluating tuned Gradient Boosting with "
    "out-of-fold threshold optimization..."
)

# Copy the best pipeline selected during fine-tuning.
gradient_boosting_tuned_pipeline = clone(
    gradient_boosting_search.best_estimator_
)

# Generate fresh out-of-fold probabilities.
# The F2-based threshold is selected only from training data.
(
    gradient_boosting_tuned_results,
    gradient_boosting_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="Gradient Boosting",
    pipeline=gradient_boosting_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Identify the model-development stage.
gradient_boosting_tuned_results[
    "model_stage"
] = "Tuned"

# Combine the initial and tuned results.
gradient_boosting_tuning_comparison = pd.DataFrame(
    [
        gradient_boosting_initial_results,
        gradient_boosting_tuned_results,
    ]
)

# Display both configurations in one table.
display(
    gradient_boosting_tuning_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the changes produced by fine-tuning.
gradient_boosting_recall_change = (
    gradient_boosting_tuned_results["recall"]
    - gradient_boosting_initial_results["recall"]
)

gradient_boosting_balanced_accuracy_change = (
    gradient_boosting_tuned_results[
        "balanced_accuracy"
    ]
    - gradient_boosting_initial_results[
        "balanced_accuracy"
    ]
)

gradient_boosting_specificity_change = (
    gradient_boosting_tuned_results["specificity"]
    - gradient_boosting_initial_results["specificity"]
)

gradient_boosting_pr_auc_change = (
    gradient_boosting_tuned_results["pr_auc"]
    - gradient_boosting_initial_results["pr_auc"]
)

# Verify that the tuned model satisfies the project guardrails.
gradient_boosting_tuned_passes_guardrails = (
    gradient_boosting_tuned_results[
        "balanced_accuracy"
    ] >= 0.70
    and gradient_boosting_tuned_results[
        "specificity"
    ] >= 0.65
)

print(
    "\nTuned Gradient Boosting evaluation completed."
)

print("\nTuned model versus initial model:")

print(
    "Recall change:",
    round(
        gradient_boosting_recall_change,
        4,
    ),
)

print(
    "Balanced-accuracy change:",
    round(
        gradient_boosting_balanced_accuracy_change,
        4,
    ),
)

print(
    "Specificity change:",
    round(
        gradient_boosting_specificity_change,
        4,
    ),
)

print(
    "PR-AUC change:",
    round(
        gradient_boosting_pr_auc_change,
        4,
    ),
)

print(
    "Tuned model passes both guardrails:",
    gradient_boosting_tuned_passes_guardrails,
)

print(
    "Selected threshold:",
    round(
        gradient_boosting_tuned_results[
            "threshold"
        ],
        4,
    ),
)

print("Primary comparison metric: Recall")
print("SMOTENC evaluated: False")
print("Final configuration selected: False")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating tuned Gradient Boosting with out-of-fold threshold optimization...


threshold  balanced_accuracy  precision  \
model             model_stage                                            
Gradient Boosting Initial         0.0891             0.7440     0.2308   
                  Tuned           0.1116             0.7399     0.2449   

                               recall  specificity     f1     f2  roc_auc  \
model             model_stage                                               
Gradient Boosting Initial      0.7302       0.7578 0.3508 0.5097   0.8001   
                  Tuned        0.6921       0.7876 0.3618 0.5070   0.8101   

                               pr_auc  predicted_positive_cases  
model             model_stage                                    
Gradient Boosting Initial      0.2620                      1161  
                  Tuned        0.2930                      1037


Tuned Gradient Boosting evaluation completed.

Tuned model versus initial model:
Recall change: -0.0381
Balanced-accuracy change: -0.0042
Specificity change: 0.0298
PR-AUC change: 0.031
Tuned model passes both guardrails: True
Selected threshold: 0.1116
Primary comparison metric: Recall
SMOTENC evaluated: False
Final configuration selected: False
Ordinary accuracy included: False
Test set evaluated: False


In [94]:
# Step 38: Evaluate tuned Gradient Boosting with SMOTENC

print(
    "Evaluating tuned Gradient Boosting with SMOTENC..."
)

# Impute numeric predictors while keeping them numeric.
gradient_boosting_smote_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

# Impute and ordinal-encode categorical predictors.
# Ordinal encoding is required before applying SMOTENC.
gradient_boosting_smote_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Preprocess numeric columns first and categorical columns second.
gradient_boosting_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            gradient_boosting_smote_numeric_pipeline,
            numeric_columns,
        ),
        (
            "categorical",
            gradient_boosting_smote_categorical_pipeline,
            categorical_columns,
        ),
    ],
    remainder="drop",
)

# Identify the positions of categorical predictors after transformation.
gradient_boosting_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns)
        + len(categorical_columns),
    )
)

# Reuse the tuned Gradient Boosting classifier.
gradient_boosting_smote_classifier = clone(
    gradient_boosting_search.best_estimator_
    .named_steps["classifier"]
)

# Apply SMOTENC only inside each cross-validation training fold.
# This prevents synthetic observations from leaking into validation folds.
gradient_boosting_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            gradient_boosting_smote_preprocessor,
        ),
        (
            "smote",
            SMOTENC(
                categorical_features=(
                    gradient_boosting_categorical_indices
                ),
                sampling_strategy="auto",
                k_neighbors=5,
                random_state=RANDOM_STATE,
            ),
        ),
        (
            "classifier",
            gradient_boosting_smote_classifier,
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the F2 threshold.
(
    gradient_boosting_smote_results,
    gradient_boosting_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="Gradient Boosting",
    pipeline=gradient_boosting_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Label this model-development stage.
gradient_boosting_smote_results[
    "model_stage"
] = "Tuned + SMOTENC"

# Display the SMOTENC result.
display(
    pd.DataFrame(
        [gradient_boosting_smote_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Check the project guardrails.
gradient_boosting_smote_passes_guardrails = (
    gradient_boosting_smote_results[
        "balanced_accuracy"
    ] >= 0.70
    and gradient_boosting_smote_results[
        "specificity"
    ] >= 0.65
)

print(
    "\nGradient Boosting SMOTENC evaluation completed."
)
print("Imbalance handling: SMOTENC inside cross-validation")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary comparison metric: Recall")
print(
    "Balanced-accuracy and specificity guardrails passed:",
    gradient_boosting_smote_passes_guardrails,
)
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating tuned Gradient Boosting with SMOTENC...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Gradient Boosting,Tuned + SMOTENC,0.4585,0.7077,0.2043,0.6785,0.7369,0.3140,0.4633,0.7748,0.2664,1219



Gradient Boosting SMOTENC evaluation completed.
Imbalance handling: SMOTENC inside cross-validation
Threshold selected using: Maximum out-of-fold F2
Primary comparison metric: Recall
Balanced-accuracy and specificity guardrails passed: True
Ordinary accuracy included: False
Test set evaluated: False


In [95]:
# Step 39: Compare all Gradient Boosting configurations

print(
    "Comparing all Gradient Boosting configurations..."
)

# Combine all Gradient Boosting experiments.
gradient_boosting_comparison = pd.DataFrame(
    [
        gradient_boosting_initial_results,
        gradient_boosting_tuned_results,
        gradient_boosting_smote_results,
    ]
)

# Display all configurations together.
display(
    gradient_boosting_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Identify which configurations satisfy both guardrails.
gradient_boosting_eligible_results = (
    gradient_boosting_comparison[
        (
            gradient_boosting_comparison[
                "balanced_accuracy"
            ] >= 0.70
        )
        & (
            gradient_boosting_comparison[
                "specificity"
            ] >= 0.65
        )
    ].copy()
)

# Rank eligible configurations by the primary metric:
# recall. PR-AUC is used as the first tie-breaker.
gradient_boosting_ranked_results = (
    gradient_boosting_eligible_results.sort_values(
        by=[
            "recall",
            "pr_auc",
            "balanced_accuracy",
        ],
        ascending=False,
    )
)

print(
    "\nEligible configurations ranked by recall:"
)

display(
    gradient_boosting_ranked_results[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Retain the initial model because it produced the
# highest recall while passing both guardrails.
gradient_boosting_final_pipeline = clone(
    gradient_boosting_initial_pipeline
)

gradient_boosting_final_results = (
    gradient_boosting_initial_results.copy()
)

gradient_boosting_final_oof_probabilities = (
    gradient_boosting_initial_oof_probabilities.copy()
)

# Calculate the main differences for documentation.
tuned_recall_difference = (
    gradient_boosting_tuned_results["recall"]
    - gradient_boosting_initial_results["recall"]
)

smote_recall_difference = (
    gradient_boosting_smote_results["recall"]
    - gradient_boosting_initial_results["recall"]
)

tuned_pr_auc_difference = (
    gradient_boosting_tuned_results["pr_auc"]
    - gradient_boosting_initial_results["pr_auc"]
)

smote_pr_auc_difference = (
    gradient_boosting_smote_results["pr_auc"]
    - gradient_boosting_initial_results["pr_auc"]
)

print(
    "\nGradient Boosting comparison completed."
)

print(
    "Tuned recall difference:",
    round(tuned_recall_difference, 4),
)

print(
    "SMOTENC recall difference:",
    round(smote_recall_difference, 4),
)

print(
    "Tuned PR-AUC difference:",
    round(tuned_pr_auc_difference, 4),
)

print(
    "SMOTENC PR-AUC difference:",
    round(smote_pr_auc_difference, 4),
)

print(
    "\nSelected Gradient Boosting configuration:",
    "Initial Gradient Boosting",
)

print(
    "Selected recall:",
    round(
        gradient_boosting_final_results["recall"],
        4,
    ),
)

print(
    "Selected balanced accuracy:",
    round(
        gradient_boosting_final_results[
            "balanced_accuracy"
        ],
        4,
    ),
)

print(
    "Selected specificity:",
    round(
        gradient_boosting_final_results[
            "specificity"
        ],
        4,
    ),
)

print(
    "Selected PR-AUC:",
    round(
        gradient_boosting_final_results["pr_auc"],
        4,
    ),
)

print("Fine-tuned configuration retained: False")
print("SMOTENC configuration retained: False")
print("Selection based primarily on recall: True")
print("Test set evaluated: False")

Comparing all Gradient Boosting configurations...


threshold  balanced_accuracy  precision  \
model             model_stage                                                
Gradient Boosting Initial             0.0891             0.7440     0.2308   
                  Tuned               0.1116             0.7399     0.2449   
                  Tuned + SMOTENC     0.4585             0.7077     0.2043   

                                   recall  specificity     f1     f2  roc_auc  \
model             model_stage                                                   
Gradient Boosting Initial          0.7302       0.7578 0.3508 0.5097   0.8001   
                  Tuned            0.6921       0.7876 0.3618 0.5070   0.8101   
                  Tuned + SMOTENC  0.6785       0.7369 0.3140 0.4633   0.7748   

                                   pr_auc  predicted_positive_cases  
model             model_stage                                        
Gradient Boosting Initial          0.2620                      1161  
                  Tuned            0.2930                      1037  
                  Tuned + SMOTENC  0.2664                      1219


Eligible configurations ranked by recall:


threshold  balanced_accuracy  precision  \
model             model_stage                                                
Gradient Boosting Initial             0.0891             0.7440     0.2308   
                  Tuned               0.1116             0.7399     0.2449   
                  Tuned + SMOTENC     0.4585             0.7077     0.2043   

                                   recall  specificity     f1     f2  roc_auc  \
model             model_stage                                                   
Gradient Boosting Initial          0.7302       0.7578 0.3508 0.5097   0.8001   
                  Tuned            0.6921       0.7876 0.3618 0.5070   0.8101   
                  Tuned + SMOTENC  0.6785       0.7369 0.3140 0.4633   0.7748   

                                   pr_auc  predicted_positive_cases  
model             model_stage                                        
Gradient Boosting Initial          0.2620                      1161  
                  Tuned            0.2930                      1037  
                  Tuned + SMOTENC  0.2664                      1219


Gradient Boosting comparison completed.
Tuned recall difference: -0.0381
SMOTENC recall difference: -0.0518
Tuned PR-AUC difference: 0.031
SMOTENC PR-AUC difference: 0.0045

Selected Gradient Boosting configuration: Initial Gradient Boosting
Selected recall: 0.7302
Selected balanced accuracy: 0.744
Selected specificity: 0.7578
Selected PR-AUC: 0.262
Fine-tuned configuration retained: False
SMOTENC configuration retained: False
Selection based primarily on recall: True
Test set evaluated: False


In [96]:
# Step 40: Check the selected Gradient Boosting model for overfitting

print(
    "Checking the selected Gradient Boosting model "
    "for overfitting..."
)

# Use the threshold selected from the initial model's
# out-of-fold probabilities.
gradient_boosting_selected_threshold = float(
    gradient_boosting_final_results["threshold"]
)

# Fit a separate copy of the selected pipeline using
# the complete training set.
# The test set remains untouched.
gradient_boosting_overfitting_pipeline = clone(
    gradient_boosting_final_pipeline
)

gradient_boosting_overfitting_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample training probabilities.
gradient_boosting_training_probabilities = (
    gradient_boosting_overfitting_pipeline
    .predict_proba(X_train)[:, 1]
)

# Apply the same out-of-fold-selected threshold
# to the training and out-of-fold probabilities.
gradient_boosting_training_predictions = (
    gradient_boosting_training_probabilities
    >= gradient_boosting_selected_threshold
).astype(int)

gradient_boosting_oof_predictions = (
    gradient_boosting_final_oof_probabilities
    >= gradient_boosting_selected_threshold
).astype(int)

# Calculate recall for training and out-of-fold predictions.
gradient_boosting_training_recall = recall_score(
    y_train,
    gradient_boosting_training_predictions,
)

gradient_boosting_oof_recall = recall_score(
    y_train,
    gradient_boosting_oof_predictions,
)

# Calculate PR-AUC for training and out-of-fold probabilities.
gradient_boosting_training_pr_auc = (
    average_precision_score(
        y_train,
        gradient_boosting_training_probabilities,
    )
)

gradient_boosting_oof_pr_auc = (
    average_precision_score(
        y_train,
        gradient_boosting_final_oof_probabilities,
    )
)

# Calculate the training-to-out-of-fold gaps.
gradient_boosting_recall_gap = (
    gradient_boosting_training_recall
    - gradient_boosting_oof_recall
)

gradient_boosting_pr_auc_gap = (
    gradient_boosting_training_pr_auc
    - gradient_boosting_oof_pr_auc
)

# Create the overfitting comparison table.
gradient_boosting_overfitting_results = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "PR-AUC",
        ],
        "training_score": [
            gradient_boosting_training_recall,
            gradient_boosting_training_pr_auc,
        ],
        "out_of_fold_score": [
            gradient_boosting_oof_recall,
            gradient_boosting_oof_pr_auc,
        ],
        "overfitting_gap": [
            gradient_boosting_recall_gap,
            gradient_boosting_pr_auc_gap,
        ],
    }
).set_index("metric")

display(
    gradient_boosting_overfitting_results.round(4)
)

# Treat a training-to-out-of-fold gap above 0.10
# as evidence of meaningful overfitting.
gradient_boosting_meaningful_overfitting = (
    gradient_boosting_recall_gap > 0.10
    or gradient_boosting_pr_auc_gap > 0.10
)

if gradient_boosting_meaningful_overfitting:
    gradient_boosting_overfitting_status = (
        "Meaningful overfitting detected"
    )
else:
    gradient_boosting_overfitting_status = (
        "No meaningful overfitting detected"
    )

print(
    "\nGradient Boosting overfitting check completed."
)

print(
    "Selected threshold:",
    round(
        gradient_boosting_selected_threshold,
        4,
    ),
)

print(
    "Recall gap:",
    round(
        gradient_boosting_recall_gap,
        4,
    ),
)

print(
    "PR-AUC gap:",
    round(
        gradient_boosting_pr_auc_gap,
        4,
    ),
)

print(
    "Overfitting assessment:",
    gradient_boosting_overfitting_status,
)

print(
    "Final Gradient Boosting configuration:",
    "Initial Gradient Boosting",
)

print("Test set evaluated: False")

Checking the selected Gradient Boosting model for overfitting...


,training_score,out_of_fold_score,overfitting_gap
metric,,,
Recall,0.8883,0.7302,0.1580
PR-AUC,0.6531,0.2620,0.3911



Gradient Boosting overfitting check completed.
Selected threshold: 0.0891
Recall gap: 0.158
PR-AUC gap: 0.3911
Overfitting assessment: Meaningful overfitting detected
Final Gradient Boosting configuration: Initial Gradient Boosting
Test set evaluated: False


In [97]:
# Step 41: Check the tuned Gradient Boosting model for overfitting

print(
    "Checking the tuned Gradient Boosting model "
    "for overfitting..."
)

# Use the tuned model's out-of-fold-selected threshold.
gradient_boosting_tuned_threshold = float(
    gradient_boosting_tuned_results["threshold"]
)

# Fit a separate copy of the tuned model on the
# complete training set.
gradient_boosting_tuned_overfitting_pipeline = clone(
    gradient_boosting_tuned_pipeline
)

gradient_boosting_tuned_overfitting_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample training probabilities.
gradient_boosting_tuned_training_probabilities = (
    gradient_boosting_tuned_overfitting_pipeline
    .predict_proba(X_train)[:, 1]
)

# Apply the same tuned threshold to training and
# out-of-fold probabilities.
gradient_boosting_tuned_training_predictions = (
    gradient_boosting_tuned_training_probabilities
    >= gradient_boosting_tuned_threshold
).astype(int)

gradient_boosting_tuned_oof_predictions = (
    gradient_boosting_tuned_oof_probabilities
    >= gradient_boosting_tuned_threshold
).astype(int)

# Calculate training and out-of-fold recall.
gradient_boosting_tuned_training_recall = (
    recall_score(
        y_train,
        gradient_boosting_tuned_training_predictions,
    )
)

gradient_boosting_tuned_oof_recall = recall_score(
    y_train,
    gradient_boosting_tuned_oof_predictions,
)

# Calculate training and out-of-fold PR-AUC.
gradient_boosting_tuned_training_pr_auc = (
    average_precision_score(
        y_train,
        gradient_boosting_tuned_training_probabilities,
    )
)

gradient_boosting_tuned_oof_pr_auc = (
    average_precision_score(
        y_train,
        gradient_boosting_tuned_oof_probabilities,
    )
)

# Calculate overfitting gaps.
gradient_boosting_tuned_recall_gap = (
    gradient_boosting_tuned_training_recall
    - gradient_boosting_tuned_oof_recall
)

gradient_boosting_tuned_pr_auc_gap = (
    gradient_boosting_tuned_training_pr_auc
    - gradient_boosting_tuned_oof_pr_auc
)

# Display the tuned model's overfitting results.
gradient_boosting_tuned_overfitting_results = (
    pd.DataFrame(
        {
            "metric": [
                "Recall",
                "PR-AUC",
            ],
            "training_score": [
                gradient_boosting_tuned_training_recall,
                gradient_boosting_tuned_training_pr_auc,
            ],
            "out_of_fold_score": [
                gradient_boosting_tuned_oof_recall,
                gradient_boosting_tuned_oof_pr_auc,
            ],
            "overfitting_gap": [
                gradient_boosting_tuned_recall_gap,
                gradient_boosting_tuned_pr_auc_gap,
            ],
        }
    ).set_index("metric")
)

display(
    gradient_boosting_tuned_overfitting_results.round(4)
)

# Use the same 0.10 warning level applied to every model.
gradient_boosting_tuned_meaningful_overfitting = (
    gradient_boosting_tuned_recall_gap > 0.10
    or gradient_boosting_tuned_pr_auc_gap > 0.10
)

if gradient_boosting_tuned_meaningful_overfitting:
    gradient_boosting_tuned_overfitting_status = (
        "Meaningful overfitting detected"
    )
else:
    gradient_boosting_tuned_overfitting_status = (
        "No meaningful overfitting detected"
    )

print(
    "\nTuned Gradient Boosting overfitting check completed."
)

print(
    "Selected threshold:",
    round(
        gradient_boosting_tuned_threshold,
        4,
    ),
)

print(
    "Recall gap:",
    round(
        gradient_boosting_tuned_recall_gap,
        4,
    ),
)

print(
    "PR-AUC gap:",
    round(
        gradient_boosting_tuned_pr_auc_gap,
        4,
    ),
)

print(
    "Overfitting assessment:",
    gradient_boosting_tuned_overfitting_status,
)

print(
    "Initial model recall:",
    round(
        gradient_boosting_initial_results["recall"],
        4,
    ),
)

print(
    "Tuned model recall:",
    round(
        gradient_boosting_tuned_results["recall"],
        4,
    ),
)

print("Final configuration selected: False")
print("Test set evaluated: False")

Checking the tuned Gradient Boosting model for overfitting...


,training_score,out_of_fold_score,overfitting_gap
metric,,,
Recall,0.7084,0.6921,0.0163
PR-AUC,0.3458,0.2930,0.0529



Tuned Gradient Boosting overfitting check completed.
Selected threshold: 0.1116
Recall gap: 0.0163
PR-AUC gap: 0.0529
Overfitting assessment: No meaningful overfitting detected
Initial model recall: 0.7302
Tuned model recall: 0.6921
Final configuration selected: False
Test set evaluated: False


In [98]:
# Step 42: Finalize the Gradient Boosting model

print("Finalizing the Gradient Boosting model...")

# Compare overfitting behavior between the initial
# and tuned Gradient Boosting models.
gradient_boosting_overfitting_comparison = pd.DataFrame(
    {
        "model_stage": [
            "Initial",
            "Tuned",
        ],
        "out_of_fold_recall": [
            gradient_boosting_initial_results["recall"],
            gradient_boosting_tuned_results["recall"],
        ],
        "recall_gap": [
            gradient_boosting_recall_gap,
            gradient_boosting_tuned_recall_gap,
        ],
        "out_of_fold_pr_auc": [
            gradient_boosting_initial_results["pr_auc"],
            gradient_boosting_tuned_results["pr_auc"],
        ],
        "pr_auc_gap": [
            gradient_boosting_pr_auc_gap,
            gradient_boosting_tuned_pr_auc_gap,
        ],
        "meaningful_overfitting": [
            gradient_boosting_meaningful_overfitting,
            gradient_boosting_tuned_meaningful_overfitting,
        ],
    }
).set_index("model_stage")

display(
    gradient_boosting_overfitting_comparison.round(4)
)

# Retain the tuned model because it passes both performance
# guardrails and does not show meaningful overfitting.
gradient_boosting_final_pipeline = clone(
    gradient_boosting_tuned_pipeline
)

gradient_boosting_final_results = (
    gradient_boosting_tuned_results.copy()
)

gradient_boosting_final_oof_probabilities = (
    gradient_boosting_tuned_oof_probabilities.copy()
)

# Record why the initial model was rejected.
gradient_boosting_initial_rejected_for_overfitting = (
    gradient_boosting_meaningful_overfitting
)

# Calculate the recall trade-off accepted for
# improved generalization.
gradient_boosting_recall_tradeoff = (
    gradient_boosting_initial_results["recall"]
    - gradient_boosting_tuned_results["recall"]
)

print("\nGradient Boosting final decision completed.")

print(
    "Selected configuration:",
    "Tuned Gradient Boosting",
)

print(
    "Selected threshold:",
    round(
        gradient_boosting_final_results["threshold"],
        4,
    ),
)

print(
    "Selected recall:",
    round(
        gradient_boosting_final_results["recall"],
        4,
    ),
)

print(
    "Selected balanced accuracy:",
    round(
        gradient_boosting_final_results[
            "balanced_accuracy"
        ],
        4,
    ),
)

print(
    "Selected specificity:",
    round(
        gradient_boosting_final_results["specificity"],
        4,
    ),
)

print(
    "Selected PR-AUC:",
    round(
        gradient_boosting_final_results["pr_auc"],
        4,
    ),
)

print(
    "Recall trade-off versus initial model:",
    round(
        gradient_boosting_recall_tradeoff,
        4,
    ),
)

print(
    "Initial model rejected for overfitting:",
    gradient_boosting_initial_rejected_for_overfitting,
)

print(
    "Selected model has meaningful overfitting:",
    gradient_boosting_tuned_meaningful_overfitting,
)

print("SMOTENC retained: False")
print("Test set evaluated: False")

Finalizing the Gradient Boosting model...


,out_of_fold_recall,recall_gap,out_of_fold_pr_auc,pr_auc_gap,meaningful_overfitting
model_stage,,,,,
Initial,0.7302,0.1580,0.2620,0.3911,True
Tuned,0.6921,0.0163,0.2930,0.0529,False



Gradient Boosting final decision completed.
Selected configuration: Tuned Gradient Boosting
Selected threshold: 0.1116
Selected recall: 0.6921
Selected balanced accuracy: 0.7399
Selected specificity: 0.7876
Selected PR-AUC: 0.293
Recall trade-off versus initial model: 0.0381
Initial model rejected for overfitting: True
Selected model has meaningful overfitting: False
SMOTENC retained: False
Test set evaluated: False


## Gradient Boosting Conclusion

Three Gradient Boosting configurations were evaluated: Initial, Tuned, and Tuned with SMOTENC. Gradient Boosting did not require numerical feature scaling.

The Initial model achieved the highest recall of **0.7302**, but it showed substantial overfitting. Its training-to-validation recall gap was **0.1580**, and its PR-AUC gap was **0.3911**. Therefore, the Initial model was not retained.

SMOTENC reduced recall to **0.6785**, ROC-AUC to **0.7748**, and F2 to **0.4633**. It did not provide a meaningful improvement and was not retained.

The Tuned Gradient Boosting model provided the most reliable performance. Using the maximum out-of-fold F2 threshold of **0.1116**, it achieved:

- Recall: **0.6921**
- ROC-AUC: **0.8101**
- Specificity: **0.7876**
- Balanced accuracy: **0.7399**
- PR-AUC: **0.2930**

The tuned model’s recall gap was **0.0163**, and its PR-AUC gap was **0.0529**. Both were below the **0.10** overfitting threshold, so no meaningful overfitting was detected.

Although tuning reduced recall by **0.0381**, it produced a much more stable model and improved both ROC-AUC and PR-AUC. Therefore, the **Tuned Gradient Boosting model** was retained for the final model comparison.

The test set remained untouched.

## XGBoost

XGBoost is an optimized gradient-boosting algorithm that builds decision trees sequentially while using regularization to control model complexity.

Because the dataset is imbalanced, the initial model uses `scale_pos_weight`, calculated from the ratio of below-threshold to elevated-risk participants. Stratified cross-validation, out-of-fold probability estimation and F2-based threshold optimization are retained.

The test set remains untouched during model development.

In [99]:
# Step 43: Build and evaluate the initial XGBoost model

from xgboost import XGBClassifier

print("Evaluating initial XGBoost...")

# Calculate the imbalance ratio using only the training target.
# This gives greater weight to elevated-risk observations.
xgboost_scale_pos_weight = (
    (y_train == 0).sum()
    / (y_train == 1).sum()
)

print(
    "Training class-weight ratio:",
    round(xgboost_scale_pos_weight, 4),
)

# Create the initial XGBoost pipeline.
xgboost_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                min_child_weight=5,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.0,
                reg_lambda=1.0,
                scale_pos_weight=xgboost_scale_pos_weight,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the
# classification threshold using F2.
(
    xgboost_initial_results,
    xgboost_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="XGBoost",
    pipeline=xgboost_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Identify the model-development stage.
xgboost_initial_results["model_stage"] = "Initial"

# Display the initial XGBoost performance.
display(
    pd.DataFrame(
        [xgboost_initial_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("\nInitial XGBoost evaluation completed.")
print("Imbalance handling: scale_pos_weight")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary comparison metric: Recall")
print(
    "Balanced-accuracy and specificity guardrails applied:",
    True,
)
print("Fine-tuning completed: False")
print("SMOTENC evaluated: False")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating initial XGBoost...
Training class-weight ratio: 10.0463


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
XGBoost,Initial,0.4171,0.7359,0.2256,0.7166,0.7551,0.3431,0.4992,0.7985,0.2759,1166



Initial XGBoost evaluation completed.
Imbalance handling: scale_pos_weight
Threshold selected using: Maximum out-of-fold F2
Primary comparison metric: Recall
Balanced-accuracy and specificity guardrails applied: True
Fine-tuning completed: False
SMOTENC evaluated: False
Ordinary accuracy included: False
Test set evaluated: False


In [100]:
# Step 44: Fine-tune XGBoost using cross-validation PR-AUC

print("Fine-tuning XGBoost...")

# Define reproducible stratified cross-validation.
xgboost_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Include nearby class-weight ratios so the search can determine
# whether slightly weaker or stronger positive-class weighting
# improves probability ranking.
xgboost_weight_options = [
    xgboost_scale_pos_weight * 0.75,
    xgboost_scale_pos_weight,
    xgboost_scale_pos_weight * 1.25,
]

# Search parameters controlling tree complexity,
# learning speed, sampling and regularization.
xgboost_parameter_distributions = {
    "classifier__n_estimators": [
        100,
        200,
        300,
        500,
    ],
    "classifier__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.08,
        0.10,
    ],
    "classifier__max_depth": [
        2,
        3,
        4,
        5,
    ],
    "classifier__min_child_weight": [
        1,
        3,
        5,
        10,
        20,
    ],
    "classifier__subsample": [
        0.60,
        0.80,
        1.00,
    ],
    "classifier__colsample_bytree": [
        0.60,
        0.80,
        1.00,
    ],
    "classifier__gamma": [
        0.0,
        0.10,
        0.25,
        0.50,
        1.00,
    ],
    "classifier__reg_alpha": [
        0.0,
        0.01,
        0.10,
        0.50,
        1.00,
    ],
    "classifier__reg_lambda": [
        1.0,
        2.0,
        5.0,
        10.0,
        20.0,
    ],
    "classifier__scale_pos_weight": (
        xgboost_weight_options
    ),
}

# Use one CPU thread inside each XGBoost model.
# RandomizedSearchCV will parallelize the candidate models.
xgboost_search_pipeline = clone(
    xgboost_initial_pipeline
)

xgboost_search_pipeline.set_params(
    classifier__n_jobs=1
)

# Optimize PR-AUC because the elevated-risk class
# represents only a small portion of the dataset.
xgboost_search = RandomizedSearchCV(
    estimator=xgboost_search_pipeline,
    param_distributions=(
        xgboost_parameter_distributions
    ),
    n_iter=40,
    scoring="average_precision",
    cv=xgboost_cv,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fit only on the training set.
xgboost_search.fit(
    X_train,
    y_train,
)

# Locate the best parameter combination.
best_xgboost_index = xgboost_search.best_index_

# Extract training and validation PR-AUC for
# a preliminary overfitting assessment.
xgboost_training_pr_auc = (
    xgboost_search.cv_results_[
        "mean_train_score"
    ][best_xgboost_index]
)

xgboost_validation_pr_auc = (
    xgboost_search.cv_results_[
        "mean_test_score"
    ][best_xgboost_index]
)

xgboost_search_gap = (
    xgboost_training_pr_auc
    - xgboost_validation_pr_auc
)

print("\nXGBoost fine-tuning completed.")

print("\nBest parameters:")
display(
    xgboost_search.best_params_
)

print(
    "Best cross-validation PR-AUC:",
    round(
        xgboost_search.best_score_,
        4,
    ),
)

print(
    "Training PR-AUC:",
    round(
        xgboost_training_pr_auc,
        4,
    ),
)

print(
    "Validation PR-AUC:",
    round(
        xgboost_validation_pr_auc,
        4,
    ),
)

print(
    "Preliminary overfitting gap:",
    round(
        xgboost_search_gap,
        4,
    ),
)

print("Fine-tuning optimization metric: PR-AUC")
print("Final model-selection metric: Recall")
print("Ordinary accuracy used: False")
print("Threshold optimization completed: False")
print("Test set evaluated: False")

Fine-tuning XGBoost...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

XGBoost fine-tuning completed.

Best parameters:


{'classifier__subsample': 1.0,
 'classifier__scale_pos_weight': np.float64(12.557901907356948),
 'classifier__reg_lambda': 20.0,
 'classifier__reg_alpha': 0.5,
 'classifier__n_estimators': 100,
 'classifier__min_child_weight': 3,
 'classifier__max_depth': 4,
 'classifier__learning_rate': 0.03,
 'classifier__gamma': 0.5,
 'classifier__colsample_bytree': 1.0}

Best cross-validation PR-AUC: 0.2993
Training PR-AUC: 0.481
Validation PR-AUC: 0.2993
Preliminary overfitting gap: 0.1817
Fine-tuning optimization metric: PR-AUC
Final model-selection metric: Recall
Ordinary accuracy used: False
Threshold optimization completed: False
Test set evaluated: False


In [101]:
# Step 45: Evaluate and compare the tuned XGBoost model

print(
    "Evaluating tuned XGBoost with "
    "out-of-fold threshold optimization..."
)

# Copy the best pipeline identified by RandomizedSearchCV.
xgboost_tuned_pipeline = clone(
    xgboost_search.best_estimator_
)

# Restore parallel processing for the final candidate.
xgboost_tuned_pipeline.set_params(
    classifier__n_jobs=-1
)

# Generate fresh out-of-fold probabilities and select
# the F2-based classification threshold.
(
    xgboost_tuned_results,
    xgboost_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="XGBoost",
    pipeline=xgboost_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Identify the model-development stage.
xgboost_tuned_results["model_stage"] = "Tuned"

# Combine the initial and tuned results.
xgboost_tuning_comparison = pd.DataFrame(
    [
        xgboost_initial_results,
        xgboost_tuned_results,
    ]
)

# Display both XGBoost configurations.
display(
    xgboost_tuning_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the changes produced by fine-tuning.
xgboost_recall_change = (
    xgboost_tuned_results["recall"]
    - xgboost_initial_results["recall"]
)

xgboost_balanced_accuracy_change = (
    xgboost_tuned_results["balanced_accuracy"]
    - xgboost_initial_results["balanced_accuracy"]
)

xgboost_specificity_change = (
    xgboost_tuned_results["specificity"]
    - xgboost_initial_results["specificity"]
)

xgboost_pr_auc_change = (
    xgboost_tuned_results["pr_auc"]
    - xgboost_initial_results["pr_auc"]
)

# Verify that the tuned model passes the project guardrails.
xgboost_tuned_passes_guardrails = (
    xgboost_tuned_results["balanced_accuracy"] >= 0.70
    and xgboost_tuned_results["specificity"] >= 0.65
)

print("\nTuned XGBoost evaluation completed.")

print("\nTuned model versus initial model:")

print(
    "Recall change:",
    round(xgboost_recall_change, 4),
)

print(
    "Balanced-accuracy change:",
    round(
        xgboost_balanced_accuracy_change,
        4,
    ),
)

print(
    "Specificity change:",
    round(xgboost_specificity_change, 4),
)

print(
    "PR-AUC change:",
    round(xgboost_pr_auc_change, 4),
)

print(
    "Tuned model passes both guardrails:",
    xgboost_tuned_passes_guardrails,
)

print(
    "Selected threshold:",
    round(
        xgboost_tuned_results["threshold"],
        4,
    ),
)

print(
    "Preliminary tuning PR-AUC gap:",
    round(xgboost_search_gap, 4),
)

print("Primary comparison metric: Recall")
print("Overfitting check completed: False")
print("SMOTENC evaluated: False")
print("Final configuration selected: False")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating tuned XGBoost with out-of-fold threshold optimization...


threshold  balanced_accuracy  precision  recall  \
model   model_stage                                                    
XGBoost Initial         0.4171             0.7359     0.2256  0.7166   
        Tuned           0.5115             0.7430     0.2255  0.7384   

                     specificity     f1     f2  roc_auc  pr_auc  \
model   model_stage                                               
XGBoost Initial           0.7551 0.3431 0.4992   0.7985  0.2759   
        Tuned             0.7475 0.3454 0.5075   0.8008  0.2838   

                     predicted_positive_cases  
model   model_stage                            
XGBoost Initial                          1166  
        Tuned                            1202


Tuned XGBoost evaluation completed.

Tuned model versus initial model:
Recall change: 0.0218
Balanced-accuracy change: 0.0071
Specificity change: -0.0076
PR-AUC change: 0.0079
Tuned model passes both guardrails: True
Selected threshold: 0.5115
Preliminary tuning PR-AUC gap: 0.1817
Primary comparison metric: Recall
Overfitting check completed: False
SMOTENC evaluated: False
Final configuration selected: False
Ordinary accuracy included: False
Test set evaluated: False


In [102]:
# Step 46: Compare initial and tuned XGBoost overfitting

print(
    "Checking initial and tuned XGBoost models "
    "for overfitting..."
)

# Store the two XGBoost configurations that must
# be checked using the same evaluation method.
xgboost_overfitting_candidates = {
    "Initial": {
        "pipeline": xgboost_initial_pipeline,
        "threshold": float(
            xgboost_initial_results["threshold"]
        ),
        "oof_probabilities": (
            xgboost_initial_oof_probabilities
        ),
    },
    "Tuned": {
        "pipeline": xgboost_tuned_pipeline,
        "threshold": float(
            xgboost_tuned_results["threshold"]
        ),
        "oof_probabilities": (
            xgboost_tuned_oof_probabilities
        ),
    },
}

# Collect overfitting results for both configurations.
xgboost_overfitting_records = []

for stage, candidate in (
    xgboost_overfitting_candidates.items()
):
    # Fit a separate pipeline copy on the complete
    # training set.
    fitted_pipeline = clone(
        candidate["pipeline"]
    )

    fitted_pipeline.fit(
        X_train,
        y_train,
    )

    # Generate in-sample training probabilities.
    training_probabilities = (
        fitted_pipeline.predict_proba(X_train)[:, 1]
    )

    # Retrieve the out-of-fold probabilities.
    oof_probabilities = candidate[
        "oof_probabilities"
    ]

    selected_threshold = candidate["threshold"]

    # Apply the same out-of-fold-selected threshold
    # to both probability sets.
    training_predictions = (
        training_probabilities
        >= selected_threshold
    ).astype(int)

    oof_predictions = (
        oof_probabilities
        >= selected_threshold
    ).astype(int)

    # Calculate training and out-of-fold recall.
    training_recall = recall_score(
        y_train,
        training_predictions,
    )

    oof_recall = recall_score(
        y_train,
        oof_predictions,
    )

    # Calculate training and out-of-fold PR-AUC.
    training_pr_auc = average_precision_score(
        y_train,
        training_probabilities,
    )

    oof_pr_auc = average_precision_score(
        y_train,
        oof_probabilities,
    )

    # Calculate generalization gaps.
    recall_gap = (
        training_recall
        - oof_recall
    )

    pr_auc_gap = (
        training_pr_auc
        - oof_pr_auc
    )

    # A gap above 0.10 indicates meaningful overfitting.
    meaningful_overfitting = (
        recall_gap > 0.10
        or pr_auc_gap > 0.10
    )

    xgboost_overfitting_records.append(
        {
            "model_stage": stage,
            "threshold": selected_threshold,
            "training_recall": training_recall,
            "out_of_fold_recall": oof_recall,
            "recall_gap": recall_gap,
            "training_pr_auc": training_pr_auc,
            "out_of_fold_pr_auc": oof_pr_auc,
            "pr_auc_gap": pr_auc_gap,
            "meaningful_overfitting": (
                meaningful_overfitting
            ),
        }
    )

# Create and display the comparison table.
xgboost_overfitting_comparison = pd.DataFrame(
    xgboost_overfitting_records
).set_index("model_stage")

display(
    xgboost_overfitting_comparison.round(4)
)

# Save the individual assessments for later selection.
xgboost_initial_meaningful_overfitting = bool(
    xgboost_overfitting_comparison.loc[
        "Initial",
        "meaningful_overfitting",
    ]
)

xgboost_tuned_meaningful_overfitting = bool(
    xgboost_overfitting_comparison.loc[
        "Tuned",
        "meaningful_overfitting",
    ]
)

xgboost_initial_recall_gap = float(
    xgboost_overfitting_comparison.loc[
        "Initial",
        "recall_gap",
    ]
)

xgboost_initial_pr_auc_gap = float(
    xgboost_overfitting_comparison.loc[
        "Initial",
        "pr_auc_gap",
    ]
)

xgboost_tuned_recall_gap = float(
    xgboost_overfitting_comparison.loc[
        "Tuned",
        "recall_gap",
    ]
)

xgboost_tuned_pr_auc_gap = float(
    xgboost_overfitting_comparison.loc[
        "Tuned",
        "pr_auc_gap",
    ]
)

print("\nXGBoost overfitting comparison completed.")

print(
    "Initial model meaningful overfitting:",
    xgboost_initial_meaningful_overfitting,
)

print(
    "Tuned model meaningful overfitting:",
    xgboost_tuned_meaningful_overfitting,
)

print(
    "Tuned recall improvement:",
    round(xgboost_recall_change, 4),
)

print("Final configuration selected: False")
print("SMOTENC evaluated: False")
print("Test set evaluated: False")

Checking initial and tuned XGBoost models for overfitting...


,threshold,training_recall,out_of_fold_recall,recall_gap,training_pr_auc,out_of_fold_pr_auc,pr_auc_gap,meaningful_overfitting
model_stage,,,,,,,,
Initial,0.4171,0.9264,0.7166,0.2098,0.5691,0.2759,0.2932,True
Tuned,0.5115,0.8583,0.7384,0.1199,0.4477,0.2838,0.1638,True



XGBoost overfitting comparison completed.
Initial model meaningful overfitting: True
Tuned model meaningful overfitting: True
Tuned recall improvement: 0.0218
Final configuration selected: False
SMOTENC evaluated: False
Test set evaluated: False


In [103]:
# Step 47: Run a targeted regularized XGBoost search

print(
    "Searching for a more strongly regularized "
    "XGBoost configuration..."
)

# Focus the second search on settings designed to
# reduce the overfitting found in both earlier models.
xgboost_regularized_parameter_distributions = {
    # Use fewer and shallower trees.
    "classifier__n_estimators": [
        75,
        100,
        150,
        200,
    ],
    "classifier__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05,
    ],
    "classifier__max_depth": [
        1,
        2,
        3,
    ],

    # Require more observations before creating
    # specialized branches.
    "classifier__min_child_weight": [
        10,
        20,
        30,
        40,
    ],

    # Require a larger loss reduction before
    # adding a new tree split.
    "classifier__gamma": [
        1.0,
        2.0,
        3.0,
        5.0,
    ],

    # Apply stronger L1 and L2 regularization.
    "classifier__reg_alpha": [
        1.0,
        2.0,
        5.0,
        10.0,
    ],
    "classifier__reg_lambda": [
        20.0,
        30.0,
        50.0,
        75.0,
    ],

    # Add row and feature sampling to reduce variance.
    "classifier__subsample": [
        0.70,
        0.85,
        1.00,
    ],
    "classifier__colsample_bytree": [
        0.70,
        0.85,
        1.00,
    ],

    # Retain imbalance-aware weighting.
    "classifier__scale_pos_weight": [
        xgboost_scale_pos_weight * 0.75,
        xgboost_scale_pos_weight,
        xgboost_scale_pos_weight * 1.25,
    ],
}

# Begin with the initial pipeline and search only
# the more conservative parameter ranges.
xgboost_regularized_search_pipeline = clone(
    xgboost_initial_pipeline
)

# Avoid nested CPU parallelism.
xgboost_regularized_search_pipeline.set_params(
    classifier__n_jobs=1
)

xgboost_regularized_search = RandomizedSearchCV(
    estimator=xgboost_regularized_search_pipeline,
    param_distributions=(
        xgboost_regularized_parameter_distributions
    ),
    n_iter=30,
    scoring="average_precision",
    cv=xgboost_cv,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Use only the training set.
xgboost_regularized_search.fit(
    X_train,
    y_train,
)

# Locate the selected parameter combination.
best_regularized_xgboost_index = (
    xgboost_regularized_search.best_index_
)

# Extract cross-validation training and validation PR-AUC.
xgboost_regularized_training_pr_auc = (
    xgboost_regularized_search.cv_results_[
        "mean_train_score"
    ][best_regularized_xgboost_index]
)

xgboost_regularized_validation_pr_auc = (
    xgboost_regularized_search.cv_results_[
        "mean_test_score"
    ][best_regularized_xgboost_index]
)

xgboost_regularized_search_gap = (
    xgboost_regularized_training_pr_auc
    - xgboost_regularized_validation_pr_auc
)

print(
    "\nRegularized XGBoost search completed."
)

print("\nBest regularized parameters:")
display(
    xgboost_regularized_search.best_params_
)

print(
    "Training PR-AUC:",
    round(
        xgboost_regularized_training_pr_auc,
        4,
    ),
)

print(
    "Validation PR-AUC:",
    round(
        xgboost_regularized_validation_pr_auc,
        4,
    ),
)

print(
    "Preliminary PR-AUC gap:",
    round(
        xgboost_regularized_search_gap,
        4,
    ),
)

print(
    "Previous tuned PR-AUC gap:",
    round(xgboost_search_gap, 4),
)

print(
    "Preliminary gap reduction:",
    round(
        xgboost_search_gap
        - xgboost_regularized_search_gap,
        4,
    ),
)

print("Optimization metric: PR-AUC")
print("Threshold optimization completed: False")
print("Final configuration selected: False")
print("SMOTENC evaluated: False")
print("Test set evaluated: False")

Searching for a more strongly regularized XGBoost configuration...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Regularized XGBoost search completed.

Best regularized parameters:


{'classifier__subsample': 0.85,
 'classifier__scale_pos_weight': np.float64(10.046321525885558),
 'classifier__reg_lambda': 75.0,
 'classifier__reg_alpha': 5.0,
 'classifier__n_estimators': 75,
 'classifier__min_child_weight': 10,
 'classifier__max_depth': 3,
 'classifier__learning_rate': 0.05,
 'classifier__gamma': 3.0,
 'classifier__colsample_bytree': 0.85}

Training PR-AUC: 0.3788
Validation PR-AUC: 0.3017
Preliminary PR-AUC gap: 0.0771
Previous tuned PR-AUC gap: 0.1817
Preliminary gap reduction: 0.1046
Optimization metric: PR-AUC
Threshold optimization completed: False
Final configuration selected: False
SMOTENC evaluated: False
Test set evaluated: False


In [104]:
# Step 48: Evaluate the regularized XGBoost candidate

print(
    "Evaluating the regularized XGBoost candidate..."
)

# Copy the best strongly regularized pipeline.
xgboost_regularized_pipeline = clone(
    xgboost_regularized_search.best_estimator_
)

# Restore XGBoost parallel processing.
xgboost_regularized_pipeline.set_params(
    classifier__n_jobs=-1
)

# Generate fresh out-of-fold probabilities and select
# the F2-based classification threshold.
(
    xgboost_regularized_results,
    xgboost_regularized_oof_probabilities,
) = create_oof_evaluation(
    model_name="XGBoost",
    pipeline=xgboost_regularized_pipeline,
    X_training=X_train,
    y_training=y_train,
)

# Label the model-development stage.
xgboost_regularized_results[
    "model_stage"
] = "Regularized"

# Compare all XGBoost configurations evaluated so far.
xgboost_model_comparison = pd.DataFrame(
    [
        xgboost_initial_results,
        xgboost_tuned_results,
        xgboost_regularized_results,
    ]
)

display(
    xgboost_model_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Fit a separate copy on the complete training set
# for an in-sample versus out-of-fold comparison.
xgboost_regularized_overfitting_pipeline = clone(
    xgboost_regularized_pipeline
)

xgboost_regularized_overfitting_pipeline.fit(
    X_train,
    y_train,
)

# Generate in-sample training probabilities.
xgboost_regularized_training_probabilities = (
    xgboost_regularized_overfitting_pipeline
    .predict_proba(X_train)[:, 1]
)

# Use the threshold selected from out-of-fold probabilities.
xgboost_regularized_threshold = float(
    xgboost_regularized_results["threshold"]
)

# Apply the same threshold to training and OOF probabilities.
xgboost_regularized_training_predictions = (
    xgboost_regularized_training_probabilities
    >= xgboost_regularized_threshold
).astype(int)

xgboost_regularized_oof_predictions = (
    xgboost_regularized_oof_probabilities
    >= xgboost_regularized_threshold
).astype(int)

# Calculate training and out-of-fold recall.
xgboost_regularized_training_recall = recall_score(
    y_train,
    xgboost_regularized_training_predictions,
)

xgboost_regularized_oof_recall = recall_score(
    y_train,
    xgboost_regularized_oof_predictions,
)

# Calculate training and out-of-fold PR-AUC.
xgboost_regularized_training_pr_auc_final = (
    average_precision_score(
        y_train,
        xgboost_regularized_training_probabilities,
    )
)

xgboost_regularized_oof_pr_auc = (
    average_precision_score(
        y_train,
        xgboost_regularized_oof_probabilities,
    )
)

# Calculate final overfitting gaps.
xgboost_regularized_recall_gap = (
    xgboost_regularized_training_recall
    - xgboost_regularized_oof_recall
)

xgboost_regularized_pr_auc_gap = (
    xgboost_regularized_training_pr_auc_final
    - xgboost_regularized_oof_pr_auc
)

# Determine whether meaningful overfitting remains.
xgboost_regularized_meaningful_overfitting = (
    xgboost_regularized_recall_gap > 0.10
    or xgboost_regularized_pr_auc_gap > 0.10
)

# Verify the project performance guardrails.
xgboost_regularized_passes_guardrails = (
    xgboost_regularized_results[
        "balanced_accuracy"
    ] >= 0.70
    and xgboost_regularized_results[
        "specificity"
    ] >= 0.65
)

# Display the regularized overfitting assessment.
xgboost_regularized_overfitting_results = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "PR-AUC",
        ],
        "training_score": [
            xgboost_regularized_training_recall,
            xgboost_regularized_training_pr_auc_final,
        ],
        "out_of_fold_score": [
            xgboost_regularized_oof_recall,
            xgboost_regularized_oof_pr_auc,
        ],
        "overfitting_gap": [
            xgboost_regularized_recall_gap,
            xgboost_regularized_pr_auc_gap,
        ],
    }
).set_index("metric")

print("\nRegularized XGBoost overfitting check:")

display(
    xgboost_regularized_overfitting_results.round(4)
)

# Calculate changes compared with the previous tuned model.
xgboost_regularized_recall_change = (
    xgboost_regularized_results["recall"]
    - xgboost_tuned_results["recall"]
)

xgboost_regularized_pr_auc_change = (
    xgboost_regularized_results["pr_auc"]
    - xgboost_tuned_results["pr_auc"]
)

print(
    "\nRegularized XGBoost evaluation completed."
)

print(
    "Regularized threshold:",
    round(xgboost_regularized_threshold, 4),
)

print(
    "Recall:",
    round(
        xgboost_regularized_results["recall"],
        4,
    ),
)

print(
    "Recall change versus tuned model:",
    round(
        xgboost_regularized_recall_change,
        4,
    ),
)

print(
    "PR-AUC:",
    round(
        xgboost_regularized_results["pr_auc"],
        4,
    ),
)

print(
    "PR-AUC change versus tuned model:",
    round(
        xgboost_regularized_pr_auc_change,
        4,
    ),
)

print(
    "Recall overfitting gap:",
    round(xgboost_regularized_recall_gap, 4),
)

print(
    "PR-AUC overfitting gap:",
    round(xgboost_regularized_pr_auc_gap, 4),
)

print(
    "Meaningful overfitting:",
    xgboost_regularized_meaningful_overfitting,
)

print(
    "Passes both performance guardrails:",
    xgboost_regularized_passes_guardrails,
)

print("Final configuration selected: False")
print("SMOTENC evaluated: False")
print("Test set evaluated: False")

Evaluating the regularized XGBoost candidate...


threshold  balanced_accuracy  precision  recall  \
model   model_stage                                                    
XGBoost Initial         0.4171             0.7359     0.2256  0.7166   
        Tuned           0.5115             0.7430     0.2255  0.7384   
        Regularized     0.4624             0.7497     0.2202  0.7711   

                     specificity     f1     f2  roc_auc  pr_auc  \
model   model_stage                                               
XGBoost Initial           0.7551 0.3431 0.4992   0.7985  0.2759   
        Tuned             0.7475 0.3454 0.5075   0.8008  0.2838   
        Regularized       0.7282 0.3426 0.5140   0.8093  0.2935   

                     predicted_positive_cases  
model   model_stage                            
XGBoost Initial                          1166  
        Tuned                            1202  
        Regularized                      1285


Regularized XGBoost overfitting check:


,training_score,out_of_fold_score,overfitting_gap
metric,,,
Recall,0.8256,0.7711,0.0545
PR-AUC,0.3617,0.2935,0.0683



Regularized XGBoost evaluation completed.
Regularized threshold: 0.4624
Recall: 0.7711
Recall change versus tuned model: 0.0327
PR-AUC: 0.2935
PR-AUC change versus tuned model: 0.0096
Recall overfitting gap: 0.0545
PR-AUC overfitting gap: 0.0683
Meaningful overfitting: False
Passes both performance guardrails: True
Final configuration selected: False
SMOTENC evaluated: False
Test set evaluated: False


## XGBoost Conclusion

Three XGBoost configurations were evaluated: Initial, Tuned, and Regularized. XGBoost did not require numerical feature scaling.

The Initial model achieved a recall of **0.7166**. Hyperparameter tuning improved recall to **0.7384**, ROC-AUC to **0.8008**, and PR-AUC to **0.2838**.

Additional regularization produced the strongest and most stable XGBoost configuration. Compared with the tuned model, recall increased by **0.0327**, and PR-AUC increased by **0.0096**.

Using the maximum out-of-fold F2 threshold of **0.4624**, the Regularized XGBoost achieved:

- Recall: **0.7711**
- ROC-AUC: **0.8093**
- Specificity: **0.7282**
- Balanced accuracy: **0.7497**
- PR-AUC: **0.2935**

The training-to-validation recall gap was **0.0545**, and the PR-AUC gap was **0.0683**. Both were below the **0.10** overfitting threshold, so no meaningful overfitting was detected.

XGBoost handled class imbalance through its internal class-weighting configuration, so an additional SMOTENC experiment was not required.

The **Regularized XGBoost model** was retained for the final model comparison. The test set remained untouched.

## CatBoost

CatBoost is a gradient-boosting model that can capture nonlinear relationships and interactions between predictors.

Because the target is imbalanced, the initial CatBoost model uses class weights. Its probability threshold is selected from out-of-fold training predictions using the maximum F2 score while enforcing the balanced-accuracy and specificity guardrails.

The test set remains untouched.

In [108]:
# Step 51: Build and evaluate the initial CatBoost model

print("Evaluating initial CatBoost...")

# CatBoost automatically calculates class weights from each
# cross-validation training fold.
catboost_initial_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            CatBoostClassifier(
                loss_function="Logloss",
                auto_class_weights="Balanced",
                random_seed=RANDOM_STATE,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            ),
        ),
    ]
)

# Generate out-of-fold probabilities and select the threshold
# using only the training data.
(
    catboost_initial_results,
    catboost_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="CatBoost",
    pipeline=catboost_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

catboost_initial_results["model_stage"] = "Initial"

# Display the initial CatBoost results.
display(
    pd.DataFrame([catboost_initial_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("\nInitial CatBoost evaluation completed.")
print("Imbalance handling: CatBoost automatic balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Fine-tuning metric: PR-AUC")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating initial CatBoost...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
CatBoost,Initial,0.2739,0.7208,0.2445,0.6376,0.8039,0.3535,0.4825,0.7845,0.2545,957



Initial CatBoost evaluation completed.
Imbalance handling: CatBoost automatic balanced class weights
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Fine-tuning metric: PR-AUC
Ordinary accuracy included: False
Test set evaluated: False


In [110]:
# Step 52: Fine-tune CatBoost using stratified cross-validation PR-AUC

print("Fine-tuning CatBoost...")

# Define the cross-validation strategy inside this cell so that
# the tuning process does not depend on an earlier variable.
catboost_cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Use one CatBoost processing thread per search job.
# RandomizedSearchCV performs the parallel processing.
catboost_search_pipeline = clone(
    catboost_initial_pipeline
).set_params(
    classifier__thread_count=1
)

# Search parameters controlling model complexity,
# regularization, learning speed, and randomness.
catboost_parameter_distributions = {
    "classifier__iterations": [
        150,
        250,
        400,
        600,
    ],
    "classifier__depth": [
        3,
        4,
        5,
        6,
        7,
    ],
    "classifier__learning_rate": [
        0.02,
        0.03,
        0.05,
        0.08,
        0.10,
    ],
    "classifier__l2_leaf_reg": [
        3,
        5,
        10,
        20,
        40,
    ],
    "classifier__random_strength": [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0,
    ],
    "classifier__bagging_temperature": [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0,
    ],
    "classifier__border_count": [
        32,
        64,
        128,
    ],
}

# Select hyperparameters using validation PR-AUC.
catboost_search = RandomizedSearchCV(
    estimator=catboost_search_pipeline,
    param_distributions=catboost_parameter_distributions,
    n_iter=25,
    scoring="average_precision",
    cv=catboost_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Use only the training set.
catboost_search.fit(X_train, y_train)

# Locate the selected parameter combination.
best_catboost_index = catboost_search.best_index_

# Extract training and validation PR-AUC.
catboost_search_training_pr_auc = (
    catboost_search.cv_results_["mean_train_score"][
        best_catboost_index
    ]
)

catboost_search_validation_pr_auc = (
    catboost_search.cv_results_["mean_test_score"][
        best_catboost_index
    ]
)

catboost_search_pr_auc_gap = (
    catboost_search_training_pr_auc
    - catboost_search_validation_pr_auc
)

print("\nCatBoost fine-tuning completed.")

print("\nBest parameters:")
display(catboost_search.best_params_)

print(
    "Training PR-AUC:",
    round(catboost_search_training_pr_auc, 4),
)
print(
    "Validation PR-AUC:",
    round(catboost_search_validation_pr_auc, 4),
)
print(
    "Preliminary PR-AUC gap:",
    round(catboost_search_pr_auc_gap, 4),
)

print("Optimization metric: PR-AUC")
print("Cross-validation: 5-fold stratified")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Fine-tuning CatBoost...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

CatBoost fine-tuning completed.

Best parameters:


{'classifier__random_strength': 5.0,
 'classifier__learning_rate': 0.05,
 'classifier__l2_leaf_reg': 10,
 'classifier__iterations': 250,
 'classifier__depth': 3,
 'classifier__border_count': 32,
 'classifier__bagging_temperature': 0.0}

Training PR-AUC: 0.4428
Validation PR-AUC: 0.3068
Preliminary PR-AUC gap: 0.136
Optimization metric: PR-AUC
Cross-validation: 5-fold stratified
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [111]:
# Step 53: Evaluate the tuned CatBoost model with threshold optimization

print("Evaluating tuned CatBoost...")

# Copy the best pipeline selected during fine-tuning.
catboost_tuned_pipeline = clone(
    catboost_search.best_estimator_
)

# Allow CatBoost to use the available processing threads
# during the final out-of-fold evaluation.
catboost_tuned_pipeline.set_params(
    classifier__thread_count=-1
)

# Generate new out-of-fold probabilities using the complete
# tuned pipeline and select the maximum-F2 threshold.
(
    catboost_tuned_results,
    catboost_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="CatBoost",
    pipeline=catboost_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

catboost_tuned_results["model_stage"] = "Tuned"

# Compare the initial and tuned CatBoost configurations.
catboost_initial_tuned_comparison = pd.DataFrame(
    [
        catboost_initial_results,
        catboost_tuned_results,
    ]
)

display(
    catboost_initial_tuned_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the changes caused by fine-tuning.
catboost_recall_change = (
    catboost_tuned_results["recall"]
    - catboost_initial_results["recall"]
)

catboost_specificity_change = (
    catboost_tuned_results["specificity"]
    - catboost_initial_results["specificity"]
)

catboost_balanced_accuracy_change = (
    catboost_tuned_results["balanced_accuracy"]
    - catboost_initial_results["balanced_accuracy"]
)

catboost_pr_auc_change = (
    catboost_tuned_results["pr_auc"]
    - catboost_initial_results["pr_auc"]
)

# Verify the tuned model's performance guardrails.
catboost_tuned_passes_guardrails = (
    catboost_tuned_results["balanced_accuracy"] >= 0.70
    and catboost_tuned_results["specificity"] >= 0.65
)

print("\nTuned CatBoost evaluation completed.")

print(
    "Recall change:",
    round(catboost_recall_change, 4),
)
print(
    "Specificity change:",
    round(catboost_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(catboost_balanced_accuracy_change, 4),
)
print(
    "PR-AUC change:",
    round(catboost_pr_auc_change, 4),
)
print(
    "Tuned model passes guardrails:",
    catboost_tuned_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Test set evaluated: False")

Evaluating tuned CatBoost...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                    
CatBoost Initial         0.2739             0.7208     0.2445  0.6376   
         Tuned           0.4913             0.7558     0.2433  0.7411   

                      specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                               
CatBoost Initial           0.8039 0.3535 0.4825   0.7845  0.2545   
         Tuned             0.7705 0.3663 0.5259   0.8152  0.2998   

                      predicted_positive_cases  
model    model_stage                            
CatBoost Initial                           957  
         Tuned                            1118


Tuned CatBoost evaluation completed.
Recall change: 0.1035
Specificity change: -0.0334
Balanced-accuracy change: 0.0351
PR-AUC change: 0.0454
Tuned model passes guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Test set evaluated: False


In [112]:
# Step 54: Check the tuned CatBoost model for overfitting

print("Checking tuned CatBoost for overfitting...")

# Fit a separate copy of the tuned pipeline on the complete
# training set. The test set remains untouched.
catboost_tuned_fitted_pipeline = clone(
    catboost_tuned_pipeline
)

catboost_tuned_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate probabilities for the complete training set.
catboost_training_probabilities = (
    catboost_tuned_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Use the same threshold selected from out-of-fold predictions.
catboost_tuned_threshold = float(
    catboost_tuned_results["threshold"]
)

catboost_training_predictions = (
    catboost_training_probabilities
    >= catboost_tuned_threshold
).astype(int)

# Calculate training metrics.
catboost_training_recall = recall_score(
    y_train,
    catboost_training_predictions,
)

catboost_training_pr_auc = average_precision_score(
    y_train,
    catboost_training_probabilities,
)

# Retrieve the unbiased out-of-fold metrics.
catboost_oof_recall = catboost_tuned_results["recall"]
catboost_oof_pr_auc = catboost_tuned_results["pr_auc"]

# Calculate train–OOF performance gaps.
catboost_recall_gap = (
    catboost_training_recall
    - catboost_oof_recall
)

catboost_pr_auc_gap = (
    catboost_training_pr_auc
    - catboost_oof_pr_auc
)

# Flag meaningful overfitting when either gap exceeds 0.10.
catboost_tuned_meaningful_overfitting = (
    catboost_recall_gap > 0.10
    or catboost_pr_auc_gap > 0.10
)

# Display the training and out-of-fold results together.
catboost_overfitting_comparison = pd.DataFrame(
    {
        "dataset": [
            "Complete training data",
            "Out-of-fold validation",
        ],
        "recall": [
            catboost_training_recall,
            catboost_oof_recall,
        ],
        "pr_auc": [
            catboost_training_pr_auc,
            catboost_oof_pr_auc,
        ],
    }
).set_index("dataset")

display(
    catboost_overfitting_comparison.round(4)
)

print("\nTuned CatBoost overfitting assessment completed.")
print(
    "Recall gap:",
    round(catboost_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(catboost_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting detected:",
    catboost_tuned_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training predictions")
print("Test set evaluated: False")

Checking tuned CatBoost for overfitting...


,recall,pr_auc
dataset,,
Complete training data,0.8256,0.4156
Out-of-fold validation,0.7411,0.2998



Tuned CatBoost overfitting assessment completed.
Recall gap: 0.0845
PR-AUC gap: 0.1157
Meaningful overfitting detected: True
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training predictions
Test set evaluated: False


In [113]:
# Step 55: Search for a more strongly regularized CatBoost model

print("Searching for a more strongly regularized CatBoost model...")

# Start from the initial CatBoost pipeline so the search does not
# inherit any unnecessary fitted state.
catboost_regularization_pipeline = clone(
    catboost_initial_pipeline
).set_params(
    classifier__thread_count=1
)

# Use shallower trees, stronger L2 regularization, greater
# randomization, and controlled feature sampling to reduce overfitting.
catboost_regularization_parameters = {
    "classifier__iterations": [
        100,
        150,
        200,
        250,
        300,
    ],
    "classifier__depth": [
        2,
        3,
        4,
    ],
    "classifier__learning_rate": [
        0.02,
        0.03,
        0.04,
        0.05,
    ],
    "classifier__l2_leaf_reg": [
        20,
        30,
        40,
        60,
        80,
        100,
    ],
    "classifier__random_strength": [
        5.0,
        7.5,
        10.0,
        15.0,
        20.0,
    ],
    "classifier__bagging_temperature": [
        0.5,
        1.0,
        2.0,
        3.0,
        5.0,
    ],
    "classifier__rsm": [
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    "classifier__border_count": [
        32,
        64,
    ],
}

# Optimize validation PR-AUC using stratified cross-validation.
catboost_regularization_search = RandomizedSearchCV(
    estimator=catboost_regularization_pipeline,
    param_distributions=catboost_regularization_parameters,
    n_iter=25,
    scoring="average_precision",
    cv=catboost_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Use only the training set.
catboost_regularization_search.fit(
    X_train,
    y_train,
)

# Extract the selected search result.
best_catboost_regularization_index = (
    catboost_regularization_search.best_index_
)

catboost_regularization_training_pr_auc = (
    catboost_regularization_search.cv_results_[
        "mean_train_score"
    ][best_catboost_regularization_index]
)

catboost_regularization_validation_pr_auc = (
    catboost_regularization_search.cv_results_[
        "mean_test_score"
    ][best_catboost_regularization_index]
)

catboost_regularization_search_gap = (
    catboost_regularization_training_pr_auc
    - catboost_regularization_validation_pr_auc
)

print("\nRegularized CatBoost search completed.")

print("\nBest regularized parameters:")
display(
    catboost_regularization_search.best_params_
)

print(
    "Training PR-AUC:",
    round(
        catboost_regularization_training_pr_auc,
        4,
    ),
)
print(
    "Validation PR-AUC:",
    round(
        catboost_regularization_validation_pr_auc,
        4,
    ),
)
print(
    "Preliminary PR-AUC gap:",
    round(
        catboost_regularization_search_gap,
        4,
    ),
)

print("Optimization metric: PR-AUC")
print("Purpose: Reduce tuned CatBoost overfitting")
print("Cross-validation: 5-fold stratified")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Searching for a more strongly regularized CatBoost model...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Regularized CatBoost search completed.

Best regularized parameters:


{'classifier__rsm': 0.7,
 'classifier__random_strength': 15.0,
 'classifier__learning_rate': 0.02,
 'classifier__l2_leaf_reg': 100,
 'classifier__iterations': 300,
 'classifier__depth': 4,
 'classifier__border_count': 64,
 'classifier__bagging_temperature': 5.0}

Training PR-AUC: 0.3854
Validation PR-AUC: 0.3049
Preliminary PR-AUC gap: 0.0804
Optimization metric: PR-AUC
Purpose: Reduce tuned CatBoost overfitting
Cross-validation: 5-fold stratified
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [114]:
# Step 56: Evaluate the regularized CatBoost model

print("Evaluating regularized CatBoost...")

# Copy the best pipeline from the focused regularization search.
catboost_regularized_pipeline = clone(
    catboost_regularization_search.best_estimator_
)

# Allow CatBoost to use the available processing threads
# during out-of-fold evaluation.
catboost_regularized_pipeline.set_params(
    classifier__thread_count=-1
)

# Generate out-of-fold probabilities and select the
# maximum-F2 threshold using only the training data.
(
    catboost_regularized_results,
    catboost_regularized_oof_probabilities,
) = create_oof_evaluation(
    model_name="CatBoost",
    pipeline=catboost_regularized_pipeline,
    X_training=X_train,
    y_training=y_train,
)

catboost_regularized_results[
    "model_stage"
] = "Regularized"

# Compare all completed CatBoost configurations.
catboost_regularized_comparison = pd.DataFrame(
    [
        catboost_initial_results,
        catboost_tuned_results,
        catboost_regularized_results,
    ]
)

display(
    catboost_regularized_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes relative to the tuned CatBoost model.
catboost_regularized_recall_change = (
    catboost_regularized_results["recall"]
    - catboost_tuned_results["recall"]
)

catboost_regularized_specificity_change = (
    catboost_regularized_results["specificity"]
    - catboost_tuned_results["specificity"]
)

catboost_regularized_balanced_accuracy_change = (
    catboost_regularized_results["balanced_accuracy"]
    - catboost_tuned_results["balanced_accuracy"]
)

catboost_regularized_pr_auc_change = (
    catboost_regularized_results["pr_auc"]
    - catboost_tuned_results["pr_auc"]
)

# Check the established performance guardrails.
catboost_regularized_passes_guardrails = (
    catboost_regularized_results[
        "balanced_accuracy"
    ] >= 0.70
    and catboost_regularized_results[
        "specificity"
    ] >= 0.65
)

print("\nRegularized CatBoost evaluation completed.")

print(
    "Recall change versus tuned model:",
    round(
        catboost_regularized_recall_change,
        4,
    ),
)
print(
    "Specificity change versus tuned model:",
    round(
        catboost_regularized_specificity_change,
        4,
    ),
)
print(
    "Balanced-accuracy change versus tuned model:",
    round(
        catboost_regularized_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change versus tuned model:",
    round(
        catboost_regularized_pr_auc_change,
        4,
    ),
)
print(
    "Regularized model passes guardrails:",
    catboost_regularized_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized CatBoost...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                    
CatBoost Initial         0.2739             0.7208     0.2445  0.6376   
         Tuned           0.4913             0.7558     0.2433  0.7411   
         Regularized     0.4762             0.7505     0.2191  0.7766   

                      specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                               
CatBoost Initial           0.8039 0.3535 0.4825   0.7845  0.2545   
         Tuned             0.7705 0.3663 0.5259   0.8152  0.2998   
         Regularized       0.7244 0.3417 0.5146   0.8122  0.2964   

                      predicted_positive_cases  
model    model_stage                            
CatBoost Initial                           957  
         Tuned                            1118  
         Regularized                      1301


Regularized CatBoost evaluation completed.
Recall change versus tuned model: 0.0354
Specificity change versus tuned model: -0.0461
Balanced-accuracy change versus tuned model: -0.0053
PR-AUC change versus tuned model: -0.0034
Regularized model passes guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Ordinary accuracy included: False
Test set evaluated: False


In [115]:
# Step 57: Check the regularized CatBoost model for overfitting

print("Checking regularized CatBoost for overfitting...")

# Fit a separate copy on the complete training set.
# The test set remains untouched.
catboost_regularized_fitted_pipeline = clone(
    catboost_regularized_pipeline
)

catboost_regularized_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate complete-training-set probabilities.
catboost_regularized_training_probabilities = (
    catboost_regularized_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Use the threshold selected from out-of-fold predictions.
catboost_regularized_threshold = float(
    catboost_regularized_results["threshold"]
)

catboost_regularized_training_predictions = (
    catboost_regularized_training_probabilities
    >= catboost_regularized_threshold
).astype(int)

# Calculate complete-training-set metrics.
catboost_regularized_training_recall = recall_score(
    y_train,
    catboost_regularized_training_predictions,
)

catboost_regularized_training_pr_auc = (
    average_precision_score(
        y_train,
        catboost_regularized_training_probabilities,
    )
)

# Retrieve the out-of-fold metrics.
catboost_regularized_oof_recall = (
    catboost_regularized_results["recall"]
)

catboost_regularized_oof_pr_auc = (
    catboost_regularized_results["pr_auc"]
)

# Calculate train–OOF gaps.
catboost_regularized_recall_gap = (
    catboost_regularized_training_recall
    - catboost_regularized_oof_recall
)

catboost_regularized_pr_auc_gap = (
    catboost_regularized_training_pr_auc
    - catboost_regularized_oof_pr_auc
)

# Flag meaningful overfitting when either gap exceeds 0.10.
catboost_regularized_meaningful_overfitting = (
    catboost_regularized_recall_gap > 0.10
    or catboost_regularized_pr_auc_gap > 0.10
)

# Compare training and out-of-fold performance.
catboost_regularized_overfitting_table = pd.DataFrame(
    {
        "dataset": [
            "Complete training data",
            "Out-of-fold validation",
        ],
        "recall": [
            catboost_regularized_training_recall,
            catboost_regularized_oof_recall,
        ],
        "pr_auc": [
            catboost_regularized_training_pr_auc,
            catboost_regularized_oof_pr_auc,
        ],
    }
).set_index("dataset")

display(
    catboost_regularized_overfitting_table.round(4)
)

# Compare the tuned and regularized overfitting gaps.
catboost_gap_comparison = pd.DataFrame(
    {
        "configuration": [
            "Tuned",
            "Regularized",
        ],
        "recall_gap": [
            catboost_recall_gap,
            catboost_regularized_recall_gap,
        ],
        "pr_auc_gap": [
            catboost_pr_auc_gap,
            catboost_regularized_pr_auc_gap,
        ],
        "meaningful_overfitting": [
            catboost_tuned_meaningful_overfitting,
            catboost_regularized_meaningful_overfitting,
        ],
    }
).set_index("configuration")

display(
    catboost_gap_comparison.round(4)
)

print("\nRegularized CatBoost overfitting assessment completed.")
print(
    "Recall gap:",
    round(catboost_regularized_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(catboost_regularized_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting detected:",
    catboost_regularized_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training predictions")
print("Test set evaluated: False")

Checking regularized CatBoost for overfitting...


,recall,pr_auc
dataset,,
Complete training data,0.8202,0.3663
Out-of-fold validation,0.7766,0.2964


,recall_gap,pr_auc_gap,meaningful_overfitting
configuration,,,
Tuned,0.0845,0.1157,True
Regularized,0.0436,0.0699,False



Regularized CatBoost overfitting assessment completed.
Recall gap: 0.0436
PR-AUC gap: 0.0699
Meaningful overfitting detected: False
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training predictions
Test set evaluated: False


In [118]:
# Step 58: Evaluate regularized CatBoost with SMOTENC

print("Evaluating regularized CatBoost with SMOTENC...")

# Impute numeric predictors without scaling.
catboost_smote_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

# Prepare categorical predictors for SMOTENC.
catboost_smote_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Place numeric columns first and categorical columns second.
catboost_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            catboost_smote_numeric_transformer,
            numeric_columns,
        ),
        (
            "categorical",
            catboost_smote_categorical_transformer,
            categorical_columns,
        ),
    ],
    remainder="drop",
)

# Keep the transformed data as a NumPy array.
catboost_smote_preprocessor.set_output(
    transform="default"
)

# Define transformed feature positions.
catboost_smote_numeric_indices = list(
    range(len(numeric_columns))
)

catboost_smote_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns)
        + len(categorical_columns),
    )
)

# One-hot encode categorical predictors after SMOTENC.
catboost_post_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            catboost_smote_numeric_indices,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            catboost_smote_categorical_indices,
        ),
    ],
    remainder="drop",
)

catboost_post_smote_preprocessor.set_output(
    transform="default"
)

# Build a fresh classifier using the selected regularized
# hyperparameters. Do not apply class weights because SMOTENC
# balances the classes inside every cross-validation fold.
catboost_smote_classifier = CatBoostClassifier(
    loss_function="Logloss",
    iterations=300,
    depth=4,
    learning_rate=0.02,
    l2_leaf_reg=100,
    random_strength=15.0,
    bagging_temperature=5.0,
    rsm=0.70,
    border_count=64,
    random_seed=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False,
    thread_count=-1,
)

# Keep preprocessing and synthetic resampling inside
# cross-validation to prevent data leakage.
catboost_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            catboost_smote_preprocessor,
        ),
        (
            "smote",
            SMOTENC(
                categorical_features=(
                    catboost_smote_categorical_indices
                ),
                sampling_strategy="auto",
                random_state=RANDOM_STATE,
                k_neighbors=5,
            ),
        ),
        (
            "postprocessor",
            catboost_post_smote_preprocessor,
        ),
        (
            "classifier",
            catboost_smote_classifier,
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the threshold.
(
    catboost_smote_results,
    catboost_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="CatBoost",
    pipeline=catboost_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

catboost_smote_results[
    "model_stage"
] = "Regularized + SMOTENC"

# Compare SMOTENC against the regularized class-weighted model.
catboost_smote_comparison = pd.DataFrame(
    [
        catboost_regularized_results,
        catboost_smote_results,
    ]
)

display(
    catboost_smote_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes caused by SMOTENC.
catboost_smote_recall_change = (
    catboost_smote_results["recall"]
    - catboost_regularized_results["recall"]
)

catboost_smote_specificity_change = (
    catboost_smote_results["specificity"]
    - catboost_regularized_results["specificity"]
)

catboost_smote_balanced_accuracy_change = (
    catboost_smote_results["balanced_accuracy"]
    - catboost_regularized_results["balanced_accuracy"]
)

catboost_smote_pr_auc_change = (
    catboost_smote_results["pr_auc"]
    - catboost_regularized_results["pr_auc"]
)

# Apply the same guardrails.
catboost_smote_passes_guardrails = (
    catboost_smote_results["balanced_accuracy"] >= 0.70
    and catboost_smote_results["specificity"] >= 0.65
)

print("\nCatBoost SMOTENC evaluation completed.")
print(
    "Recall change:",
    round(catboost_smote_recall_change, 4),
)
print(
    "Specificity change:",
    round(catboost_smote_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(
        catboost_smote_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change:",
    round(catboost_smote_pr_auc_change, 4),
)
print(
    "SMOTENC passes guardrails:",
    catboost_smote_passes_guardrails,
)
print("Class weighting combined with SMOTENC: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized CatBoost with SMOTENC...


threshold  balanced_accuracy  precision  \
model    model_stage                                                      
CatBoost Regularized               0.4762             0.7505     0.2191   
         Regularized + SMOTENC     0.3488             0.7050     0.1842   

                                recall  specificity     f1     f2  roc_auc  \
model    model_stage                                                         
CatBoost Regularized            0.7766       0.7244 0.3417 0.5146   0.8122   
         Regularized + SMOTENC  0.7330       0.6770 0.2945 0.4594   0.7687   

                                pr_auc  predicted_positive_cases  
model    model_stage                                              
CatBoost Regularized            0.2964                      1301  
         Regularized + SMOTENC  0.2500                      1460


CatBoost SMOTENC evaluation completed.
Recall change: -0.0436
Specificity change: -0.0475
Balanced-accuracy change: -0.0455
PR-AUC change: -0.0465
SMOTENC passes guardrails: True
Class weighting combined with SMOTENC: False
Threshold selected using: Maximum out-of-fold F2
Ordinary accuracy included: False
Test set evaluated: False


In [119]:
# Step 59: Compare all CatBoost configurations and select the final model

print("Comparing all CatBoost configurations...")

# Combine all completed CatBoost experiments.
catboost_comparison = pd.DataFrame(
    [
        catboost_initial_results,
        catboost_tuned_results,
        catboost_regularized_results,
        catboost_smote_results,
    ]
).copy()

# Determine whether every configuration meets the established
# balanced-accuracy and specificity guardrails.
catboost_comparison["passes_guardrails"] = (
    (catboost_comparison["balanced_accuracy"] >= 0.70)
    & (catboost_comparison["specificity"] >= 0.65)
)

# Record the completed overfitting assessments.
catboost_comparison["overfitting_status"] = [
    "Not selected for final use",
    "Meaningful overfitting",
    "No meaningful overfitting",
    "Not selected for final use",
]

# Mark the regularized class-weighted model as the final choice.
catboost_comparison["selected_final_model"] = (
    catboost_comparison["model_stage"]
    == "Regularized"
)

catboost_final_display_columns = [
    "model",
    "model_stage",
    "threshold",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
    "passes_guardrails",
    "overfitting_status",
    "selected_final_model",
]

display(
    catboost_comparison[
        catboost_final_display_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Retain the regularized class-weighted CatBoost model.
catboost_final_pipeline = clone(
    catboost_regularized_pipeline
)

catboost_final_results = (
    catboost_regularized_results.copy()
)

catboost_final_oof_probabilities = (
    catboost_regularized_oof_probabilities.copy()
)

catboost_final_results[
    "model_stage"
] = "Final selected"

catboost_final_threshold = float(
    catboost_final_results["threshold"]
)

print("\nFinal CatBoost model selected successfully.")
print(
    "Selected configuration: "
    "Regularized class-weighted CatBoost"
)
print(
    "Selected threshold:",
    round(catboost_final_threshold, 4),
)
print(
    "Recall:",
    round(catboost_final_results["recall"], 4),
)
print(
    "Specificity:",
    round(catboost_final_results["specificity"], 4),
)
print(
    "Balanced accuracy:",
    round(
        catboost_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "PR-AUC:",
    round(catboost_final_results["pr_auc"], 4),
)
print(
    "Recall train–OOF gap:",
    round(catboost_regularized_recall_gap, 4),
)
print(
    "PR-AUC train–OOF gap:",
    round(catboost_regularized_pr_auc_gap, 4),
)
print("Meaningful overfitting detected: False")
print("SMOTENC selected: False")
print(
    "Reason SMOTENC was rejected: "
    "recall, specificity, balanced accuracy, and PR-AUC all decreased."
)
print("Primary final-comparison metric: Recall")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Comparing all CatBoost configurations...


threshold  balanced_accuracy  precision  \
model    model_stage                                                      
CatBoost Initial                   0.2739             0.7208     0.2445   
         Tuned                     0.4913             0.7558     0.2433   
         Regularized               0.4762             0.7505     0.2191   
         Regularized + SMOTENC     0.3488             0.7050     0.1842   

                                recall  specificity     f1     f2  roc_auc  \
model    model_stage                                                         
CatBoost Initial                0.6376       0.8039 0.3535 0.4825   0.7845   
         Tuned                  0.7411       0.7705 0.3663 0.5259   0.8152   
         Regularized            0.7766       0.7244 0.3417 0.5146   0.8122   
         Regularized + SMOTENC  0.7330       0.6770 0.2945 0.4594   0.7687   

                                pr_auc  passes_guardrails  \
model    model_stage                                        
CatBoost Initial                0.2545               True   
         Tuned                  0.2998               True   
         Regularized            0.2964               True   
         Regularized + SMOTENC  0.2500               True   

                                        overfitting_status  \
model    model_stage                                         
CatBoost Initial                Not selected for final use   
         Tuned                      Meaningful overfitting   
         Regularized             No meaningful overfitting   
         Regularized + SMOTENC  Not selected for final use   

                                selected_final_model  
model    model_stage                                  
CatBoost Initial                               False  
         Tuned                                 False  
         Regularized                            True  
         Regularized + SMOTENC                 False


Final CatBoost model selected successfully.
Selected configuration: Regularized class-weighted CatBoost
Selected threshold: 0.4762
Recall: 0.7766
Specificity: 0.7244
Balanced accuracy: 0.7505
PR-AUC: 0.2964
Recall train–OOF gap: 0.0436
PR-AUC train–OOF gap: 0.0699
Meaningful overfitting detected: False
SMOTENC selected: False
Reason SMOTENC was rejected: recall, specificity, balanced accuracy, and PR-AUC all decreased.
Primary final-comparison metric: Recall
Ordinary accuracy included: False
Test set evaluated: False


### CatBoost Conclusion

The **regularized class-weighted CatBoost** was selected as the final CatBoost model.

- **Recall:** 0.7766
- **ROC-AUC:** 0.8122
- **Specificity:** 0.7244
- **Balanced accuracy:** 0.7505
- **Selected threshold:** 0.4762

Regularization reduced the recall overfitting gap to **0.0436** and the PR-AUC gap to **0.0699**, indicating no meaningful overfitting. SMOTENC was rejected because it reduced recall, specificity, balanced accuracy, and PR-AUC.

Therefore, the regularized CatBoost model provides the best overall balance and will advance to the final model comparison. The test set remained untouched.

## LightGBM

LightGBM is a gradient-boosting algorithm that builds trees efficiently and can capture nonlinear relationships and interactions among predictors.

Because the target is imbalanced, the initial model uses balanced class weights. Its probability threshold is selected from out-of-fold training predictions using the maximum F2 score while enforcing the balanced-accuracy and specificity guardrails.

The test set remains untouched.

In [120]:
# Step 60: Build and evaluate the initial LightGBM model

print("Evaluating initial LightGBM...")

# Build the initial LightGBM pipeline.
# Balanced class weights address the imbalanced target.
lightgbm_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(preprocessor),
        ),
        (
            "classifier",
            LGBMClassifier(
                objective="binary",
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
        ),
    ]
)

# Generate out-of-fold probabilities and select the
# maximum-F2 threshold using only the training data.
(
    lightgbm_initial_results,
    lightgbm_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="LightGBM",
    pipeline=lightgbm_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

lightgbm_initial_results["model_stage"] = "Initial"

# Display the initial LightGBM results.
display(
    pd.DataFrame(
        [lightgbm_initial_results]
    )[comparison_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Check the established performance guardrails.
lightgbm_initial_passes_guardrails = (
    lightgbm_initial_results["balanced_accuracy"] >= 0.70
    and lightgbm_initial_results["specificity"] >= 0.65
)

print("\nInitial LightGBM evaluation completed.")
print("Imbalance handling: Balanced class weights")
print(
    "Passes performance guardrails:",
    lightgbm_initial_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Fine-tuning metric: PR-AUC")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating initial LightGBM...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
LightGBM,Initial,0.1143,0.7125,0.2030,0.6975,0.7274,0.3145,0.4690,0.7718,0.2324,1261



Initial LightGBM evaluation completed.
Imbalance handling: Balanced class weights
Passes performance guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Fine-tuning metric: PR-AUC
Ordinary accuracy included: False
Test set evaluated: False


In [121]:
# Step 61: Fine-tune LightGBM using stratified cross-validation PR-AUC

print("Fine-tuning LightGBM...")

# Define stratified folds inside this cell to preserve
# the minority-class proportion during every validation split.
lightgbm_cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# RandomizedSearchCV handles parallel processing, so each
# individual LightGBM fit uses one processing thread.
lightgbm_search_pipeline = clone(
    lightgbm_initial_pipeline
).set_params(
    classifier__n_jobs=1
)

# Search model-complexity, sampling, and regularization parameters.
lightgbm_parameter_distributions = {
    "classifier__n_estimators": [
        100,
        200,
        300,
        500,
        700,
    ],
    "classifier__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05,
        0.08,
        0.10,
    ],
    "classifier__num_leaves": [
        7,
        15,
        31,
        63,
    ],
    "classifier__max_depth": [
        3,
        4,
        5,
        6,
        8,
        -1,
    ],
    "classifier__min_child_samples": [
        10,
        20,
        30,
        50,
        80,
        120,
    ],
    "classifier__subsample": [
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    "classifier__colsample_bytree": [
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    "classifier__reg_alpha": [
        0.0,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
    ],
    "classifier__reg_lambda": [
        0.0,
        1.0,
        5.0,
        10.0,
        20.0,
        50.0,
    ],
}

# Select hyperparameters using validation PR-AUC.
lightgbm_search = RandomizedSearchCV(
    estimator=lightgbm_search_pipeline,
    param_distributions=lightgbm_parameter_distributions,
    n_iter=30,
    scoring="average_precision",
    cv=lightgbm_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fine-tune using only the training set.
lightgbm_search.fit(
    X_train,
    y_train,
)

# Retrieve the selected search result.
best_lightgbm_index = lightgbm_search.best_index_

lightgbm_search_training_pr_auc = (
    lightgbm_search.cv_results_[
        "mean_train_score"
    ][best_lightgbm_index]
)

lightgbm_search_validation_pr_auc = (
    lightgbm_search.cv_results_[
        "mean_test_score"
    ][best_lightgbm_index]
)

lightgbm_search_pr_auc_gap = (
    lightgbm_search_training_pr_auc
    - lightgbm_search_validation_pr_auc
)

print("\nLightGBM fine-tuning completed.")

print("\nBest parameters:")
display(lightgbm_search.best_params_)

print(
    "Training PR-AUC:",
    round(lightgbm_search_training_pr_auc, 4),
)
print(
    "Validation PR-AUC:",
    round(lightgbm_search_validation_pr_auc, 4),
)
print(
    "Preliminary PR-AUC gap:",
    round(lightgbm_search_pr_auc_gap, 4),
)

print("Optimization metric: PR-AUC")
print("Cross-validation: 5-fold stratified")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Fine-tuning LightGBM...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

LightGBM fine-tuning completed.

Best parameters:


{'classifier__subsample': 0.7,
 'classifier__reg_lambda': 10.0,
 'classifier__reg_alpha': 2.0,
 'classifier__num_leaves': 15,
 'classifier__n_estimators': 200,
 'classifier__min_child_samples': 30,
 'classifier__max_depth': 3,
 'classifier__learning_rate': 0.02,
 'classifier__colsample_bytree': 0.7}

Training PR-AUC: 0.4127
Validation PR-AUC: 0.2911
Preliminary PR-AUC gap: 0.1216
Optimization metric: PR-AUC
Cross-validation: 5-fold stratified
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [122]:
# Step 62: Evaluate the tuned LightGBM model

print("Evaluating tuned LightGBM...")

# Copy the best pipeline selected during fine-tuning.
lightgbm_tuned_pipeline = clone(
    lightgbm_search.best_estimator_
)

# Allow LightGBM to use available processing threads during
# the final out-of-fold evaluation.
lightgbm_tuned_pipeline.set_params(
    classifier__n_jobs=-1
)

# Generate out-of-fold probabilities and select the
# maximum-F2 threshold using only training data.
(
    lightgbm_tuned_results,
    lightgbm_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="LightGBM",
    pipeline=lightgbm_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

lightgbm_tuned_results["model_stage"] = "Tuned"

# Compare the initial and tuned LightGBM configurations.
lightgbm_initial_tuned_comparison = pd.DataFrame(
    [
        lightgbm_initial_results,
        lightgbm_tuned_results,
    ]
)

display(
    lightgbm_initial_tuned_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes produced by fine-tuning.
lightgbm_recall_change = (
    lightgbm_tuned_results["recall"]
    - lightgbm_initial_results["recall"]
)

lightgbm_specificity_change = (
    lightgbm_tuned_results["specificity"]
    - lightgbm_initial_results["specificity"]
)

lightgbm_balanced_accuracy_change = (
    lightgbm_tuned_results["balanced_accuracy"]
    - lightgbm_initial_results["balanced_accuracy"]
)

lightgbm_pr_auc_change = (
    lightgbm_tuned_results["pr_auc"]
    - lightgbm_initial_results["pr_auc"]
)

# Apply the established performance guardrails.
lightgbm_tuned_passes_guardrails = (
    lightgbm_tuned_results["balanced_accuracy"] >= 0.70
    and lightgbm_tuned_results["specificity"] >= 0.65
)

print("\nTuned LightGBM evaluation completed.")
print(
    "Recall change:",
    round(lightgbm_recall_change, 4),
)
print(
    "Specificity change:",
    round(lightgbm_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(
        lightgbm_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change:",
    round(lightgbm_pr_auc_change, 4),
)
print(
    "Tuned model passes guardrails:",
    lightgbm_tuned_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating tuned LightGBM...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                    
LightGBM Initial         0.1143             0.7125     0.2030  0.6975   
         Tuned           0.5120             0.7460     0.2442  0.7112   

                      specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                               
LightGBM Initial           0.7274 0.3145 0.4690   0.7718  0.2324   
         Tuned             0.7809 0.3635 0.5144   0.8087  0.2796   

                      predicted_positive_cases  
model    model_stage                            
LightGBM Initial                          1261  
         Tuned                            1069


Tuned LightGBM evaluation completed.
Recall change: 0.0136
Specificity change: 0.0534
Balanced-accuracy change: 0.0335
PR-AUC change: 0.0472
Tuned model passes guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Ordinary accuracy included: False
Test set evaluated: False


In [123]:
# Step 63: Check the tuned LightGBM model for overfitting

print("Checking tuned LightGBM for overfitting...")

# Fit a separate copy of the tuned pipeline on the complete
# training set. The test set remains untouched.
lightgbm_tuned_fitted_pipeline = clone(
    lightgbm_tuned_pipeline
)

lightgbm_tuned_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate probabilities for the complete training set.
lightgbm_training_probabilities = (
    lightgbm_tuned_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Use the threshold selected from out-of-fold predictions.
lightgbm_tuned_threshold = float(
    lightgbm_tuned_results["threshold"]
)

lightgbm_training_predictions = (
    lightgbm_training_probabilities
    >= lightgbm_tuned_threshold
).astype(int)

# Calculate complete-training-set metrics.
lightgbm_training_recall = recall_score(
    y_train,
    lightgbm_training_predictions,
)

lightgbm_training_pr_auc = average_precision_score(
    y_train,
    lightgbm_training_probabilities,
)

# Retrieve the unbiased out-of-fold metrics.
lightgbm_oof_recall = lightgbm_tuned_results["recall"]
lightgbm_oof_pr_auc = lightgbm_tuned_results["pr_auc"]

# Calculate train–OOF performance gaps.
lightgbm_recall_gap = (
    lightgbm_training_recall
    - lightgbm_oof_recall
)

lightgbm_pr_auc_gap = (
    lightgbm_training_pr_auc
    - lightgbm_oof_pr_auc
)

# Flag meaningful overfitting if either gap exceeds 0.10.
lightgbm_tuned_meaningful_overfitting = (
    lightgbm_recall_gap > 0.10
    or lightgbm_pr_auc_gap > 0.10
)

# Display the training and validation metrics.
lightgbm_overfitting_comparison = pd.DataFrame(
    {
        "dataset": [
            "Complete training data",
            "Out-of-fold validation",
        ],
        "recall": [
            lightgbm_training_recall,
            lightgbm_oof_recall,
        ],
        "pr_auc": [
            lightgbm_training_pr_auc,
            lightgbm_oof_pr_auc,
        ],
    }
).set_index("dataset")

display(
    lightgbm_overfitting_comparison.round(4)
)

print("\nTuned LightGBM overfitting assessment completed.")
print(
    "Recall gap:",
    round(lightgbm_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(lightgbm_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting detected:",
    lightgbm_tuned_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training predictions")
print("Test set evaluated: False")

Checking tuned LightGBM for overfitting...


,recall,pr_auc
dataset,,
Complete training data,0.8065,0.3942
Out-of-fold validation,0.7112,0.2796



Tuned LightGBM overfitting assessment completed.
Recall gap: 0.0954
PR-AUC gap: 0.1146
Meaningful overfitting detected: True
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training predictions
Test set evaluated: False


In [124]:
# Step 64: Search for a more strongly regularized LightGBM model

print("Searching for a more strongly regularized LightGBM model...")

# Start from the initial class-weighted LightGBM pipeline.
# RandomizedSearchCV performs the parallel processing.
lightgbm_regularization_pipeline = clone(
    lightgbm_initial_pipeline
).set_params(
    classifier__n_jobs=1
)

# Restrict the search to simpler trees and stronger
# regularization to reduce the observed overfitting.
lightgbm_regularization_parameters = {
    "classifier__n_estimators": [
        100,
        150,
        200,
        250,
        300,
    ],
    "classifier__learning_rate": [
        0.01,
        0.015,
        0.02,
        0.03,
    ],
    "classifier__num_leaves": [
        5,
        7,
        10,
        15,
    ],
    "classifier__max_depth": [
        2,
        3,
        4,
    ],
    "classifier__min_child_samples": [
        40,
        60,
        80,
        100,
        120,
        150,
    ],
    "classifier__min_split_gain": [
        0.0,
        0.1,
        0.3,
        0.5,
        1.0,
    ],
    "classifier__subsample": [
        0.60,
        0.70,
        0.80,
        0.90,
    ],
    # This activates row sampling in LightGBM.
    "classifier__subsample_freq": [
        1,
        3,
        5,
    ],
    "classifier__colsample_bytree": [
        0.50,
        0.60,
        0.70,
        0.80,
    ],
    "classifier__reg_alpha": [
        2.0,
        5.0,
        10.0,
        15.0,
        20.0,
    ],
    "classifier__reg_lambda": [
        10.0,
        20.0,
        40.0,
        60.0,
        80.0,
        100.0,
    ],
    "classifier__max_bin": [
        31,
        63,
        127,
    ],
}

# Select a regularized configuration using validation PR-AUC.
lightgbm_regularization_search = RandomizedSearchCV(
    estimator=lightgbm_regularization_pipeline,
    param_distributions=lightgbm_regularization_parameters,
    n_iter=30,
    scoring="average_precision",
    cv=lightgbm_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Use only the training set.
lightgbm_regularization_search.fit(
    X_train,
    y_train,
)

# Locate the selected parameter combination.
best_lightgbm_regularization_index = (
    lightgbm_regularization_search.best_index_
)

# Extract training and validation PR-AUC.
lightgbm_regularization_training_pr_auc = (
    lightgbm_regularization_search.cv_results_[
        "mean_train_score"
    ][best_lightgbm_regularization_index]
)

lightgbm_regularization_validation_pr_auc = (
    lightgbm_regularization_search.cv_results_[
        "mean_test_score"
    ][best_lightgbm_regularization_index]
)

lightgbm_regularization_search_gap = (
    lightgbm_regularization_training_pr_auc
    - lightgbm_regularization_validation_pr_auc
)

print("\nRegularized LightGBM search completed.")

print("\nBest regularized parameters:")
display(
    lightgbm_regularization_search.best_params_
)

print(
    "Training PR-AUC:",
    round(
        lightgbm_regularization_training_pr_auc,
        4,
    ),
)
print(
    "Validation PR-AUC:",
    round(
        lightgbm_regularization_validation_pr_auc,
        4,
    ),
)
print(
    "Preliminary PR-AUC gap:",
    round(
        lightgbm_regularization_search_gap,
        4,
    ),
)

print("Optimization metric: PR-AUC")
print("Purpose: Reduce tuned LightGBM overfitting")
print("Row subsampling activated: True")
print("Cross-validation: 5-fold stratified")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Searching for a more strongly regularized LightGBM model...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Regularized LightGBM search completed.

Best regularized parameters:


{'classifier__subsample_freq': 1,
 'classifier__subsample': 0.6,
 'classifier__reg_lambda': 20.0,
 'classifier__reg_alpha': 2.0,
 'classifier__num_leaves': 7,
 'classifier__n_estimators': 200,
 'classifier__min_split_gain': 0.5,
 'classifier__min_child_samples': 60,
 'classifier__max_depth': 3,
 'classifier__max_bin': 127,
 'classifier__learning_rate': 0.03,
 'classifier__colsample_bytree': 0.5}

Training PR-AUC: 0.3962
Validation PR-AUC: 0.2983
Preliminary PR-AUC gap: 0.098
Optimization metric: PR-AUC
Purpose: Reduce tuned LightGBM overfitting
Row subsampling activated: True
Cross-validation: 5-fold stratified
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [125]:
# Step 65: Evaluate the regularized LightGBM model

print("Evaluating regularized LightGBM...")

# Copy the best pipeline selected by the focused
# regularization search.
lightgbm_regularized_pipeline = clone(
    lightgbm_regularization_search.best_estimator_
)

# Allow LightGBM to use available processing threads
# during out-of-fold evaluation.
lightgbm_regularized_pipeline.set_params(
    classifier__n_jobs=-1
)

# Generate out-of-fold probabilities and select the
# maximum-F2 threshold using only the training data.
(
    lightgbm_regularized_results,
    lightgbm_regularized_oof_probabilities,
) = create_oof_evaluation(
    model_name="LightGBM",
    pipeline=lightgbm_regularized_pipeline,
    X_training=X_train,
    y_training=y_train,
)

lightgbm_regularized_results[
    "model_stage"
] = "Regularized"

# Compare all completed LightGBM configurations.
lightgbm_regularized_comparison = pd.DataFrame(
    [
        lightgbm_initial_results,
        lightgbm_tuned_results,
        lightgbm_regularized_results,
    ]
)

display(
    lightgbm_regularized_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes relative to the tuned model.
lightgbm_regularized_recall_change = (
    lightgbm_regularized_results["recall"]
    - lightgbm_tuned_results["recall"]
)

lightgbm_regularized_specificity_change = (
    lightgbm_regularized_results["specificity"]
    - lightgbm_tuned_results["specificity"]
)

lightgbm_regularized_balanced_accuracy_change = (
    lightgbm_regularized_results["balanced_accuracy"]
    - lightgbm_tuned_results["balanced_accuracy"]
)

lightgbm_regularized_pr_auc_change = (
    lightgbm_regularized_results["pr_auc"]
    - lightgbm_tuned_results["pr_auc"]
)

# Apply the established performance guardrails.
lightgbm_regularized_passes_guardrails = (
    lightgbm_regularized_results[
        "balanced_accuracy"
    ] >= 0.70
    and lightgbm_regularized_results[
        "specificity"
    ] >= 0.65
)

print("\nRegularized LightGBM evaluation completed.")
print(
    "Recall change versus tuned model:",
    round(
        lightgbm_regularized_recall_change,
        4,
    ),
)
print(
    "Specificity change versus tuned model:",
    round(
        lightgbm_regularized_specificity_change,
        4,
    ),
)
print(
    "Balanced-accuracy change versus tuned model:",
    round(
        lightgbm_regularized_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change versus tuned model:",
    round(
        lightgbm_regularized_pr_auc_change,
        4,
    ),
)
print(
    "Regularized model passes guardrails:",
    lightgbm_regularized_passes_guardrails,
)
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary final-comparison metric: Recall")
print("Row subsampling active: True")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized LightGBM...


threshold  balanced_accuracy  precision  recall  \
model    model_stage                                                    
LightGBM Initial         0.1143             0.7125     0.2030  0.6975   
         Tuned           0.5120             0.7460     0.2442  0.7112   
         Regularized     0.5058             0.7511     0.2417  0.7302   

                      specificity     f1     f2  roc_auc  pr_auc  \
model    model_stage                                               
LightGBM Initial           0.7274 0.3145 0.4690   0.7718  0.2324   
         Tuned             0.7809 0.3635 0.5144   0.8087  0.2796   
         Regularized       0.7719 0.3631 0.5200   0.8101  0.2897   

                      predicted_positive_cases  
model    model_stage                            
LightGBM Initial                          1261  
         Tuned                            1069  
         Regularized                      1109


Regularized LightGBM evaluation completed.
Recall change versus tuned model: 0.0191
Specificity change versus tuned model: -0.009
Balanced-accuracy change versus tuned model: 0.0051
PR-AUC change versus tuned model: 0.0101
Regularized model passes guardrails: True
Threshold selected using: Maximum out-of-fold F2
Primary final-comparison metric: Recall
Row subsampling active: True
Ordinary accuracy included: False
Test set evaluated: False


In [126]:
# Step 66: Check the regularized LightGBM model for overfitting

print("Checking regularized LightGBM for overfitting...")

# Fit a separate copy on the complete training set.
# The test set remains untouched.
lightgbm_regularized_fitted_pipeline = clone(
    lightgbm_regularized_pipeline
)

lightgbm_regularized_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate complete-training-set probabilities.
lightgbm_regularized_training_probabilities = (
    lightgbm_regularized_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Use the threshold selected from out-of-fold predictions.
lightgbm_regularized_threshold = float(
    lightgbm_regularized_results["threshold"]
)

lightgbm_regularized_training_predictions = (
    lightgbm_regularized_training_probabilities
    >= lightgbm_regularized_threshold
).astype(int)

# Calculate complete-training-set metrics.
lightgbm_regularized_training_recall = recall_score(
    y_train,
    lightgbm_regularized_training_predictions,
)

lightgbm_regularized_training_pr_auc = (
    average_precision_score(
        y_train,
        lightgbm_regularized_training_probabilities,
    )
)

# Retrieve the out-of-fold metrics.
lightgbm_regularized_oof_recall = (
    lightgbm_regularized_results["recall"]
)

lightgbm_regularized_oof_pr_auc = (
    lightgbm_regularized_results["pr_auc"]
)

# Calculate train–OOF performance gaps.
lightgbm_regularized_recall_gap = (
    lightgbm_regularized_training_recall
    - lightgbm_regularized_oof_recall
)

lightgbm_regularized_pr_auc_gap = (
    lightgbm_regularized_training_pr_auc
    - lightgbm_regularized_oof_pr_auc
)

# Flag meaningful overfitting if either gap exceeds 0.10.
lightgbm_regularized_meaningful_overfitting = (
    lightgbm_regularized_recall_gap > 0.10
    or lightgbm_regularized_pr_auc_gap > 0.10
)

# Display complete-training and out-of-fold performance.
lightgbm_regularized_overfitting_table = pd.DataFrame(
    {
        "dataset": [
            "Complete training data",
            "Out-of-fold validation",
        ],
        "recall": [
            lightgbm_regularized_training_recall,
            lightgbm_regularized_oof_recall,
        ],
        "pr_auc": [
            lightgbm_regularized_training_pr_auc,
            lightgbm_regularized_oof_pr_auc,
        ],
    }
).set_index("dataset")

display(
    lightgbm_regularized_overfitting_table.round(4)
)

# Compare the tuned and regularized overfitting gaps.
lightgbm_gap_comparison = pd.DataFrame(
    {
        "configuration": [
            "Tuned",
            "Regularized",
        ],
        "recall_gap": [
            lightgbm_recall_gap,
            lightgbm_regularized_recall_gap,
        ],
        "pr_auc_gap": [
            lightgbm_pr_auc_gap,
            lightgbm_regularized_pr_auc_gap,
        ],
        "meaningful_overfitting": [
            lightgbm_tuned_meaningful_overfitting,
            lightgbm_regularized_meaningful_overfitting,
        ],
    }
).set_index("configuration")

display(
    lightgbm_gap_comparison.round(4)
)

print("\nRegularized LightGBM overfitting assessment completed.")
print(
    "Recall gap:",
    round(lightgbm_regularized_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(lightgbm_regularized_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting detected:",
    lightgbm_regularized_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training predictions")
print("Test set evaluated: False")

Checking regularized LightGBM for overfitting...


,recall,pr_auc
dataset,,
Complete training data,0.8038,0.3873
Out-of-fold validation,0.7302,0.2897


,recall_gap,pr_auc_gap,meaningful_overfitting
configuration,,,
Tuned,0.0954,0.1146,True
Regularized,0.0736,0.0976,False



Regularized LightGBM overfitting assessment completed.
Recall gap: 0.0736
PR-AUC gap: 0.0976
Meaningful overfitting detected: False
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training predictions
Test set evaluated: False


In [127]:
# Step 67: Evaluate regularized LightGBM with SMOTENC

print("Evaluating regularized LightGBM with SMOTENC...")

# Impute numeric predictors without scaling because
# LightGBM is a tree-based model.
lightgbm_smote_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

# Prepare categorical predictors for SMOTENC.
lightgbm_smote_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Place numeric predictors first and categorical predictors second.
lightgbm_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            lightgbm_smote_numeric_transformer,
            numeric_columns,
        ),
        (
            "categorical",
            lightgbm_smote_categorical_transformer,
            categorical_columns,
        ),
    ],
    remainder="drop",
)

# Force NumPy output so SMOTENC can use positional indices.
lightgbm_smote_preprocessor.set_output(
    transform="default"
)

# Define transformed feature positions.
lightgbm_smote_numeric_indices = list(
    range(len(numeric_columns))
)

lightgbm_smote_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns)
        + len(categorical_columns),
    )
)

# One-hot encode the categorical predictors after resampling.
lightgbm_post_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            lightgbm_smote_numeric_indices,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            lightgbm_smote_categorical_indices,
        ),
    ],
    remainder="drop",
)

lightgbm_post_smote_preprocessor.set_output(
    transform="default"
)

# Copy the selected regularized classifier.
lightgbm_smote_classifier = clone(
    lightgbm_regularized_pipeline.named_steps[
        "classifier"
    ]
)

# Disable class weighting because SMOTENC balances the
# minority class inside every cross-validation training fold.
lightgbm_smote_classifier.set_params(
    class_weight=None,
    n_jobs=-1,
)

# Apply preprocessing and SMOTENC inside cross-validation.
lightgbm_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            lightgbm_smote_preprocessor,
        ),
        (
            "smote",
            SMOTENC(
                categorical_features=(
                    lightgbm_smote_categorical_indices
                ),
                sampling_strategy="auto",
                random_state=RANDOM_STATE,
                k_neighbors=5,
            ),
        ),
        (
            "postprocessor",
            lightgbm_post_smote_preprocessor,
        ),
        (
            "classifier",
            lightgbm_smote_classifier,
        ),
    ]
)

# Generate out-of-fold probabilities and optimize the threshold.
(
    lightgbm_smote_results,
    lightgbm_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="LightGBM",
    pipeline=lightgbm_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

lightgbm_smote_results[
    "model_stage"
] = "Regularized + SMOTENC"

# Compare SMOTENC against the regularized class-weighted model.
lightgbm_smote_comparison = pd.DataFrame(
    [
        lightgbm_regularized_results,
        lightgbm_smote_results,
    ]
)

display(
    lightgbm_smote_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes caused by SMOTENC.
lightgbm_smote_recall_change = (
    lightgbm_smote_results["recall"]
    - lightgbm_regularized_results["recall"]
)

lightgbm_smote_specificity_change = (
    lightgbm_smote_results["specificity"]
    - lightgbm_regularized_results["specificity"]
)

lightgbm_smote_balanced_accuracy_change = (
    lightgbm_smote_results["balanced_accuracy"]
    - lightgbm_regularized_results["balanced_accuracy"]
)

lightgbm_smote_pr_auc_change = (
    lightgbm_smote_results["pr_auc"]
    - lightgbm_regularized_results["pr_auc"]
)

# Apply the established performance guardrails.
lightgbm_smote_passes_guardrails = (
    lightgbm_smote_results["balanced_accuracy"] >= 0.70
    and lightgbm_smote_results["specificity"] >= 0.65
)

print("\nLightGBM SMOTENC evaluation completed.")
print(
    "Recall change:",
    round(lightgbm_smote_recall_change, 4),
)
print(
    "Specificity change:",
    round(lightgbm_smote_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(
        lightgbm_smote_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change:",
    round(lightgbm_smote_pr_auc_change, 4),
)
print(
    "SMOTENC passes guardrails:",
    lightgbm_smote_passes_guardrails,
)
print("Class weighting combined with SMOTENC: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized LightGBM with SMOTENC...


threshold  balanced_accuracy  precision  \
model    model_stage                                                      
LightGBM Regularized               0.5058             0.7511     0.2417   
         Regularized + SMOTENC     0.3032             0.7103     0.2024   

                                recall  specificity     f1     f2  roc_auc  \
model    model_stage                                                         
LightGBM Regularized            0.7302       0.7719 0.3631 0.5200   0.8101   
         Regularized + SMOTENC  0.6921       0.7285 0.3132 0.4664   0.7670   

                                pr_auc  predicted_positive_cases  
model    model_stage                                              
LightGBM Regularized            0.2897                      1109  
         Regularized + SMOTENC  0.2442                      1255


LightGBM SMOTENC evaluation completed.
Recall change: -0.0381
Specificity change: -0.0434
Balanced-accuracy change: -0.0408
PR-AUC change: -0.0455
SMOTENC passes guardrails: True
Class weighting combined with SMOTENC: False
Threshold selected using: Maximum out-of-fold F2
Ordinary accuracy included: False
Test set evaluated: False


In [128]:
# Step 68: Compare all LightGBM stages and select the final version

print("Comparing all LightGBM model stages...")

# Combine the results produced during the LightGBM experiments.
lightgbm_comparison = pd.DataFrame(
    [
        lightgbm_initial_results,
        lightgbm_tuned_results,
        lightgbm_regularized_results,
        lightgbm_smote_results,
    ]
).copy()

# Give every experiment a clear name.
lightgbm_comparison["model_stage"] = [
    "Initial",
    "Tuned",
    "Regularized",
    "Regularized + SMOTENC",
]

# Apply the same minimum clinical-performance guardrails
# used throughout the notebook.
lightgbm_comparison["passes_guardrails"] = (
    (lightgbm_comparison["balanced_accuracy"] >= 0.70)
    & (lightgbm_comparison["specificity"] >= 0.65)
)

# Record the overfitting conclusion for each relevant stage.
lightgbm_comparison["overfitting_status"] = [
    "Not selected for detailed check",
    "Meaningful overfitting",
    "No meaningful overfitting",
    "Not retained because performance decreased",
]

# Display the full LightGBM comparison.
lightgbm_display_columns = [
    "model",
    "model_stage",
    "threshold",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "f2",
    "roc_auc",
    "pr_auc",
    "passes_guardrails",
    "overfitting_status",
]

display(
    lightgbm_comparison[lightgbm_display_columns]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Select the regularized class-weighted LightGBM.
#
# Reasons:
# 1. It produced the highest recall among the acceptable final candidates.
# 2. It passed both performance guardrails.
# 3. Its exact train-versus-OOF check found no meaningful overfitting.
# 4. SMOTENC reduced recall, balanced accuracy, F2, and PR-AUC.
lightgbm_final_pipeline = clone(lightgbm_regularized_pipeline)

lightgbm_final_results = lightgbm_regularized_results.copy()
lightgbm_final_results["model_stage"] = "Final selected"

lightgbm_final_oof_probabilities = (
    lightgbm_regularized_oof_probabilities.copy()
)

lightgbm_final_threshold = float(
    lightgbm_final_results["threshold"]
)

# Store the final LightGBM overfitting results for the
# final all-model comparison.
lightgbm_final_training_recall = (
    lightgbm_regularized_training_recall
)

lightgbm_final_oof_recall = (
    lightgbm_regularized_oof_recall
)

lightgbm_final_recall_gap = (
    lightgbm_regularized_recall_gap
)

lightgbm_final_training_pr_auc = (
    lightgbm_regularized_training_pr_auc
)

lightgbm_final_oof_pr_auc = (
    lightgbm_regularized_oof_pr_auc
)

lightgbm_final_pr_auc_gap = (
    lightgbm_regularized_pr_auc_gap
)

lightgbm_final_meaningful_overfitting = (
    lightgbm_regularized_meaningful_overfitting
)

# Display the selected LightGBM result.
print("\nFinal LightGBM selected:")
display(
    pd.DataFrame([lightgbm_final_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Selected version: Regularized class-weighted LightGBM")
print(
    "Selected threshold:",
    round(lightgbm_final_threshold, 4),
)
print(
    "Recall:",
    round(lightgbm_final_results["recall"], 4),
)
print(
    "Specificity:",
    round(lightgbm_final_results["specificity"], 4),
)
print(
    "Balanced accuracy:",
    round(
        lightgbm_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "PR-AUC:",
    round(lightgbm_final_results["pr_auc"], 4),
)
print(
    "Meaningful overfitting:",
    lightgbm_final_meaningful_overfitting,
)
print("SMOTENC retained: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Comparing all LightGBM model stages...


threshold  balanced_accuracy  precision  \
model    model_stage                                                      
LightGBM Initial                   0.1143             0.7125     0.2030   
         Tuned                     0.5120             0.7460     0.2442   
         Regularized               0.5058             0.7511     0.2417   
         Regularized + SMOTENC     0.3032             0.7103     0.2024   

                                recall  specificity     f1     f2  roc_auc  \
model    model_stage                                                         
LightGBM Initial                0.6975       0.7274 0.3145 0.4690   0.7718   
         Tuned                  0.7112       0.7809 0.3635 0.5144   0.8087   
         Regularized            0.7302       0.7719 0.3631 0.5200   0.8101   
         Regularized + SMOTENC  0.6921       0.7285 0.3132 0.4664   0.7670   

                                pr_auc  passes_guardrails  \
model    model_stage                                        
LightGBM Initial                0.2324               True   
         Tuned                  0.2796               True   
         Regularized            0.2897               True   
         Regularized + SMOTENC  0.2442               True   

                                                        overfitting_status  
model    model_stage                                                        
LightGBM Initial                           Not selected for detailed check  
         Tuned                                      Meaningful overfitting  
         Regularized                             No meaningful overfitting  
         Regularized + SMOTENC  Not retained because performance decreased


Final LightGBM selected:


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
LightGBM,Final selected,0.5058,0.7511,0.2417,0.7302,0.7719,0.3631,0.5200,0.8101,0.2897,1109


Selected version: Regularized class-weighted LightGBM
Selected threshold: 0.5058
Recall: 0.7302
Specificity: 0.7719
Balanced accuracy: 0.7511
PR-AUC: 0.2897
Meaningful overfitting: False
SMOTENC retained: False
Ordinary accuracy used: False
Test set evaluated: False


### LightGBM Conclusion

The **regularized class-weighted LightGBM** was selected as the final LightGBM model.

- **Recall:** 0.7302
- **ROC-AUC:** 0.8101
- **Specificity:** 0.7719
- **Balanced accuracy:** 0.7511
- **Selected threshold:** 0.5058

The tuned model showed meaningful overfitting, with a PR-AUC gap of **0.1146**. Stronger regularization reduced the recall gap to **0.0736** and the PR-AUC gap to **0.0976**, indicating no meaningful overfitting. It also improved recall, ROC-AUC, and balanced accuracy compared with the tuned model.

SMOTENC was rejected because it reduced recall, specificity, balanced accuracy, ROC-AUC, and PR-AUC. Therefore, the regularized class-weighted LightGBM provides the best overall balance and will advance to the final model comparison. The test set remained untouched.

## Histogram Gradient Boosting

Histogram Gradient Boosting groups continuous values into bins before building decision trees. This can improve training efficiency and reduce sensitivity to small variations in the data.

The model is evaluated using stratified five-fold cross-validation. Class-balanced weights address the target imbalance, and the classification threshold is selected from out-of-fold training probabilities by maximizing the F2 score.

The primary tuning metric is PR-AUC. Recall is the main final comparison metric, subject to minimum balanced-accuracy and specificity requirements. The test set remains untouched.

In [129]:
# Step 69: Build and evaluate the initial Histogram Gradient Boosting model

print("Evaluating initial Histogram Gradient Boosting...")

# Histogram Gradient Boosting requires dense input.
# Create a dedicated preprocessor that returns a dense matrix.
histogram_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

histogram_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

histogram_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            histogram_numeric_pipeline,
            numeric_columns,
        ),
        (
            "categorical",
            histogram_categorical_pipeline,
            categorical_columns,
        ),
    ],
    remainder="drop",
)

# Use balanced class weights because elevated-risk cases
# represent only a small portion of the training data.
histogram_initial_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            histogram_preprocessor,
        ),
        (
            "classifier",
            HistGradientBoostingClassifier(
                learning_rate=0.10,
                max_iter=100,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=0.0,
                class_weight="balanced",
                early_stopping=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# Generate out-of-fold probabilities using only the training set.
# The threshold is selected by maximizing the F2 score.
(
    histogram_initial_results,
    histogram_initial_oof_probabilities,
) = create_oof_evaluation(
    model_name="Histogram Gradient Boosting",
    pipeline=histogram_initial_pipeline,
    X_training=X_train,
    y_training=y_train,
)

histogram_initial_results["model_stage"] = "Initial"

# Check whether the initial model meets the minimum guardrails.
histogram_initial_passes_guardrails = (
    histogram_initial_results["balanced_accuracy"] >= 0.70
    and histogram_initial_results["specificity"] >= 0.65
)

# Display the initial performance.
display(
    pd.DataFrame([histogram_initial_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

print("Initial Histogram Gradient Boosting evaluation completed.")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Primary tuning metric: PR-AUC")
print("Final comparison metric: Recall")
print(
    "Passes performance guardrails:",
    histogram_initial_passes_guardrails,
)
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating initial Histogram Gradient Boosting...


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Histogram Gradient Boosting,Initial,0.1163,0.7229,0.2128,0.7057,0.7402,0.3270,0.4823,0.7648,0.2313,1217


Initial Histogram Gradient Boosting evaluation completed.
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Primary tuning metric: PR-AUC
Final comparison metric: Recall
Passes performance guardrails: True
Ordinary accuracy included: False
Test set evaluated: False


In [131]:
# Step 70: Fine-tune Histogram Gradient Boosting using cross-validation PR-AUC

print("Fine-tuning Histogram Gradient Boosting...")

# Define stratified five-fold cross-validation locally.
# Stratification preserves the minority-class percentage in every fold.
histogram_cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Search model-complexity and regularization parameters.
histogram_parameter_distributions = {
    "classifier__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05,
        0.08,
        0.10,
    ],
    "classifier__max_iter": [
        100,
        150,
        200,
        300,
        400,
    ],
    "classifier__max_leaf_nodes": [
        7,
        15,
        31,
        63,
    ],
    "classifier__max_depth": [
        2,
        3,
        4,
        5,
        None,
    ],
    "classifier__min_samples_leaf": [
        10,
        20,
        30,
        40,
        60,
        80,
    ],
    "classifier__l2_regularization": [
        0.0,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
        20.0,
    ],
    "classifier__max_bins": [
        63,
        127,
        255,
    ],
}

# Randomly evaluate 40 parameter combinations.
# PR-AUC is used because the target is imbalanced.
histogram_search = RandomizedSearchCV(
    estimator=clone(histogram_initial_pipeline),
    param_distributions=histogram_parameter_distributions,
    n_iter=40,
    scoring="average_precision",
    cv=histogram_cv_strategy,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fit using only the training data.
# The test set remains untouched.
histogram_search.fit(X_train, y_train)

# Locate the best parameter combination.
best_histogram_index = histogram_search.best_index_

# Extract training and validation PR-AUC for a preliminary
# assessment of possible overfitting.
histogram_search_training_pr_auc = (
    histogram_search.cv_results_["mean_train_score"][
        best_histogram_index
    ]
)

histogram_search_validation_pr_auc = (
    histogram_search.cv_results_["mean_test_score"][
        best_histogram_index
    ]
)

histogram_search_pr_auc_gap = (
    histogram_search_training_pr_auc
    - histogram_search_validation_pr_auc
)

print("\nHistogram Gradient Boosting fine-tuning completed.")

print("Best parameters:")
display(histogram_search.best_params_)

print(
    "Best cross-validation PR-AUC:",
    round(histogram_search.best_score_, 4),
)
print(
    "Training PR-AUC:",
    round(histogram_search_training_pr_auc, 4),
)
print(
    "Validation PR-AUC:",
    round(histogram_search_validation_pr_auc, 4),
)
print(
    "Preliminary overfitting gap:",
    round(histogram_search_pr_auc_gap, 4),
)

print("Optimization metric: PR-AUC")
print("Cross-validation: Stratified 5-fold")
print("Imbalance handling: Balanced class weights")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Fine-tuning Histogram Gradient Boosting...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Histogram Gradient Boosting fine-tuning completed.
Best parameters:


{'classifier__min_samples_leaf': 80,
 'classifier__max_leaf_nodes': 15,
 'classifier__max_iter': 300,
 'classifier__max_depth': 3,
 'classifier__max_bins': 63,
 'classifier__learning_rate': 0.02,
 'classifier__l2_regularization': 10.0}

Best cross-validation PR-AUC: 0.293
Training PR-AUC: 0.4337
Validation PR-AUC: 0.293
Preliminary overfitting gap: 0.1406
Optimization metric: PR-AUC
Cross-validation: Stratified 5-fold
Imbalance handling: Balanced class weights
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [132]:
# Step 71: Evaluate the tuned Histogram Gradient Boosting model

print("Evaluating tuned Histogram Gradient Boosting...")

# Copy the best pipeline selected by RandomizedSearchCV.
histogram_tuned_pipeline = clone(
    histogram_search.best_estimator_
)

# Generate new out-of-fold probabilities using the selected
# hyperparameters and optimize the threshold by maximum F2.
(
    histogram_tuned_results,
    histogram_tuned_oof_probabilities,
) = create_oof_evaluation(
    model_name="Histogram Gradient Boosting",
    pipeline=histogram_tuned_pipeline,
    X_training=X_train,
    y_training=y_train,
)

histogram_tuned_results["model_stage"] = "Tuned"

# Check whether the tuned model satisfies the required
# balanced-accuracy and specificity guardrails.
histogram_tuned_passes_guardrails = (
    histogram_tuned_results["balanced_accuracy"] >= 0.70
    and histogram_tuned_results["specificity"] >= 0.65
)

# Compare the initial and tuned versions.
histogram_initial_vs_tuned = pd.DataFrame(
    [
        histogram_initial_results,
        histogram_tuned_results,
    ]
)

display(
    histogram_initial_vs_tuned[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate the changes produced by fine-tuning.
histogram_recall_change = (
    histogram_tuned_results["recall"]
    - histogram_initial_results["recall"]
)

histogram_specificity_change = (
    histogram_tuned_results["specificity"]
    - histogram_initial_results["specificity"]
)

histogram_balanced_accuracy_change = (
    histogram_tuned_results["balanced_accuracy"]
    - histogram_initial_results["balanced_accuracy"]
)

histogram_pr_auc_change = (
    histogram_tuned_results["pr_auc"]
    - histogram_initial_results["pr_auc"]
)

histogram_f2_change = (
    histogram_tuned_results["f2"]
    - histogram_initial_results["f2"]
)

print("\nTuned Histogram Gradient Boosting evaluation completed.")

print(
    "Selected threshold:",
    round(histogram_tuned_results["threshold"], 4),
)
print(
    "Recall change:",
    round(histogram_recall_change, 4),
)
print(
    "Specificity change:",
    round(histogram_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(histogram_balanced_accuracy_change, 4),
)
print(
    "PR-AUC change:",
    round(histogram_pr_auc_change, 4),
)
print(
    "F2 change:",
    round(histogram_f2_change, 4),
)

print(
    "Passes performance guardrails:",
    histogram_tuned_passes_guardrails,
)
print(
    "Preliminary search PR-AUC gap:",
    round(histogram_search_pr_auc_gap, 4),
)
print("Exact overfitting check completed: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating tuned Histogram Gradient Boosting...


threshold  balanced_accuracy  \
model                       model_stage                                 
Histogram Gradient Boosting Initial         0.1163             0.7229   
                            Tuned           0.5131             0.7418   

                                         precision  recall  specificity  \
model                       model_stage                                   
Histogram Gradient Boosting Initial         0.2128  0.7057       0.7402   
                            Tuned           0.2434  0.7003       0.7833   

                                            f1     f2  roc_auc  pr_auc  \
model                       model_stage                                  
Histogram Gradient Boosting Initial     0.3270 0.4823   0.7648  0.2313   
                            Tuned       0.3612 0.5091   0.8040  0.2838   

                                         predicted_positive_cases  
model                       model_stage                            
Histogram Gradient Boosting Initial                          1217  
                            Tuned                            1056


Tuned Histogram Gradient Boosting evaluation completed.
Selected threshold: 0.5131
Recall change: -0.0054
Specificity change: 0.0431
Balanced-accuracy change: 0.0188
PR-AUC change: 0.0525
F2 change: 0.0268
Passes performance guardrails: True
Preliminary search PR-AUC gap: 0.1406
Exact overfitting check completed: False
Threshold selected using: Maximum out-of-fold F2
Ordinary accuracy included: False
Test set evaluated: False


In [133]:
# Step 72: Check the tuned Histogram Gradient Boosting model for overfitting

print("Checking tuned Histogram Gradient Boosting for overfitting...")

# Fit a copy of the tuned model on the complete training set.
# This fitted copy is used only to measure training performance.
histogram_tuned_fitted_pipeline = clone(
    histogram_tuned_pipeline
)

histogram_tuned_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate probabilities for the same training observations.
histogram_tuned_training_probabilities = (
    histogram_tuned_fitted_pipeline.predict_proba(X_train)[:, 1]
)

# Use the threshold selected exclusively from out-of-fold
# training probabilities.
histogram_tuned_threshold = float(
    histogram_tuned_results["threshold"]
)

histogram_tuned_training_predictions = (
    histogram_tuned_training_probabilities
    >= histogram_tuned_threshold
).astype(int)

# Calculate training recall and PR-AUC.
histogram_tuned_training_recall = recall_score(
    y_train,
    histogram_tuned_training_predictions,
)

histogram_tuned_training_pr_auc = average_precision_score(
    y_train,
    histogram_tuned_training_probabilities,
)

# Retrieve the corresponding out-of-fold metrics.
histogram_tuned_oof_recall = float(
    histogram_tuned_results["recall"]
)

histogram_tuned_oof_pr_auc = float(
    histogram_tuned_results["pr_auc"]
)

# Calculate train-versus-OOF performance gaps.
histogram_tuned_recall_gap = (
    histogram_tuned_training_recall
    - histogram_tuned_oof_recall
)

histogram_tuned_pr_auc_gap = (
    histogram_tuned_training_pr_auc
    - histogram_tuned_oof_pr_auc
)

# A gap greater than 0.10 in either measure is treated
# as evidence of meaningful overfitting.
histogram_tuned_meaningful_overfitting = (
    histogram_tuned_recall_gap > 0.10
    or histogram_tuned_pr_auc_gap > 0.10
)

# Display the exact overfitting comparison.
histogram_overfitting_table = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "PR-AUC",
        ],
        "training_score": [
            histogram_tuned_training_recall,
            histogram_tuned_training_pr_auc,
        ],
        "oof_score": [
            histogram_tuned_oof_recall,
            histogram_tuned_oof_pr_auc,
        ],
        "gap": [
            histogram_tuned_recall_gap,
            histogram_tuned_pr_auc_gap,
        ],
    }
).set_index("metric")

display(histogram_overfitting_table.round(4))

print("Exact overfitting check completed.")
print(
    "Recall gap:",
    round(histogram_tuned_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(histogram_tuned_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting:",
    histogram_tuned_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training probabilities")
print("Test set evaluated: False")

Checking tuned Histogram Gradient Boosting for overfitting...


,training_score,oof_score,gap
metric,,,
Recall,0.8202,0.7003,0.1199
PR-AUC,0.4167,0.2838,0.1330


Exact overfitting check completed.
Recall gap: 0.1199
PR-AUC gap: 0.133
Meaningful overfitting: True
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training probabilities
Test set evaluated: False


In [134]:
# Step 73: Perform focused regularization for Histogram Gradient Boosting

print(
    "Running focused Histogram Gradient Boosting "
    "regularization search..."
)

# Define the same stratified five-fold cross-validation.
histogram_regularized_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# The first search selected:
# - max_depth = 3
# - max_leaf_nodes = 15
# - min_samples_leaf = 80
# - l2_regularization = 10
#
# Because that model overfit, this focused search explores
# simpler trees, larger leaves, and stronger L2 penalties.
histogram_regularized_parameters = {
    "classifier__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05,
    ],
    "classifier__max_iter": [
        150,
        200,
        300,
        400,
    ],
    "classifier__max_leaf_nodes": [
        3,
        5,
        7,
        10,
        15,
    ],
    "classifier__max_depth": [
        2,
        3,
    ],
    "classifier__min_samples_leaf": [
        80,
        100,
        120,
        150,
        200,
    ],
    "classifier__l2_regularization": [
        10.0,
        20.0,
        30.0,
        50.0,
        100.0,
    ],
    "classifier__max_bins": [
        31,
        63,
        127,
    ],
}

# Optimize PR-AUC while restricting the search to more
# strongly regularized model configurations.
histogram_regularized_search = RandomizedSearchCV(
    estimator=clone(histogram_initial_pipeline),
    param_distributions=histogram_regularized_parameters,
    n_iter=40,
    scoring="average_precision",
    cv=histogram_regularized_cv,
    refit=True,
    return_train_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

# Fit using only the training data.
# The test set remains untouched.
histogram_regularized_search.fit(
    X_train,
    y_train,
)

# Locate the selected parameter combination.
best_histogram_regularized_index = (
    histogram_regularized_search.best_index_
)

# Extract its training and validation PR-AUC.
histogram_regularized_search_training_pr_auc = (
    histogram_regularized_search.cv_results_[
        "mean_train_score"
    ][best_histogram_regularized_index]
)

histogram_regularized_search_validation_pr_auc = (
    histogram_regularized_search.cv_results_[
        "mean_test_score"
    ][best_histogram_regularized_index]
)

histogram_regularized_search_pr_auc_gap = (
    histogram_regularized_search_training_pr_auc
    - histogram_regularized_search_validation_pr_auc
)

print(
    "\nFocused Histogram Gradient Boosting "
    "regularization search completed."
)

print("Best regularized parameters:")
display(histogram_regularized_search.best_params_)

print(
    "Best cross-validation PR-AUC:",
    round(
        histogram_regularized_search.best_score_,
        4,
    ),
)
print(
    "Training PR-AUC:",
    round(
        histogram_regularized_search_training_pr_auc,
        4,
    ),
)
print(
    "Validation PR-AUC:",
    round(
        histogram_regularized_search_validation_pr_auc,
        4,
    ),
)
print(
    "Preliminary overfitting gap:",
    round(
        histogram_regularized_search_pr_auc_gap,
        4,
    ),
)

print("Optimization metric: PR-AUC")
print("Regularization strengthened: True")
print("Imbalance handling: Balanced class weights")
print("Threshold optimization performed during search: False")
print("Ordinary accuracy used: False")
print("Test set evaluated: False")

Running focused Histogram Gradient Boosting regularization search...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Focused Histogram Gradient Boosting regularization search completed.
Best regularized parameters:


{'classifier__min_samples_leaf': 150,
 'classifier__max_leaf_nodes': 5,
 'classifier__max_iter': 400,
 'classifier__max_depth': 3,
 'classifier__max_bins': 63,
 'classifier__learning_rate': 0.02,
 'classifier__l2_regularization': 100.0}

Best cross-validation PR-AUC: 0.3024
Training PR-AUC: 0.3694
Validation PR-AUC: 0.3024
Preliminary overfitting gap: 0.0669
Optimization metric: PR-AUC
Regularization strengthened: True
Imbalance handling: Balanced class weights
Threshold optimization performed during search: False
Ordinary accuracy used: False
Test set evaluated: False


In [135]:
# Step 74: Evaluate the regularized Histogram Gradient Boosting model

print(
    "Evaluating regularized Histogram Gradient Boosting..."
)

# Copy the best model found during the focused
# regularization search.
histogram_regularized_pipeline = clone(
    histogram_regularized_search.best_estimator_
)

# Generate out-of-fold probabilities and select the
# probability threshold that maximizes the F2 score.
(
    histogram_regularized_results,
    histogram_regularized_oof_probabilities,
) = create_oof_evaluation(
    model_name="Histogram Gradient Boosting",
    pipeline=histogram_regularized_pipeline,
    X_training=X_train,
    y_training=y_train,
)

histogram_regularized_results[
    "model_stage"
] = "Regularized"

# Check the minimum performance requirements.
histogram_regularized_passes_guardrails = (
    histogram_regularized_results[
        "balanced_accuracy"
    ] >= 0.70
    and histogram_regularized_results[
        "specificity"
    ] >= 0.65
)

# Compare all Histogram Gradient Boosting stages completed
# so far using exactly the same OOF framework.
histogram_stage_comparison = pd.DataFrame(
    [
        histogram_initial_results,
        histogram_tuned_results,
        histogram_regularized_results,
    ]
)

display(
    histogram_stage_comparison[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes between the tuned and regularized models.
histogram_regularized_recall_change = (
    histogram_regularized_results["recall"]
    - histogram_tuned_results["recall"]
)

histogram_regularized_specificity_change = (
    histogram_regularized_results["specificity"]
    - histogram_tuned_results["specificity"]
)

histogram_regularized_balanced_accuracy_change = (
    histogram_regularized_results["balanced_accuracy"]
    - histogram_tuned_results["balanced_accuracy"]
)

histogram_regularized_pr_auc_change = (
    histogram_regularized_results["pr_auc"]
    - histogram_tuned_results["pr_auc"]
)

histogram_regularized_f2_change = (
    histogram_regularized_results["f2"]
    - histogram_tuned_results["f2"]
)

print(
    "\nRegularized Histogram Gradient Boosting "
    "evaluation completed."
)

print(
    "Selected threshold:",
    round(
        histogram_regularized_results["threshold"],
        4,
    ),
)
print(
    "Recall change versus tuned:",
    round(
        histogram_regularized_recall_change,
        4,
    ),
)
print(
    "Specificity change versus tuned:",
    round(
        histogram_regularized_specificity_change,
        4,
    ),
)
print(
    "Balanced-accuracy change versus tuned:",
    round(
        histogram_regularized_balanced_accuracy_change,
        4,
    ),
)
print(
    "PR-AUC change versus tuned:",
    round(
        histogram_regularized_pr_auc_change,
        4,
    ),
)
print(
    "F2 change versus tuned:",
    round(
        histogram_regularized_f2_change,
        4,
    ),
)

print(
    "Passes performance guardrails:",
    histogram_regularized_passes_guardrails,
)
print(
    "Preliminary regularized PR-AUC gap:",
    round(
        histogram_regularized_search_pr_auc_gap,
        4,
    ),
)
print("Exact overfitting check completed: False")
print("Imbalance handling: Balanced class weights")
print("Threshold selected using: Maximum out-of-fold F2")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized Histogram Gradient Boosting...


threshold  balanced_accuracy  \
model                       model_stage                                 
Histogram Gradient Boosting Initial         0.1163             0.7229   
                            Tuned           0.5131             0.7418   
                            Regularized     0.5269             0.7532   

                                         precision  recall  specificity  \
model                       model_stage                                   
Histogram Gradient Boosting Initial         0.2128  0.7057       0.7402   
                            Tuned           0.2434  0.7003       0.7833   
                            Regularized     0.2437  0.7330       0.7735   

                                            f1     f2  roc_auc  pr_auc  \
model                       model_stage                                  
Histogram Gradient Boosting Initial     0.3270 0.4823   0.7648  0.2313   
                            Tuned       0.3612 0.5091   0.8040  0.2838   
                            Regularized 0.3657 0.5229   0.8092  0.2914   

                                         predicted_positive_cases  
model                       model_stage                            
Histogram Gradient Boosting Initial                          1217  
                            Tuned                            1056  
                            Regularized                      1104


Regularized Histogram Gradient Boosting evaluation completed.
Selected threshold: 0.5269
Recall change versus tuned: 0.0327
Specificity change versus tuned: -0.0098
Balanced-accuracy change versus tuned: 0.0115
PR-AUC change versus tuned: 0.0076
F2 change versus tuned: 0.0138
Passes performance guardrails: True
Preliminary regularized PR-AUC gap: 0.0669
Exact overfitting check completed: False
Imbalance handling: Balanced class weights
Threshold selected using: Maximum out-of-fold F2
Ordinary accuracy included: False
Test set evaluated: False


In [136]:
# Step 75: Check the regularized Histogram Gradient Boosting model for overfitting

print(
    "Checking regularized Histogram Gradient Boosting "
    "for overfitting..."
)

# Fit a copy of the regularized model on the complete
# training dataset.
histogram_regularized_fitted_pipeline = clone(
    histogram_regularized_pipeline
)

histogram_regularized_fitted_pipeline.fit(
    X_train,
    y_train,
)

# Generate training probabilities.
histogram_regularized_training_probabilities = (
    histogram_regularized_fitted_pipeline.predict_proba(
        X_train
    )[:, 1]
)

# Use the threshold selected from out-of-fold probabilities.
histogram_regularized_threshold = float(
    histogram_regularized_results["threshold"]
)

histogram_regularized_training_predictions = (
    histogram_regularized_training_probabilities
    >= histogram_regularized_threshold
).astype(int)

# Calculate training recall and PR-AUC.
histogram_regularized_training_recall = recall_score(
    y_train,
    histogram_regularized_training_predictions,
)

histogram_regularized_training_pr_auc = (
    average_precision_score(
        y_train,
        histogram_regularized_training_probabilities,
    )
)

# Retrieve the out-of-fold scores.
histogram_regularized_oof_recall = float(
    histogram_regularized_results["recall"]
)

histogram_regularized_oof_pr_auc = float(
    histogram_regularized_results["pr_auc"]
)

# Calculate train-versus-OOF gaps.
histogram_regularized_recall_gap = (
    histogram_regularized_training_recall
    - histogram_regularized_oof_recall
)

histogram_regularized_pr_auc_gap = (
    histogram_regularized_training_pr_auc
    - histogram_regularized_oof_pr_auc
)

# Meaningful overfitting is defined as a gap above 0.10
# in either recall or PR-AUC.
histogram_regularized_meaningful_overfitting = (
    histogram_regularized_recall_gap > 0.10
    or histogram_regularized_pr_auc_gap > 0.10
)

# Display the exact comparison.
histogram_regularized_overfitting_table = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "PR-AUC",
        ],
        "training_score": [
            histogram_regularized_training_recall,
            histogram_regularized_training_pr_auc,
        ],
        "oof_score": [
            histogram_regularized_oof_recall,
            histogram_regularized_oof_pr_auc,
        ],
        "gap": [
            histogram_regularized_recall_gap,
            histogram_regularized_pr_auc_gap,
        ],
    }
).set_index("metric")

display(
    histogram_regularized_overfitting_table.round(4)
)

print(
    "Exact regularized-model overfitting check completed."
)
print(
    "Recall gap:",
    round(histogram_regularized_recall_gap, 4),
)
print(
    "PR-AUC gap:",
    round(histogram_regularized_pr_auc_gap, 4),
)
print(
    "Meaningful overfitting:",
    histogram_regularized_meaningful_overfitting,
)
print("Meaningful-gap threshold: 0.10")
print("Threshold source: Out-of-fold training probabilities")
print("Test set evaluated: False")

Checking regularized Histogram Gradient Boosting for overfitting...


,training_score,oof_score,gap
metric,,,
Recall,0.7875,0.7330,0.0545
PR-AUC,0.3617,0.2914,0.0704


Exact regularized-model overfitting check completed.
Recall gap: 0.0545
PR-AUC gap: 0.0704
Meaningful overfitting: False
Meaningful-gap threshold: 0.10
Threshold source: Out-of-fold training probabilities
Test set evaluated: False


In [137]:
# Step 76: Evaluate regularized Histogram Gradient Boosting with SMOTENC

print(
    "Evaluating regularized Histogram Gradient Boosting "
    "with SMOTENC..."
)

# Before SMOTENC:
# - Impute numeric predictors with their medians.
# - Impute and ordinal-encode categorical predictors.
#
# SMOTENC needs categorical predictors represented as
# single encoded columns rather than one-hot columns.
histogram_smote_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

histogram_smote_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

histogram_smote_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            histogram_smote_numeric_pipeline,
            numeric_columns,
        ),
        (
            "categorical",
            histogram_smote_categorical_pipeline,
            categorical_columns,
        ),
    ],
    remainder="drop",
)

# Numeric predictors appear first in the transformed matrix.
# The remaining positions represent categorical predictors.
histogram_smote_categorical_indices = list(
    range(
        len(numeric_columns),
        len(numeric_columns) + len(categorical_columns),
    )
)

# After SMOTENC, one-hot encode categorical predictors
# so the model receives a dense numerical matrix.
histogram_smote_postprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            list(range(len(numeric_columns))),
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            histogram_smote_categorical_indices,
        ),
    ],
    remainder="drop",
)

# Copy the regularized classifier and disable class weighting.
# The class imbalance is handled by SMOTENC in this experiment.
histogram_smote_classifier = clone(
    histogram_regularized_pipeline.named_steps[
        "classifier"
    ]
)

histogram_smote_classifier.set_params(
    class_weight=None,
)

# SMOTENC is placed inside the imbalanced-learn pipeline.
# It will therefore be applied only to each training fold,
# preventing synthetic observations from leaking into validation.
histogram_smote_pipeline = ImbPipeline(
    steps=[
        (
            "preprocessor",
            histogram_smote_preprocessor,
        ),
        (
            "smote",
            SMOTENC(
                categorical_features=(
                    histogram_smote_categorical_indices
                ),
                sampling_strategy=0.50,
                k_neighbors=5,
                random_state=RANDOM_STATE,
            ),
        ),
        (
            "postprocessor",
            histogram_smote_postprocessor,
        ),
        (
            "classifier",
            histogram_smote_classifier,
        ),
    ]
)

# Evaluate with the same out-of-fold framework and
# maximum-F2 threshold selection.
(
    histogram_smote_results,
    histogram_smote_oof_probabilities,
) = create_oof_evaluation(
    model_name="Histogram Gradient Boosting",
    pipeline=histogram_smote_pipeline,
    X_training=X_train,
    y_training=y_train,
)

histogram_smote_results[
    "model_stage"
] = "Regularized + SMOTENC"

# Apply the same minimum performance guardrails.
histogram_smote_passes_guardrails = (
    histogram_smote_results["balanced_accuracy"] >= 0.70
    and histogram_smote_results["specificity"] >= 0.65
)

# Compare class weighting against SMOTENC.
histogram_regularized_vs_smote = pd.DataFrame(
    [
        histogram_regularized_results,
        histogram_smote_results,
    ]
)

display(
    histogram_regularized_vs_smote[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Calculate changes caused by SMOTENC.
histogram_smote_recall_change = (
    histogram_smote_results["recall"]
    - histogram_regularized_results["recall"]
)

histogram_smote_specificity_change = (
    histogram_smote_results["specificity"]
    - histogram_regularized_results["specificity"]
)

histogram_smote_balanced_accuracy_change = (
    histogram_smote_results["balanced_accuracy"]
    - histogram_regularized_results["balanced_accuracy"]
)

histogram_smote_pr_auc_change = (
    histogram_smote_results["pr_auc"]
    - histogram_regularized_results["pr_auc"]
)

histogram_smote_f2_change = (
    histogram_smote_results["f2"]
    - histogram_regularized_results["f2"]
)

print("\nHistogram Gradient Boosting SMOTENC evaluation completed.")

print(
    "Recall change:",
    round(histogram_smote_recall_change, 4),
)
print(
    "Specificity change:",
    round(histogram_smote_specificity_change, 4),
)
print(
    "Balanced-accuracy change:",
    round(histogram_smote_balanced_accuracy_change, 4),
)
print(
    "PR-AUC change:",
    round(histogram_smote_pr_auc_change, 4),
)
print(
    "F2 change:",
    round(histogram_smote_f2_change, 4),
)

print(
    "SMOTENC passes guardrails:",
    histogram_smote_passes_guardrails,
)
print("Class weighting combined with SMOTENC: False")
print("Threshold selected using: Maximum out-of-fold F2")
print("Ordinary accuracy included: False")
print("Test set evaluated: False")

Evaluating regularized Histogram Gradient Boosting with SMOTENC...


threshold  \
model                       model_stage                        
Histogram Gradient Boosting Regularized               0.5269   
                            Regularized + SMOTENC     0.1805   

                                                   balanced_accuracy  \
model                       model_stage                                
Histogram Gradient Boosting Regularized                       0.7532   
                            Regularized + SMOTENC             0.7048   

                                                   precision  recall  \
model                       model_stage                                
Histogram Gradient Boosting Regularized               0.2437  0.7330   
                            Regularized + SMOTENC     0.1729  0.7820   

                                                   specificity     f1     f2  \
model                       model_stage                                        
Histogram Gradient Boosting Regularized                 0.7735 0.3657 0.5229   
                            Regularized + SMOTENC       0.6276 0.2832 0.4588   

                                                   roc_auc  pr_auc  \
model                       model_stage                              
Histogram Gradient Boosting Regularized             0.8092  0.2914   
                            Regularized + SMOTENC   0.7704  0.2459   

                                                   predicted_positive_cases  
model                       model_stage                                      
Histogram Gradient Boosting Regularized                                1104  
                            Regularized + SMOTENC                      1660


Histogram Gradient Boosting SMOTENC evaluation completed.
Recall change: 0.049
Specificity change: -0.1459
Balanced-accuracy change: -0.0484
PR-AUC change: -0.0454
F2 change: -0.0642
SMOTENC passes guardrails: False
Class weighting combined with SMOTENC: False
Threshold selected using: Maximum out-of-fold F2
Ordinary accuracy included: False
Test set evaluated: False


In [138]:
# Step 77: Select the final Histogram Gradient Boosting model

print("Selecting final Histogram Gradient Boosting model...")

# Compare every completed version.
histogram_final_comparison = pd.DataFrame(
    [
        histogram_initial_results,
        histogram_tuned_results,
        histogram_regularized_results,
        histogram_smote_results,
    ]
).copy()

# Apply the required performance guardrails.
histogram_final_comparison["passes_guardrails"] = (
    (
        histogram_final_comparison["balanced_accuracy"]
        >= 0.70
    )
    & (
        histogram_final_comparison["specificity"]
        >= 0.65
    )
)

histogram_final_comparison["overfitting_status"] = [
    "Not selected",
    "Meaningful overfitting",
    "No meaningful overfitting",
    "Specificity guardrail failed",
]

# Display the complete comparison.
display(
    histogram_final_comparison[
        [
            "model",
            "model_stage",
            "threshold",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "f2",
            "roc_auc",
            "pr_auc",
            "passes_guardrails",
            "overfitting_status",
        ]
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

# Retain the regularized class-weighted model.
# SMOTENC is rejected because specificity fell below 0.65.
histogram_final_pipeline = clone(
    histogram_regularized_pipeline
)

histogram_final_results = (
    histogram_regularized_results.copy()
)

histogram_final_results["model_stage"] = "Final selected"

histogram_final_oof_probabilities = (
    histogram_regularized_oof_probabilities.copy()
)

histogram_final_threshold = float(
    histogram_final_results["threshold"]
)

# Store the final overfitting results.
histogram_final_training_recall = (
    histogram_regularized_training_recall
)

histogram_final_oof_recall = (
    histogram_regularized_oof_recall
)

histogram_final_recall_gap = (
    histogram_regularized_recall_gap
)

histogram_final_training_pr_auc = (
    histogram_regularized_training_pr_auc
)

histogram_final_oof_pr_auc = (
    histogram_regularized_oof_pr_auc
)

histogram_final_pr_auc_gap = (
    histogram_regularized_pr_auc_gap
)

histogram_final_meaningful_overfitting = (
    histogram_regularized_meaningful_overfitting
)

print("\nFinal Histogram Gradient Boosting selected:")

display(
    pd.DataFrame([histogram_final_results])[
        comparison_columns
    ]
    .set_index(["model", "model_stage"])
    .round(4)
)

print(
    "Selected version: Regularized class-weighted "
    "Histogram Gradient Boosting"
)
print(
    "Threshold:",
    round(histogram_final_threshold, 4),
)
print(
    "Recall:",
    round(histogram_final_results["recall"], 4),
)
print(
    "Specificity:",
    round(histogram_final_results["specificity"], 4),
)
print(
    "Balanced accuracy:",
    round(
        histogram_final_results["balanced_accuracy"],
        4,
    ),
)
print(
    "PR-AUC:",
    round(histogram_final_results["pr_auc"], 4),
)
print(
    "Meaningful overfitting:",
    histogram_final_meaningful_overfitting,
)
print("SMOTENC retained: False")
print("Reason: Specificity was below 0.65")
print("Test set evaluated: False")

Selecting final Histogram Gradient Boosting model...


threshold  \
model                       model_stage                        
Histogram Gradient Boosting Initial                   0.1163   
                            Tuned                     0.5131   
                            Regularized               0.5269   
                            Regularized + SMOTENC     0.1805   

                                                   balanced_accuracy  \
model                       model_stage                                
Histogram Gradient Boosting Initial                           0.7229   
                            Tuned                             0.7418   
                            Regularized                       0.7532   
                            Regularized + SMOTENC             0.7048   

                                                   precision  recall  \
model                       model_stage                                
Histogram Gradient Boosting Initial                   0.2128  0.7057   
                            Tuned                     0.2434  0.7003   
                            Regularized               0.2437  0.7330   
                            Regularized + SMOTENC     0.1729  0.7820   

                                                   specificity     f1     f2  \
model                       model_stage                                        
Histogram Gradient Boosting Initial                     0.7402 0.3270 0.4823   
                            Tuned                       0.7833 0.3612 0.5091   
                            Regularized                 0.7735 0.3657 0.5229   
                            Regularized + SMOTENC       0.6276 0.2832 0.4588   

                                                   roc_auc  pr_auc  \
model                       model_stage                              
Histogram Gradient Boosting Initial                 0.7648  0.2313   
                            Tuned                   0.8040  0.2838   
                            Regularized             0.8092  0.2914   
                            Regularized + SMOTENC   0.7704  0.2459   

                                                   passes_guardrails  \
model                       model_stage                                
Histogram Gradient Boosting Initial                             True   
                            Tuned                               True   
                            Regularized                         True   
                            Regularized + SMOTENC              False   

                                                             overfitting_status  
model                       model_stage                                          
Histogram Gradient Boosting Initial                                Not selected  
                            Tuned                        Meaningful overfitting  
                            Regularized               No meaningful overfitting  
                            Regularized + SMOTENC  Specificity guardrail failed


Final Histogram Gradient Boosting selected:


,,threshold,balanced_accuracy,precision,recall,specificity,f1,f2,roc_auc,pr_auc,predicted_positive_cases
model,model_stage,,,,,,,,,,
Histogram Gradient Boosting,Final selected,0.5269,0.7532,0.2437,0.7330,0.7735,0.3657,0.5229,0.8092,0.2914,1104


Selected version: Regularized class-weighted Histogram Gradient Boosting
Threshold: 0.5269
Recall: 0.733
Specificity: 0.7735
Balanced accuracy: 0.7532
PR-AUC: 0.2914
Meaningful overfitting: False
SMOTENC retained: False
Reason: Specificity was below 0.65
Test set evaluated: False


### Histogram Gradient Boosting Conclusion

The **regularized class-weighted Histogram Gradient Boosting model** was selected as the final configuration.

- **Recall:** 0.7330
- **ROC-AUC:** 0.8092
- **Specificity:** 0.7735
- **Balanced accuracy:** 0.7532
- **Selected threshold:** 0.5269

The tuned model showed meaningful overfitting, with recall and PR-AUC gaps of **0.1199** and **0.1330**. Stronger regularization reduced these gaps to **0.0545** and **0.0704**, indicating no meaningful overfitting. It also improved recall, balanced accuracy, ROC-AUC, and PR-AUC compared with the tuned model.

SMOTENC increased recall to **0.7820**, but specificity decreased to **0.6276**, below the required **0.65** guardrail. It also reduced balanced accuracy, ROC-AUC, PR-AUC, and F2. Therefore, SMOTENC was rejected.

The regularized class-weighted model provides the best overall balance and will advance to the final model comparison. The test set remained untouched.

In [152]:
# Compare the final selected versions of all 10 models

print("Comparing the final selected models...")

# Store only the final selected configuration for each model.
# The Dummy Classifier is intentionally excluded.
final_models = [
    {
        "model": "Logistic Regression",
        "configuration": "Tuned + class weight",
        "results": logistic_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "Support Vector Machine",
        "configuration": "Tuned + class weight",
        "results": svm_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "Decision Tree",
        "configuration": "Tuned + class weight",
        "results": decision_tree_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "Random Forest",
        "configuration": "Tuned + class weight",
        "results": random_forest_final_results,
        "overfitting_status": "Possible overfitting",
    },
    {
        "model": "AdaBoost",
        "configuration": "Tuned + class weight",
        "results": adaboost_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "Gradient Boosting",
        "configuration": "Tuned + regularization",
        "results": gradient_boosting_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "XGBoost",
        "configuration": "Regularized + class weight",
        "results": xgboost_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "CatBoost",
        "configuration": "Regularized + class weight",
        "results": catboost_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "LightGBM",
        "configuration": "Regularized + class weight",
        "results": lightgbm_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
    {
        "model": "Histogram Gradient Boosting",
        "configuration": "Regularized + class weight",
        "results": histogram_final_results,
        "overfitting_status": "No meaningful overfitting",
    },
]

# Build one consistent comparison row for each model.
comparison_rows = []

for model_information in final_models:
    result = model_information["results"]

    comparison_rows.append(
        {
            "model": model_information["model"],
            "configuration": model_information["configuration"],
            "threshold": result["threshold"],
            "recall": result["recall"],
            "roc_auc": result["roc_auc"],
            "specificity": result["specificity"],
            "balanced_accuracy": result["balanced_accuracy"],
            "precision": result["precision"],
            "f1": result["f1"],
            "f2": result["f2"],
            "pr_auc": result["pr_auc"],
            "passes_guardrails": (
                result["balanced_accuracy"] >= 0.70
                and result["specificity"] >= 0.65
            ),
            "overfitting_status": model_information[
                "overfitting_status"
            ],
        }
    )

# Rank primarily by recall and secondarily by ROC-AUC.
final_model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        by=["recall", "roc_auc"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

final_model_comparison.insert(
    0,
    "rank",
    range(1, len(final_model_comparison) + 1),
)

display(
    final_model_comparison.set_index("rank").round(4)
)

print("\nFinal model comparison completed.")
print("Number of models compared:", len(final_model_comparison))
print("Primary ranking metric: Recall")
print("Secondary ranking metric: ROC-AUC")
print("Ordinary accuracy included: False")
print("Dummy Classifier included: False")
print("Test set evaluated: False")

Comparing the final selected models...


,model,configuration,threshold,recall,roc_auc,specificity,balanced_accuracy,precision,f1,f2,pr_auc,passes_guardrails,overfitting_status
rank,,,,,,,,,,,,,
1,Logistic Regression,Tuned + class weight,0.4125,0.8038,0.8077,0.6922,0.7480,0.2063,0.3283,0.5090,0.2955,True,No meaningful overfitting
2,AdaBoost,Tuned + class weight,0.3504,0.7902,0.7834,0.6794,0.7348,0.1970,0.3154,0.4932,0.2273,True,No meaningful overfitting
3,CatBoost,Regularized + class weight,0.4762,0.7766,0.8122,0.7244,0.7505,0.2191,0.3417,0.5146,0.2964,True,No meaningful overfitting
4,XGBoost,Regularized + class weight,0.4624,0.7711,0.8093,0.7282,0.7497,0.2202,0.3426,0.5140,0.2935,True,No meaningful overfitting
5,Random Forest,Tuned + class weight,0.4642,0.7520,0.8096,0.7274,0.7397,0.2155,0.3350,0.5020,0.2956,True,Possible overfitting
6,Histogram Gradient Boosting,Regularized + class weight,0.5269,0.7330,0.8092,0.7735,0.7532,0.2437,0.3657,0.5229,0.2914,True,No meaningful overfitting
7,LightGBM,Regularized + class weight,0.5058,0.7302,0.8101,0.7719,0.7511,0.2417,0.3631,0.5200,0.2897,True,No meaningful overfitting
8,Decision Tree,Tuned + class weight,0.3378,0.7275,0.7678,0.6838,0.7056,0.1863,0.2967,0.4602,0.2456,True,No meaningful overfitting
9,Support Vector Machine,Tuned + class weight,0.1030,0.7166,0.8070,0.7695,0.7430,0.2363,0.3554,0.5095,0.2916,True,No meaningful overfitting



Final model comparison completed.
Number of models compared: 10
Primary ranking metric: Recall
Secondary ranking metric: ROC-AUC
Ordinary accuracy included: False
Dummy Classifier included: False
Test set evaluated: False


### Final Cross-Validation Model Comparison

The final selected configurations of all ten models were compared using the same stratified cross-validation and out-of-fold threshold-optimization framework. Ordinary accuracy and the Dummy Classifier were excluded because the dataset is imbalanced.

**Logistic Regression** achieved the highest recall of **0.8038**, with a strong ROC-AUC of **0.8077**, specificity of **0.6922**, and balanced accuracy of **0.7480**. It passed all performance guardrails and showed no meaningful overfitting.

Although CatBoost produced the highest ROC-AUC of **0.8122**, its recall was lower at **0.7766**. Because recall is the primary selection metric for this screening application, Logistic Regression was selected as the final candidate model.

The model will now be evaluated once on the untouched test set to estimate its performance on unseen participants.

In [153]:
# Find the available Logistic Regression model and pipeline variables

print("Available Logistic Regression model variables:\n")

for variable_name in list(globals()):
    if (
        "logistic" in variable_name.lower()
        and any(
            word in variable_name.lower()
            for word in ["pipeline", "model", "search"]
        )
    ):
        print(variable_name)

Available Logistic Regression model variables:

initial_logistic_pipeline
logistic_tuning_pipeline
logistic_search
tuned_logistic_pipeline
logistic_initial_pipeline
logistic_smote_pipeline
logistic_reduced_pipeline
build_logistic_feature_pipeline
logistic_feature_selection_pipeline
smote_logistic_pipeline
build_logistic_pipeline_for_columns
selected_logistic_pipeline
logistic_pipeline_names
logistic_final_pipeline


In [154]:
# Final evaluation of the selected Logistic Regression model
# on the untouched test set

print("Evaluating the final Logistic Regression on the test set...")

# Fit a clean copy of the selected pipeline on all training data.
final_deployment_pipeline = clone(logistic_final_pipeline)
final_deployment_pipeline.fit(X_train, y_train)

# Use the threshold selected from out-of-fold training predictions.
final_prediction_threshold = logistic_final_results["threshold"]

# Generate test-set probabilities and classifications.
test_probabilities = final_deployment_pipeline.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= final_prediction_threshold
).astype(int)

# Calculate the confusion-matrix values.
tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
).ravel()

# Calculate the final test metrics.
final_test_results = {
    "model": "Logistic Regression",
    "dataset": "Untouched test set",
    "threshold": final_prediction_threshold,
    "recall": recall_score(y_test, test_predictions),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "specificity": tn / (tn + fp),
    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        test_predictions,
    ),
    "precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "f1": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "f2": fbeta_score(
        y_test,
        test_predictions,
        beta=2,
        zero_division=0,
    ),
    "pr_auc": average_precision_score(
        y_test,
        test_probabilities,
    ),
    "actual_positive_cases": int(y_test.sum()),
    "predicted_positive_cases": int(test_predictions.sum()),
    "true_positives": int(tp),
    "false_negatives": int(fn),
    "true_negatives": int(tn),
    "false_positives": int(fp),
}

# Display the final test results.
final_test_results_table = pd.DataFrame(
    [final_test_results]
).set_index(["model", "dataset"])

display(final_test_results_table.round(4))

print("\nFinal test evaluation completed.")
print(
    "Selected threshold:",
    round(final_prediction_threshold, 4),
)
print("Test set evaluated: True")
print("Ordinary accuracy included: False")
print("No further tuning should use the test results.")

Evaluating the final Logistic Regression on the test set...


,,threshold,recall,roc_auc,specificity,balanced_accuracy,precision,f1,f2,pr_auc,actual_positive_cases,predicted_positive_cases,true_positives,false_negatives,true_negatives,false_positives
model,dataset,,,,,,,,,,,,,,,
Logistic Regression,Untouched test set,0.4125,0.7609,0.7866,0.7017,0.7313,0.2029,0.3204,0.4909,0.2862,92,345,70,22,647,275



Final test evaluation completed.
Selected threshold: 0.4125
Test set evaluated: True
Ordinary accuracy included: False
No further tuning should use the test results.


### Final Test-Set Conclusion

The selected Logistic Regression model was evaluated once on the untouched test set using the previously selected probability threshold of **0.4125**.

- **Recall:** 0.7609
- **ROC-AUC:** 0.7866
- **Specificity:** 0.7017
- **Balanced accuracy:** 0.7313
- **PR-AUC:** 0.2862
- **F2 score:** 0.4909

The model correctly identified **70 of the 92 participants** with elevated depressive-symptom risk and missed 22 positive cases. Its test recall was moderately lower than the cross-validation recall of **0.8038**, while specificity slightly improved from **0.6922 to 0.7017**.

The relatively small differences between cross-validation and test performance indicate that the model generalizes reasonably well to unseen participants. Both the specificity and balanced-accuracy guardrails remained satisfied.

Therefore, Logistic Regression is retained as the final model because it provides strong sensitivity, stable generalization, interpretability, and an appropriate balance for a health-risk screening application. Its predictions represent screening support and are not a clinical diagnosis.

In [155]:
# Refit and save the finalized Logistic Regression model for deployment

from pathlib import Path
import json
import joblib
import sklearn

print("Preparing the final deployment model...")

# Combine the training and test predictors only after completing
# the final unbiased test evaluation.
X_full = pd.concat([X_train, X_test], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

# Refit the locked model configuration on all available data.
# No additional tuning or threshold optimization is performed.
production_pipeline = clone(logistic_final_pipeline)
production_pipeline.fit(X_full, y_full)

# Preserve the predictor order required by the pipeline.
selected_predictors = list(X_full.columns)

# Create the models directory.
model_directory = Path("../models")
model_directory.mkdir(parents=True, exist_ok=True)

model_path = model_directory / "final_depression_risk_model.joblib"
metadata_path = model_directory / "final_depression_risk_model_metadata.json"

# Store everything required to reproduce predictions.
model_bundle = {
    "pipeline": production_pipeline,
    "selected_predictors": selected_predictors,
    "prediction_threshold": float(
        logistic_final_results["threshold"]
    ),
    "positive_class": 1,
    "model_name": "Logistic Regression",
    "imbalance_handling": "Balanced class weights",
    "threshold_strategy": "Maximum out-of-fold F2",
    "primary_selection_metric": "Recall",
}

joblib.dump(model_bundle, model_path)

# Store readable model documentation separately.
model_metadata = {
    "model_name": "Logistic Regression",
    "number_of_predictors": len(selected_predictors),
    "selected_predictors": selected_predictors,
    "prediction_threshold": float(
        logistic_final_results["threshold"]
    ),
    "positive_class": 1,
    "imbalance_handling": "Balanced class weights",
    "threshold_strategy": "Maximum out-of-fold F2",
    "primary_selection_metric": "Recall",
    "secondary_metric": "ROC-AUC",
    "test_recall": float(final_test_results["recall"]),
    "test_roc_auc": float(final_test_results["roc_auc"]),
    "test_specificity": float(
        final_test_results["specificity"]
    ),
    "test_balanced_accuracy": float(
        final_test_results["balanced_accuracy"]
    ),
    "test_pr_auc": float(final_test_results["pr_auc"]),
    "scikit_learn_version": sklearn.__version__,
}

with open(metadata_path, "w", encoding="utf-8") as metadata_file:
    json.dump(
        model_metadata,
        metadata_file,
        indent=4,
    )

# Reload the bundle to verify successful serialization.
reloaded_bundle = joblib.load(model_path)

original_probabilities = production_pipeline.predict_proba(
    X_test[selected_predictors]
)[:, 1]

reloaded_probabilities = reloaded_bundle[
    "pipeline"
].predict_proba(
    X_test[selected_predictors]
)[:, 1]

predictions_match = np.allclose(
    original_probabilities,
    reloaded_probabilities,
)

print("\nFinal model saved successfully.")
print("Model location:", model_path)
print("Metadata location:", metadata_path)
print("Number of predictors:", len(selected_predictors))
print(
    "Saved threshold:",
    round(model_bundle["prediction_threshold"], 4),
)
print("Reloaded predictions match:", predictions_match)
print("Deployment model trained on all available records: True")

Preparing the final deployment model...

Final model saved successfully.
Model location: ../models/final_depression_risk_model.joblib
Metadata location: ../models/final_depression_risk_model_metadata.json
Number of predictors: 29
Saved threshold: 0.4125
Reloaded predictions match: True
Deployment model trained on all available records: True


### Model Finalization

The selected Logistic Regression pipeline was refitted on all available records after the unbiased test evaluation was completed.

The deployment bundle contains:

- The complete preprocessing and classification pipeline
- All 29 predictors in the required order
- The selected probability threshold of 0.4125
- The positive-class definition
- The imbalance-handling and threshold-selection strategies
- Final model metadata and test-performance results

The saved model was reloaded successfully, and its predictions exactly matched those produced before serialization. The model is now ready to be integrated into the Streamlit application.

In [156]:
# Inspect the exact input structure required by the Streamlit application

print("Final model input structure:\n")
print("Number of predictors:", len(selected_predictors))

input_structure = []

for column in selected_predictors:
    column_type = str(X_full[column].dtype)

    if column in categorical_columns:
        allowed_values = sorted(
            X_full[column].dropna().unique().tolist(),
            key=str,
        )

        input_structure.append(
            {
                "predictor": column,
                "input_type": "Categorical",
                "data_type": column_type,
                "allowed_values": allowed_values,
            }
        )
    else:
        input_structure.append(
            {
                "predictor": column,
                "input_type": "Numeric",
                "data_type": column_type,
                "minimum": X_full[column].min(),
                "median": X_full[column].median(),
                "maximum": X_full[column].max(),
            }
        )

input_structure_table = pd.DataFrame(input_structure)

display(input_structure_table)

Final model input structure:

Number of predictors: 29


,predictor,input_type,data_type,allowed_values,minimum,median,maximum
0,RIAGENDR,Categorical,float64,"[1.0, 2.0]",NaN,NaN,NaN
1,RIDAGEYR,Numeric,float64,NaN,18.0000,51.0000,80.0000
2,RIDRETH3,Categorical,float64,"[1.0, 2.0, 3.0, 4.0, 6.0, 7.0]",NaN,NaN,NaN
3,DMDEDUC2,Categorical,float64,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0]",NaN,NaN,NaN
4,DMDMARTL,Categorical,float64,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]",NaN,NaN,NaN
5,INDFMPIR,Numeric,float64,NaN,0.0000,2.1300,5.0000
6,SLQ030,Categorical,float64,"[0.0, 1.0, 2.0, 3.0]",NaN,NaN,NaN
7,SLQ040,Categorical,float64,"[0.0, 1.0, 2.0, 3.0]",NaN,NaN,NaN
8,SLQ050,Categorical,float64,"[1.0, 2.0]",NaN,NaN,NaN
9,SLQ120,Categorical,float64,"[0.0, 1.0, 2.0, 3.0, 4.0]",NaN,NaN,NaN
